In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os, glob, json, re
import numpy as np

D = '/content/drive/MyDrive/PROJECT'
print('folder utama ada:', os.path.isdir(D))

try:
    import torch_geometric
except ImportError:
    !pip -q install torch_geometric
    import torch_geometric

import torch
from torch_geometric.data import Data

print('torch', torch.__version__)
print('torch_geometric', torch_geometric.__version__)
print('cuda tersedia:', torch.cuda.is_available())

Mounted at /content/drive
folder utama ada: True
torch 2.11.0+cu130
torch_geometric 2.8.1
cuda tersedia: True


In [ ]:
print(f'=== isi {D} sampai kedalaman 2 ===\n')

for dirpath, dirnames, filenames in os.walk(D):
    rel = os.path.relpath(dirpath, D)
    depth = 0 if rel == '.' else rel.count(os.sep) + 1
    if depth > 2:
        dirnames[:] = []
        continue

    indent = '  ' * depth
    tajuk = D if rel == '.' else rel
    print(f'{indent}[DIR] {tajuk}  ({len(filenames)} file, {len(dirnames)} subfolder)')

    if depth == 2 and len(filenames) > 6:
        print(f'{indent}  ... {len(filenames)} file, contoh:', sorted(filenames)[:4])
        dirnames[:] = []
        continue

    for nama in sorted(filenames)[:30]:
        mb = os.path.getsize(os.path.join(dirpath, nama)) / 1e6
        print(f'{indent}  [file] {nama}  ({mb:.2f} MB)')
    if len(filenames) > 30:
        print(f'{indent}  ... dan {len(filenames) - 30} file lain')

=== isi /content/drive/MyDrive/PROJECT sampai kedalaman 2 ===

[DIR] /content/drive/MyDrive/PROJECT  (0 file, 3 subfolder)
  [DIR] data  (0 file, 2 subfolder)
    [DIR] data/raw  (4 file, 0 subfolder)
      [file] dataset_final.zip  (4.97 MB)
      [file] dataset_penelitian.zip  (7.58 MB)
      [file] dataset_v2.zip  (11.26 MB)
      [file] karantina.zip  (3.26 MB)
    [DIR] data/processed  (8 file, 8 subfolder)
      ... 8 file, contoh: ['kontrak_dipecah_vs_utuh.pkl', 'label_fungsi_reentrancy.pkl', 'manifest.csv', 'manifest_final.csv']
  [DIR] solc_binaries  (0 file, 1 subfolder)
    [DIR] solc_binaries/artifacts  (0 file, 61 subfolder)
  [DIR] hasil  (32 file, 0 subfolder)
    [file] benchmark_cfg_prune_0.02.csv  (0.00 MB)
    [file] benchmark_graphsmote.csv  (0.00 MB)
    [file] benchmark_probabilitas.csv  (0.00 MB)
    [file] benchmark_v3_augmented_final.csv  (0.00 MB)
    [file] cfg_satu_pasangan.csv  (0.00 MB)
    [file] graphsmote_multiseed.csv  (0.00 MB)
    [file] graphsmote_v

In [ ]:
kandidat = sorted(set(
    glob.glob(f'{D}/**/*v3*.pt', recursive=True) +
    glob.glob(f'{D}/**/*augment*.pt', recursive=True)
))

print('=== kandidat berkas dataset v3 ===')
for p in kandidat:
    print(f'  {os.path.relpath(p, D)}  ({os.path.getsize(p)/1e6:.2f} MB)')
print()

def ringkas(obj, nama_berkas):
    print(f'--- {nama_berkas} ---')
    print('tipe kontainer:', type(obj).__name__)

    if isinstance(obj, dict):
        print('kunci dict:', list(obj.keys()))
        for k, v in obj.items():
            if isinstance(v, (list, tuple)) and len(v) and isinstance(v[0], Data):
                print(f'  kunci "{k}" berisi {len(v)} objek Data')
                return list(v)
        return None

    if isinstance(obj, (list, tuple)) and len(obj) and isinstance(obj[0], Data):
        print('jumlah objek Data:', len(obj))
        return list(obj)

    print('bukan daftar Data, isi:', repr(obj)[:200])
    return None

for p in kandidat:
    isi = torch.load(p, map_location='cpu', weights_only=False)
    ds = ringkas(isi, os.path.relpath(p, D))
    if ds is None:
        print()
        continue

    print('\n  atribut tiga objek pertama:')
    for i in range(min(3, len(ds))):
        print(f'  [{i}]')
        for k, v in ds[i].to_dict().items():
            if torch.is_tensor(v):
                print(f'      {k}: tensor{tuple(v.shape)} dtype={v.dtype}'
                      + (f' nilai={v.item()}' if v.numel() == 1 else ''))
            else:
                print(f'      {k}: {type(v).__name__} = {repr(v)[:140]}')

    kunci_str = [k for k, v in ds[0].to_dict().items() if isinstance(v, str)]
    print('\n  atribut bertipe string:', kunci_str if kunci_str else 'TIDAK ADA')

    for k in kunci_str:
        nilai = [getattr(g, k) for g in ds]
        print(f'\n  contoh 12 nilai "{k}":')
        for s in nilai[:12]:
            print(f'      {s}')
        print(f'  jumlah nilai unik "{k}": {len(set(nilai))} dari {len(nilai)} graf')

    try:
        y = np.array([int(g.y) for g in ds])
        print('\n  distribusi label:', {int(c): int((y == c).sum()) for c in np.unique(y)})
    except Exception as e:
        print('\n  label tidak terbaca:', e)

    n_node = np.array([g.num_nodes for g in ds])
    print(f'  jumlah simpul: min {n_node.min()}, median {int(np.median(n_node))}, maks {n_node.max()}')
    print()

=== kandidat berkas dataset v3 ===
  data/processed/pyg/v3_augmented.pt  (49.55 MB)
  hasil/model_v3_augmented.pt  (12.63 MB)

--- data/processed/pyg/v3_augmented.pt ---
tipe kontainer: dict
kunci dict: ['train', 'benchmark', 'dim', 'tipe']
  kunci "train" berisi 1712 objek Data

  atribut tiga objek pertama:
  [0]
      x: tensor(188, 22) dtype=torch.float32
      edge_index: tensor(2, 184) dtype=torch.int64
      y: tensor(1,) dtype=torch.int64 nilai=0
      edge_type: tensor(184,) dtype=torch.int64
      nama: str = '0x01FB70eA8AcFDd5435c8e8bd1531d96D57cEd0ec.sol'
      split: str = 'train'
      folder: str = '0_normal'
  [1]
      x: tensor(240, 22) dtype=torch.float32
      edge_index: tensor(2, 236) dtype=torch.int64
      y: tensor(1,) dtype=torch.int64 nilai=0
      edge_type: tensor(236,) dtype=torch.int64
      nama: str = '0x01f5c9483869Eb249bF3D4155f0cC7BE064c3506.sol'
      split: str = 'train'
      folder: str = '0_normal'
  [2]
      x: tensor(232, 22) dtype=torch.floa

In [ ]:
isi = torch.load(f'{D}/data/processed/pyg/v3_augmented.pt', map_location='cpu', weights_only=False)
ds = isi['train']

n_node = np.array([g.num_nodes for g in ds])
nama_semua = [g.nama for g in ds]
folder_semua = [g.folder for g in ds]

print('nilai unik folder:', sorted(set(folder_semua)))
print()

print('=== 15 contoh graf TERKECIL (kemungkinan subgraf fungsi) ===')
idx_kecil = np.argsort(n_node)[:15]
for i in idx_kecil:
    print(f'  simpul={n_node[i]:>5}  folder={ds[i].folder:<15}  nama={ds[i].nama}')

print()
print('=== 15 contoh graf TERBESAR (kemungkinan kontrak utuh) ===')
idx_besar = np.argsort(n_node)[-15:]
for i in idx_besar:
    print(f'  simpul={n_node[i]:>5}  folder={ds[i].folder:<15}  nama={ds[i].nama}')

print()
print('=== distribusi ukuran, buat lihat ada dua gerombolan atau tidak ===')
for batas in [10, 30, 50, 100, 150, 200, 300, 500, 1000]:
    print(f'  jumlah graf dengan simpul <= {batas}: {(n_node <= batas).sum()}')
print(f'  jumlah graf total: {len(ds)}')

nilai unik folder: ['0_normal', '1_reentrancy']

=== 15 contoh graf TERKECIL (kemungkinan subgraf fungsi) ===
  simpul=    2  folder=1_reentrancy     nama=0x00e2c3b903bec1131fb05283c92df8a94613ca4a.sol::approve
  simpul=    2  folder=1_reentrancy     nama=0x0060228f198ec12089053786e762137d7cc70c60.sol::increaseAllowance
  simpul=    2  folder=1_reentrancy     nama=0x005aefa801f03d7cd56d577bf91c9f4ca6c3c528.sol::decreaseAllowance
  simpul=    2  folder=1_reentrancy     nama=0x340647745b69ada2d9b49c304d9eea79e36ab72b.sol::jSub
  simpul=    2  folder=1_reentrancy     nama=0xb743339d669fcd84e4f4957804350492991a3b96.sol::getVerificationVector
  simpul=    2  folder=1_reentrancy     nama=0x00d43dd0581395f9d3ccc41f28c732c2400a69a9.sol::decreaseAllowance
  simpul=    2  folder=0_normal         nama=0x10a5C4d02f5BaEcd1001bad654A9C12bae258164.sol::_remove
  simpul=    2  folder=1_reentrancy     nama=0xa06cd5c489c59b463f8fe09ef29fbfc144c74b06.sol::jSub
  simpul=    2  folder=0_normal         nama

In [ ]:
from collections import Counter

def ambil_contract_id(nama):
    return nama.split('::')[0]

groups = np.array([ambil_contract_id(g.nama) for g in ds])

for g, cid in zip(ds, groups):
    g.contract_id = cid

n_grup = len(set(groups))
print(f'jumlah grup (kontrak) unik: {n_grup}')

hitung_per_grup = Counter(groups)
ukuran_grup = np.array(list(hitung_per_grup.values()))
print(f'graf per grup: min {ukuran_grup.min()}, median {np.median(ukuran_grup):.1f}, maks {ukuran_grup.max()}')

print()
print('=== verifikasi kelengkapan graf utuh per grup ===')
ada_utuh = Counter()
for g in ds:
    if '::' not in g.nama:
        ada_utuh[g.contract_id] += 1

grup_tanpa_utuh = [cid for cid in set(groups) if ada_utuh[cid] == 0]
grup_lebih_satu_utuh = [cid for cid in set(groups) if ada_utuh[cid] > 1]

print(f'grup tanpa graf utuh sama sekali: {len(grup_tanpa_utuh)}')
if grup_tanpa_utuh:
    print('  contoh:', grup_tanpa_utuh[:5])
print(f'grup dengan lebih dari satu graf utuh: {len(grup_lebih_satu_utuh)}')
if grup_lebih_satu_utuh:
    print('  contoh:', grup_lebih_satu_utuh[:5])

jumlah_utuh = sum(1 for g in ds if '::' not in g.nama)
jumlah_subgraf = sum(1 for g in ds if '::' in g.nama)
print()
print(f'jumlah graf utuh: {jumlah_utuh}')
print(f'jumlah subgraf fungsi: {jumlah_subgraf}')
print(f'total: {jumlah_utuh + jumlah_subgraf}')

print()
print('=== distribusi kelas per grup, cek konsistensi label dalam satu grup ===')
label_per_grup = {}
for g in ds:
    label_per_grup.setdefault(g.contract_id, set()).add(int(g.y))

grup_label_campur = [cid for cid, labs in label_per_grup.items() if len(labs) > 1]
print(f'grup dengan label campur (utuh dan subgraf tidak sekelas): {len(grup_label_campur)}')
if grup_label_campur:
    print('  contoh:', grup_label_campur[:5])

jumlah grup (kontrak) unik: 607
graf per grup: min 1, median 2.0, maks 21

=== verifikasi kelengkapan graf utuh per grup ===
grup tanpa graf utuh sama sekali: 0
grup dengan lebih dari satu graf utuh: 0

jumlah graf utuh: 607
jumlah subgraf fungsi: 1105
total: 1712

=== distribusi kelas per grup, cek konsistensi label dalam satu grup ===
grup dengan label campur (utuh dan subgraf tidak sekelas): 149
  contoh: [np.str_('0x00008dde8c1a98ae1c63b8d495304bd5f95f1a24.sol'), np.str_('0x0003ed19f80564745e84b4cc411a7b6be4f0cf31.sol'), np.str_('0x0012493dd0991f292d0a4774afdd19f007895470.sol'), np.str_('0x0014966c7439d8c10aaf0204036198236d9b0912.sol'), np.str_('0x00195777bed7025e78819156281192c85fb3cf9b.sol')]


In [ ]:
print(f'total grup label campur: {len(grup_label_campur)} dari {n_grup}')
print()

print('=== detail 3 contoh grup label campur ===')
for cid in list(grup_label_campur)[:3]:
    anggota = [g for g in ds if g.contract_id == cid]
    anggota_sorted = sorted(anggota, key=lambda g: '::' not in g.nama, reverse=True)
    print(f'grup: {cid}')
    for g in anggota_sorted:
        jenis = 'GRAF UTUH' if '::' not in g.nama else 'subgraf: ' + g.nama.split('::')[1]
        print(f'    y={int(g.y)}  simpul={g.num_nodes:<4}  {jenis}')
    print()

print('=== ringkasan arah campurannya ===')
utuh_1_ada_sub_0 = 0
utuh_0_ada_sub_1 = 0
for cid in grup_label_campur:
    anggota = [g for g in ds if g.contract_id == cid]
    y_utuh = next(int(g.y) for g in anggota if '::' not in g.nama)
    y_sub = [int(g.y) for g in anggota if '::' in g.nama]
    if y_utuh == 1 and 0 in y_sub:
        utuh_1_ada_sub_0 += 1
    if y_utuh == 0 and 1 in y_sub:
        utuh_0_ada_sub_1 += 1

print(f'grup utuh=rentan tapi ada subgraf aman: {utuh_1_ada_sub_0}')
print(f'grup utuh=aman tapi ada subgraf rentan: {utuh_0_ada_sub_1}')

total grup label campur: 149 dari 607

=== detail 3 contoh grup label campur ===
grup: 0x00008dde8c1a98ae1c63b8d495304bd5f95f1a24.sol
    y=1  simpul=517   GRAF UTUH
    y=1  simpul=5     subgraf: airdrop
    y=1  simpul=4     subgraf: airdropInternal
    y=1  simpul=21    subgraf: _transfer
    y=0  simpul=13    subgraf: _transferToExcluded
    y=0  simpul=4     subgraf: setMaxTxAmount

grup: 0x0003ed19f80564745e84b4cc411a7b6be4f0cf31.sol
    y=1  simpul=303   GRAF UTUH
    y=1  simpul=27    subgraf: _transfer
    y=1  simpul=12    subgraf: openTrading
    y=0  simpul=3     subgraf: _getCurrentSupply

grup: 0x0012493dd0991f292d0a4774afdd19f007895470.sol
    y=1  simpul=511   GRAF UTUH
    y=1  simpul=15    subgraf: _transfer
    y=0  simpul=19    subgraf: constructor
    y=0  simpul=3     subgraf: setBuy
    y=0  simpul=5     subgraf: setBuyFee
    y=0  simpul=4     subgraf: setMaxWallet

=== ringkasan arah campurannya ===
grup utuh=rentan tapi ada subgraf aman: 149
grup utuh=aman tap

In [ ]:
import zipfile
import pandas as pd
from collections import Counter

print('=== isi lengkap data/processed ===')
P = f'{D}/data/processed'
for nama in sorted(os.listdir(P)):
    p = os.path.join(P, nama)
    if os.path.isdir(p):
        isi_dir = os.listdir(p)
        ext = Counter(os.path.splitext(f)[1] for f in isi_dir)
        print(f'  [DIR ] {nama}  ({len(isi_dir)} item)  ekstensi: {dict(ext)}')
        print(f'           contoh: {sorted(isi_dir)[:3]}')
    else:
        print(f'  [file] {nama}  ({os.path.getsize(p)/1e6:.2f} MB)')

print('\n=== pencarian berkas .sol di seluruh folder proyek ===')
sol = glob.glob(f'{D}/**/*.sol', recursive=True)
print('jumlah berkas .sol ditemukan:', len(sol))
if sol:
    folder_sol = Counter(os.path.dirname(os.path.relpath(p, D)) for p in sol)
    for folder, n in folder_sol.most_common(20):
        print(f'  {folder}: {n} berkas')

print('\n=== seluruh berkas .pt di data/processed ===')
for p in sorted(glob.glob(f'{D}/data/processed/**/*.pt', recursive=True)):
    print(f'  {os.path.relpath(p, D)}  ({os.path.getsize(p)/1e6:.2f} MB)')

print('\n=== isi berkas zip di data/raw ===')
for z in sorted(glob.glob(f'{D}/data/raw/*.zip')):
    with zipfile.ZipFile(z) as f:
        names = f.namelist()
    ext = Counter(os.path.splitext(n)[1] for n in names)
    print(f'\n  {os.path.basename(z)}: {len(names)} entri, ekstensi: {dict(ext)}')
    print(f'    contoh: {names[:5]}')

print('\n=== manifest ===')
for m in ['manifest.csv', 'manifest_final.csv']:
    p = f'{D}/data/processed/{m}'
    if os.path.exists(p):
        df = pd.read_csv(p)
        print(f'\n  {m}, bentuk {df.shape}')
        print('  kolom:', list(df.columns))
        print(df.head(3).to_string(index=False))

=== isi lengkap data/processed ===
  [DIR ] cfg  (760 item)  ekstensi: {'.json': 760}
           contoh: ['0_normal__0x00000000000045166c45af0fc6e4cf31d9e14b9a.json', '0_normal__0x000000000000cd17345801aa8147b8D3950260FF.json', '0_normal__0x000000000000d0151e748d25b766e77efe2a6c83.json']
  [DIR ] cfg_prune_0.02  (760 item)  ekstensi: {'.json': 760}
           contoh: ['0_normal__0x00000000000045166c45af0fc6e4cf31d9e14b9a.json', '0_normal__0x000000000000cd17345801aa8147b8D3950260FF.json', '0_normal__0x000000000000d0151e748d25b766e77efe2a6c83.json']
  [file] kontrak_dipecah_vs_utuh.pkl  (0.01 MB)
  [file] label_fungsi_reentrancy.pkl  (0.02 MB)
  [file] manifest.csv  (0.18 MB)
  [file] manifest_final.csv  (0.14 MB)
  [file] manifest_v2.csv  (0.16 MB)
  [DIR ] pyg  (5 item)  ekstensi: {'.pt': 5}
           contoh: ['cfg.pt', 'cfg_prune_0.02.pt', 'v2_cfg_prune.pt']
  [file] statistik_cfg.csv  (0.06 MB)
  [file] ukuran_reentrancy.csv  (0.01 MB)
  [DIR ] v2_cfg  (627 item)  ekstensi: {'.json'

In [ ]:
import zipfile
import pandas as pd

os.makedirs('/content/data_raw', exist_ok=True)
for z in sorted(glob.glob(f'{D}/data/raw/*.zip')):
    with zipfile.ZipFile(z) as f:
        f.extractall('/content/data_raw')
    print(f'diekstrak: {os.path.basename(z)}')

print()
print('=== struktur /content/data_raw setelah ekstraksi, sampai kedalaman 2 ===')
for dirpath, dirnames, filenames in os.walk('/content/data_raw'):
    rel = os.path.relpath(dirpath, '/content/data_raw')
    depth = 0 if rel == '.' else rel.count(os.sep) + 1
    if depth > 2:
        dirnames[:] = []
        continue
    jml_sol = sum(1 for f in filenames if f.endswith('.sol'))
    print(f'{"  "*depth}{rel}  ({jml_sol} .sol langsung di sini, {len(dirnames)} subfolder)')

semua_sol = glob.glob('/content/data_raw/**/*.sol', recursive=True)
print(f'\ntotal path .sol hasil ekstraksi (gabungan 4 zip): {len(semua_sol)}')

indeks_nama = {}
nama_di_banyak_lokasi = []
for p in semua_sol:
    nm = os.path.basename(p)
    if nm in indeks_nama and indeks_nama[nm] != p:
        nama_di_banyak_lokasi.append(nm)
    indeks_nama.setdefault(nm, []).append(p) if isinstance(indeks_nama.get(nm), list) else indeks_nama.update({nm: [p]} if nm not in indeks_nama else {nm: indeks_nama[nm] + [p]})

print(f'nama file unik: {len(indeks_nama)}')
multi = {k: v for k, v in indeks_nama.items() if len(v) > 1}
print(f'nama file yang muncul di lebih dari satu path (lintas zip): {len(multi)}')
for nm, paths in list(multi.items())[:5]:
    print(f'  {nm}')
    for p in paths:
        print(f'    {p}')

print()
print('=== manifest_v2.csv vs v2_manifest.csv ===')
mv2a = pd.read_csv(f'{D}/data/processed/manifest_v2.csv')
mv2b = pd.read_csv(f'{D}/data/processed/v2_manifest.csv')
print('manifest_v2.csv  :', mv2a.shape, list(mv2a.columns))
print('v2_manifest.csv  :', mv2b.shape, list(mv2b.columns))
if mv2a.shape == mv2b.shape:
    print('isinya identik persis:', mv2a.equals(mv2b))
    if not mv2a.equals(mv2b):
        beda = (mv2a != mv2b).any(axis=1)
        print('jumlah baris berbeda:', beda.sum())
else:
    print('bentuknya beda, sudah pasti bukan berkas yang sama')

print()
print('=== distribusi label per manifest ===')
for nm in ['manifest.csv', 'manifest_final.csv', 'manifest_v2.csv', 'v2_manifest.csv']:
    p = f'{D}/data/processed/{nm}'
    df = pd.read_csv(p)
    print(f'\n{nm}  total={len(df)}')
    if 'split' in df.columns:
        print(df.groupby(['split', 'label']).size().to_string())
    else:
        print(df['label'].value_counts().to_string())

diekstrak: dataset_final.zip
diekstrak: dataset_penelitian.zip
diekstrak: dataset_v2.zip
diekstrak: karantina.zip

=== struktur /content/data_raw setelah ekstraksi, sampai kedalaman 2 ===
.  (0 .sol langsung di sini, 3 subfolder)
  dataset_penelitian  (0 .sol langsung di sini, 4 subfolder)
    dataset_penelitian/3_expert_benchmark  (0 .sol langsung di sini, 2 subfolder)
    dataset_penelitian/2_wild_test  (500 .sol langsung di sini, 0 subfolder)
    dataset_penelitian/1_reentrancy  (318 .sol langsung di sini, 0 subfolder)
    dataset_penelitian/0_normal  (512 .sol langsung di sini, 0 subfolder)
  karantina  (0 .sol langsung di sini, 4 subfolder)
    karantina/2_wild_test  (500 .sol langsung di sini, 0 subfolder)
    karantina/3_expert_benchmark_aman_oz  (58 .sol langsung di sini, 0 subfolder)
    karantina/1_reentrancy  (80 .sol langsung di sini, 0 subfolder)
    karantina/0_normal  (12 .sol langsung di sini, 0 subfolder)
  dataset_v2  (0 .sol langsung di sini, 2 subfolder)
    dataset

In [ ]:
from collections import Counter

def hitung_prefix(folder_path):
    files = os.listdir(folder_path)
    prefix = Counter()
    for f in files:
        bagian = f.split('__', 1)
        kunci = bagian[0] if len(bagian) == 2 else '(tidak ada awalan __)'
        prefix[kunci] += 1
    return prefix, len(files)

for nama_folder in ['cfg', 'cfg_prune_0.02', 'v2_cfg', 'v2_cfg_prune_0.02', 'v3_cfg_dengan_fungsi']:
    p = f'{D}/data/processed/{nama_folder}'
    prefix, total = hitung_prefix(p)
    print(f'{nama_folder}  (total {total})')
    for k, v in sorted(prefix.items()):
        print(f'    {k}: {v}')
    print()

cfg  (total 760)
    0_normal: 500
    1_reentrancy: 244
    3_expert_benchmark: 16

cfg_prune_0.02  (total 760)
    0_normal: 500
    1_reentrancy: 244
    3_expert_benchmark: 16

v2_cfg  (total 627)
    0_normal: 367
    1_reentrancy: 244
    3_expert_benchmark: 16

v2_cfg_prune_0.02  (total 627)
    0_normal: 367
    1_reentrancy: 244
    3_expert_benchmark: 16

v3_cfg_dengan_fungsi  (total 607)
    0_normal: 363
    1_reentrancy: 244



In [ ]:
def contoh_nama(folder_path, prefix_list):
    for prefix_target in prefix_list:
        contoh = [f for f in os.listdir(folder_path) if f.startswith(prefix_target + '__')]
        print(f'  {prefix_target}: {len(contoh)} total, contoh 5:')
        for f in sorted(contoh)[:5]:
            print(f'      {f}')

print('=== folder cfg ===')
contoh_nama(f'{D}/data/processed/cfg', ['0_normal', '1_reentrancy', '3_expert_benchmark'])

print('\n=== folder v2_cfg ===')
contoh_nama(f'{D}/data/processed/v2_cfg', ['0_normal', '1_reentrancy', '3_expert_benchmark'])

print('\n=== folder v3_cfg_dengan_fungsi ===')
contoh_nama(f'{D}/data/processed/v3_cfg_dengan_fungsi', ['0_normal', '1_reentrancy'])

print('\n=== isi kunci benchmark pada v3_augmented.pt ===')
bench = isi['benchmark'] if 'benchmark' in locals().get('isi', {}) else torch.load(f'{D}/data/processed/pyg/v3_augmented.pt', map_location='cpu', weights_only=False)['benchmark']
print('tipe:', type(bench).__name__, ' jumlah:', len(bench))
for g in bench[:5]:
    print(f'   nama={g.nama}  y={int(g.y)}  simpul={g.num_nodes}')

=== folder cfg ===
  0_normal: 500 total, contoh 5:
      0_normal__0x00000000000045166c45af0fc6e4cf31d9e14b9a.json
      0_normal__0x000000000000cd17345801aa8147b8D3950260FF.json
      0_normal__0x000000000000d0151e748d25b766e77efe2a6c83.json
      0_normal__0x0000000000027f6d87be8ade118d9ee56767d993.json
      0_normal__0x000000000009546b5e66182fa242f1aa669a5051.json
  1_reentrancy: 244 total, contoh 5:
      1_reentrancy__0x0000000000b3F879cb30FE243b4Dfee438691c04.json
      1_reentrancy__0x00000000f2cfa550ad4aae0f33bcaad5164900be.json
      1_reentrancy__0x00000624264253944a702b8b812ae7bbb46af371.json
      1_reentrancy__0x00008dde8c1a98ae1c63b8d495304bd5f95f1a24.json
      1_reentrancy__0x0001b7fa59950c817b8caa0526f793863c6c2d74.json
  3_expert_benchmark: 16 total, contoh 5:
      3_expert_benchmark__aman__etherbank_fixed.json
      3_expert_benchmark__aman__etherstore_fixed.json
      3_expert_benchmark__aman__modifier_reentrancy_fixed.json
      3_expert_benchmark__aman__reentra

In [ ]:
path_per_nama = {}
for p in semua_sol:
    nm = os.path.basename(p)
    path_per_nama.setdefault(nm, []).append(p)

print('=== cek isi identik untuk nama yang muncul di lebih dari satu path ===')
beda_isi = []
for nm, paths in path_per_nama.items():
    if len(paths) > 1:
        isi_list = [open(p, 'r', encoding='utf-8', errors='replace').read() for p in paths]
        if len(set(isi_list)) > 1:
            beda_isi.append((nm, paths))

print(f'nama dengan path lebih dari satu: {sum(1 for v in path_per_nama.values() if len(v) > 1)}')
print(f'di antaranya, isinya benar beda (bukan cuma salinan): {len(beda_isi)}')
for nm, paths in beda_isi[:10]:
    print(f'  {nm}')
    for p in paths:
        print(f'      {p}')

print()
print('=== struktur satu contoh JSON CFG, folder cfg ===')
contoh_json = os.path.join(f'{D}/data/processed/cfg', sorted(f for f in os.listdir(f'{D}/data/processed/cfg') if f.startswith('1_reentrancy__'))[0])
with open(contoh_json) as f:
    isi_contoh = json.load(f) if False else __import__('json').load(f)

print('nama berkas:', os.path.basename(contoh_json))
print('tipe objek teratas:', type(isi_contoh).__name__)
if isinstance(isi_contoh, dict):
    print('kunci teratas:', list(isi_contoh.keys()))
    for k, v in isi_contoh.items():
        print(f'  kunci "{k}": tipe {type(v).__name__}', f', panjang {len(v)}' if isinstance(v, (list, dict)) else '')

import json as _json
print()
print('cuplikan JSON (1500 karakter pertama):')
print(_json.dumps(isi_contoh, indent=2)[:1500])

=== cek isi identik untuk nama yang muncul di lebih dari satu path ===
nama dengan path lebih dari satu: 886
di antaranya, isinya benar beda (bukan cuma salinan): 1
  0x0046f822ef4e6d51a40dd87d34ab1d5f088291a4.sol
      /content/data_raw/dataset_penelitian/2_wild_test/0x0046f822ef4e6d51a40dd87d34ab1d5f088291a4.sol
      /content/data_raw/dataset_penelitian/1_reentrancy/0x0046f822ef4e6d51a40dd87d34ab1d5f088291a4.sol
      /content/data_raw/karantina/2_wild_test/0x0046f822ef4e6d51a40dd87d34ab1d5f088291a4.sol
      /content/data_raw/dataset_v2/1_reentrancy/0x0046f822ef4e6d51a40dd87d34ab1d5f088291a4.sol

=== struktur satu contoh JSON CFG, folder cfg ===
nama berkas: 1_reentrancy__0x0000000000b3F879cb30FE243b4Dfee438691c04.json
tipe objek teratas: dict
kunci teratas: ['nodes', 'edges', 'n_node', 'n_edge', 'n_contract', 'nama_file', 'folder', 'label', 'split', 'solc']
  kunci "nodes": tipe list , panjang 147
  kunci "edges": tipe list , panjang 134
  kunci "n_node": tipe int 
  kunci "n_edge

In [ ]:
import hashlib, re, json as _json
from collections import Counter, defaultdict

path_per_nama = defaultdict(list)
for p in semua_sol:
    if '2_wild_test' in p:
        continue
    path_per_nama[os.path.basename(p)].append(p)

def ambil_nama_sol(nama_json):
    tanpa_ext = nama_json[:-5]
    return tanpa_ext.split('__')[-1] + '.sol'

def normalisasi_kode(kode):
    tanpa_komentar = re.sub(r'//.*', '', kode)
    tanpa_komentar = re.sub(r'/\*.*?\*/', '', tanpa_komentar, flags=re.DOTALL)
    return re.sub(r'\s+', ' ', tanpa_komentar).strip()

def hash_kode_dari_nama(nama_sol):
    paths = path_per_nama.get(nama_sol)
    if not paths:
        return None, 'TIDAK_DITEMUKAN'
    isi_set = set()
    for p in paths:
        with open(p, 'r', encoding='utf-8', errors='replace') as f:
            isi_set.add(f.read())
    if len(isi_set) > 1:
        return None, 'ISI_BEDA_DI_LOKASI_BERBEDA'
    return hashlib.sha256(normalisasi_kode(isi_set.pop()).encode('utf-8')).hexdigest(), 'OK'

def muat_entri(folder_path):
    hasil, gagal = [], []
    for f in sorted(os.listdir(folder_path)):
        if not f.endswith('.json'):
            continue
        with open(os.path.join(folder_path, f)) as fh:
            j = _json.load(fh)
        nama_sol = ambil_nama_sol(f)
        h_kode, status = hash_kode_dari_nama(nama_sol)
        tipe_multiset = tuple(sorted(Counter(n['tipe'] for n in j['nodes']).items()))
        if status != 'OK':
            gagal.append((f, nama_sol, status))
        hasil.append({
            'berkas_json': f, 'nama_sol': nama_sol, 'folder': j['folder'], 'label': j['label'],
            'n_node': j['n_node'], 'n_edge': j['n_edge'],
            'hash_kode': h_kode, 'hash_struktur': (j['n_node'], j['n_edge'], tipe_multiset),
        })
    return hasil, gagal

print('memuat v1...'); entri_v1, gagal_v1 = muat_entri(f'{D}/data/processed/cfg')
print('memuat v2...'); entri_v2, gagal_v2 = muat_entri(f'{D}/data/processed/v2_cfg')
print('memuat v3...'); entri_v3, gagal_v3 = muat_entri(f'{D}/data/processed/v3_cfg_dengan_fungsi')

print('\n=== kegagalan resolusi path sumber ===')
for nm, gagal in [('v1', gagal_v1), ('v2', gagal_v2), ('v3', gagal_v3)]:
    print(f'{nm}: {len(gagal)} gagal')
    for f, ns, st in gagal[:10]:
        print(f'    {f} -> {ns}  ({st})')

def laporkan_duplikat(nama_versi, entri, hanya_train=True):
    data = [e for e in entri if e['folder'] != '3_expert_benchmark'] if hanya_train else entri
    print(f'\n=== {nama_versi}, {"train saja" if hanya_train else "seluruh termasuk benchmark"}, total {len(data)} ===')
    for metode, kunci in [('hash kode', 'hash_kode'), ('hash struktur', 'hash_struktur')]:
        grup = defaultdict(list)
        for e in data:
            if e[kunci] is not None:
                grup[e[kunci]].append(e['nama_sol'])
        grup_dobel = {k: v for k, v in grup.items() if len(v) > 1}
        n_duplikat_entri = sum(len(v) - 1 for v in grup_dobel.values())
        print(f'  {metode}: {len(grup_dobel)} grup kembar, {n_duplikat_entri} entri berlebih')
        for k, v in list(grup_dobel.items())[:5]:
            print(f'      {v}')
    return data

data_v1 = laporkan_duplikat('v1', entri_v1)
data_v2 = laporkan_duplikat('v2', entri_v2)
data_v3 = laporkan_duplikat('v3', entri_v3)

print('\n=== tumpang tindih benchmark (16 kontrak CEI) dengan data latih ===')
bench_entri = [e for e in entri_v1 if e['folder'] == '3_expert_benchmark']
for nama_versi, data_latih in [('v1', data_v1), ('v2', data_v2), ('v3', data_v3)]:
    for metode, kunci in [('hash kode', 'hash_kode'), ('hash struktur', 'hash_struktur')]:
        set_latih = {e[kunci] for e in data_latih if e[kunci] is not None}
        tabrakan = [(b['nama_sol']) for b in bench_entri if b[kunci] in set_latih]
        print(f'  {nama_versi} vs benchmark, {metode}: {len(tabrakan)} tabrakan', tabrakan[:5] if tabrakan else '')

memuat v1...
memuat v2...
memuat v3...

=== kegagalan resolusi path sumber ===
v1: 0 gagal
v2: 4 gagal
    0_normal__0x3f21b20a9ee87c5e6945117ee37ef83b195ebd7b.json -> 0x3f21b20a9ee87c5e6945117ee37ef83b195ebd7b.sol  (TIDAK_DITEMUKAN)
    0_normal__0x5f2539a5dc504493ba9ae1207803c08634cbb091.json -> 0x5f2539a5dc504493ba9ae1207803c08634cbb091.sol  (TIDAK_DITEMUKAN)
    0_normal__0xbfa82fbe0e66d8e2b7dcc16328db9ecd70533d13.json -> 0xbfa82fbe0e66d8e2b7dcc16328db9ecd70533d13.sol  (TIDAK_DITEMUKAN)
    0_normal__0xcc6a0dd80ff3fa8c1b37011765ee19efc480f825.json -> 0xcc6a0dd80ff3fa8c1b37011765ee19efc480f825.sol  (TIDAK_DITEMUKAN)
v3: 0 gagal

=== v1, train saja, total 760 ===
  hash kode: 24 grup kembar, 46 entri berlebih
      ['0x001831de9b6Ba15207e36A4f2ecA01322E411293.sol', '0x00E3c1F30dC416dBF841435cB1b2188c1A268F7E.sol', '0x00c8187C790fA909C88e20a7c90bC1Ac817B1774.sol', '0x00dE0F192931353837B4D6Bc2208f4b58C1EC5C6.sol', '0x0129602852C366AcaFa0aD4465022125353038F9.sol', '0x0164F85A0D4EA7C2f93

In [ ]:
print('=== nilai folder yang sebenarnya pada entri_v1 ===')
print(sorted(set(e['folder'] for e in entri_v1)))

print()
print('=== contoh satu entri benchmark ===')
for e in entri_v1:
    if 'expert_benchmark' in e['folder']:
        print(e['berkas_json'], '->', repr(e['folder']))
        break

print()
print('=== coba cari 4 alamat yang hilang di v2, tanpa pedulikan besar kecil huruf ===')
path_lower = defaultdict(list)
for p in semua_sol:
    if '2_wild_test' in p:
        continue
    path_lower[os.path.basename(p).lower()].append(p)

for nm, nama_sol, status in gagal_v2:
    cocok = path_lower.get(nama_sol.lower())
    print(f'{nama_sol}: ditemukan tanpa pedulikan huruf -> {cocok}')

=== nilai folder yang sebenarnya pada entri_v1 ===
['0_normal', '1_reentrancy', '3_expert_benchmark/aman', '3_expert_benchmark/rentan']

=== contoh satu entri benchmark ===
3_expert_benchmark__aman__etherbank_fixed.json -> '3_expert_benchmark/aman'

=== coba cari 4 alamat yang hilang di v2, tanpa pedulikan besar kecil huruf ===
0x3f21b20a9ee87c5e6945117ee37ef83b195ebd7b.sol: ditemukan tanpa pedulikan huruf -> None
0x5f2539a5dc504493ba9ae1207803c08634cbb091.sol: ditemukan tanpa pedulikan huruf -> None
0xbfa82fbe0e66d8e2b7dcc16328db9ecd70533d13.sol: ditemukan tanpa pedulikan huruf -> None
0xcc6a0dd80ff3fa8c1b37011765ee19efc480f825.sol: ditemukan tanpa pedulikan huruf -> None


In [ ]:
def laporkan_duplikat(nama_versi, entri, hanya_train=True):
    data = [e for e in entri if not e['folder'].startswith('3_expert_benchmark')] if hanya_train else entri
    print(f'\n=== {nama_versi}, {"train saja" if hanya_train else "seluruh termasuk benchmark"}, total {len(data)} ===')
    for metode, kunci in [('hash kode', 'hash_kode'), ('hash struktur', 'hash_struktur')]:
        grup = defaultdict(list)
        n_tanpa_hash_kode = 0
        for e in data:
            if e[kunci] is not None:
                grup[e[kunci]].append(e['nama_sol'])
            elif kunci == 'hash_kode':
                n_tanpa_hash_kode += 1
        grup_dobel = {k: v for k, v in grup.items() if len(v) > 1}
        n_duplikat_entri = sum(len(v) - 1 for v in grup_dobel.values())
        ket = f' ({n_tanpa_hash_kode} entri tidak terhitung karena sumber tidak ketemu)' if n_tanpa_hash_kode else ''
        print(f'  {metode}: {len(grup_dobel)} grup kembar, {n_duplikat_entri} entri berlebih{ket}')
        for k, v in list(grup_dobel.items())[:5]:
            print(f'      {v}')
    return data

data_v1 = laporkan_duplikat('v1', entri_v1)
data_v2 = laporkan_duplikat('v2', entri_v2)
data_v3 = laporkan_duplikat('v3', entri_v3)

print('\n=== tumpang tindih benchmark (16 kontrak CEI) dengan data latih, versi perbaikan ===')
bench_entri = [e for e in entri_v1 if e['folder'].startswith('3_expert_benchmark')]
print(f'jumlah entri benchmark yang benar ditemukan: {len(bench_entri)}')
for nama_versi, data_latih in [('v1', data_v1), ('v2', data_v2), ('v3', data_v3)]:
    for metode, kunci in [('hash kode', 'hash_kode'), ('hash struktur', 'hash_struktur')]:
        set_latih = {e[kunci] for e in data_latih if e[kunci] is not None}
        tabrakan = [b['nama_sol'] for b in bench_entri if b[kunci] in set_latih]
        print(f'  {nama_versi} vs benchmark, {metode}: {len(tabrakan)} tabrakan', tabrakan[:5] if tabrakan else '')

print('\n=== telusuri nama_file asli dari JSON untuk 4 alamat yang hilang di v2 ===')
for nm, nama_sol, status in gagal_v2:
    p = os.path.join(f'{D}/data/processed/v2_cfg', nm)
    with open(p) as f:
        j = _json.load(f)
    print(f'{nm}')
    print(f'    nama_file (dari json): {j.get("nama_file")!r}')
    print(f'    folder: {j.get("folder")!r}  label: {j.get("label")}  split: {j.get("split")}  solc: {j.get("solc")}')


=== v1, train saja, total 744 ===
  hash kode: 24 grup kembar, 46 entri berlebih
      ['0x001831de9b6Ba15207e36A4f2ecA01322E411293.sol', '0x00E3c1F30dC416dBF841435cB1b2188c1A268F7E.sol', '0x00c8187C790fA909C88e20a7c90bC1Ac817B1774.sol', '0x00dE0F192931353837B4D6Bc2208f4b58C1EC5C6.sol', '0x0129602852C366AcaFa0aD4465022125353038F9.sol', '0x0164F85A0D4EA7C2f938cc06d7FA38C69047BF75.sol', '0x017429367D6E4f9FA7B673bDb2247887Bd9233e8.sol', '0x01B7B838b0ffac5E2e438E10C9273b705231f186.sol', '0x01d2240Eedb92e2f26B0820248de849665d93668.sol']
      ['0x002f46770160fFb092bc15699fB035526b0B65d7.sol', '0x00d84b15e2B02EbD501405772C90845ef683B557.sol', '0x0113bF81f431175b7013F3Bd4573479aB170c4c9.sol']
      ['0x0122ce89a7f9a329458A5B7D8612a69Ad187705b.sol', '0x018AA2fADAB45A6b572F23791F90a0e5454CD748.sol', '0x01dB89aEC452d1628ddc364FF8aE6dDe41f3972A.sol', '0x01e5870DE90578Be6A792A82A087750C999d8738.sol']
      ['0x2b154d86f7fe11b3545c3f3c6d76ae1f86dd1199.sol', '0xc4ae575a46b04a0696d0e9437c78c923c2ad0

In [ ]:
print('=== struktur cfg.pt, cfg_prune_0.02.pt, v2_cfg_prune.pt, v2_cfg_prune_pos.pt ===')
for nm in ['cfg.pt', 'cfg_prune_0.02.pt', 'v2_cfg_prune.pt', 'v2_cfg_prune_pos.pt']:
    p = f'{D}/data/processed/pyg/{nm}'
    obj = torch.load(p, map_location='cpu', weights_only=False)
    print(f'\n--- {nm} ---')
    print('tipe:', type(obj).__name__)
    if isinstance(obj, dict):
        print('kunci:', list(obj.keys()))
        for k, v in obj.items():
            if isinstance(v, list) and len(v) and isinstance(v[0], Data):
                ada_titik_dua = sum(1 for d in v if '::' in getattr(d, 'nama', ''))
                print(f'  kunci "{k}": {len(v)} objek Data, atribut:', list(v[0].to_dict().keys()))
                print(f'    entri bertanda subgraf (mengandung "::"): {ada_titik_dua}')
    elif isinstance(obj, list) and len(obj) and isinstance(obj[0], Data):
        ada_titik_dua = sum(1 for d in obj if '::' in getattr(d, 'nama', ''))
        print(f'{len(obj)} objek Data, atribut:', list(obj[0].to_dict().keys()))
        print(f'  entri bertanda subgraf (mengandung "::"): {ada_titik_dua}')

print()
print('=== dampak penghapusan v3 kalau ikut membawa subgraf ===')
hapus_v3_grup = daftar_hapus(entri_v3)[0] if False else None
grup = defaultdict(list)
for e in entri_v3:
    if not e['folder'].startswith('3_expert_benchmark') and e['hash_kode'] is not None:
        grup[e['hash_kode']].append(e['nama_sol'])
hapus_v3_grup = set()
for v in grup.values():
    if len(v) > 1:
        hapus_v3_grup.update(sorted(v)[1:])

print(f'jumlah kontrak (contract_id) yang akan dihapus di v3: {len(hapus_v3_grup)}')

terhapus_dari_ds = [g for g in ds if g.contract_id in hapus_v3_grup]
terhapus_graf_utuh = [g for g in terhapus_dari_ds if '::' not in g.nama]
terhapus_subgraf = [g for g in terhapus_dari_ds if '::' in g.nama]

print(f'total entri yang akan terhapus dari v3_augmented.pt (graf utuh + subgraf): {len(terhapus_dari_ds)}')
print(f'  dari situ, graf utuh: {len(terhapus_graf_utuh)}')
print(f'  dari situ, subgraf fungsi: {len(terhapus_subgraf)}')
print(f'sisa total v3 setelah dedup: {len(ds) - len(terhapus_dari_ds)} (dari {len(ds)})')

=== struktur cfg.pt, cfg_prune_0.02.pt, v2_cfg_prune.pt, v2_cfg_prune_pos.pt ===

--- cfg.pt ---
tipe: dict
kunci: ['train', 'benchmark', 'dim', 'tipe']
  kunci "train": 744 objek Data, atribut: ['x', 'edge_index', 'y', 'edge_type', 'nama', 'split', 'folder']
    entri bertanda subgraf (mengandung "::"): 0
  kunci "benchmark": 16 objek Data, atribut: ['x', 'edge_index', 'y', 'edge_type', 'nama', 'split', 'folder']
    entri bertanda subgraf (mengandung "::"): 0

--- cfg_prune_0.02.pt ---
tipe: dict
kunci: ['train', 'benchmark', 'dim', 'tipe']
  kunci "train": 742 objek Data, atribut: ['x', 'edge_index', 'y', 'edge_type', 'nama', 'split', 'folder']
    entri bertanda subgraf (mengandung "::"): 0
  kunci "benchmark": 16 objek Data, atribut: ['x', 'edge_index', 'y', 'edge_type', 'nama', 'split', 'folder']
    entri bertanda subgraf (mengandung "::"): 0

--- v2_cfg_prune.pt ---
tipe: dict
kunci: ['train', 'benchmark', 'dim', 'tipe']
  kunci "train": 610 objek Data, atribut: ['x', 'edge_ind

In [ ]:
print('=== (1) Jumlah graf utuh dan subgraf di v3, dihitung langsung dari atribut berkas ===')
n_graf_utuh = sum(1 for g in ds if '::' not in g.nama)
n_subgraf = sum(1 for g in ds if '::' in g.nama)
print(f'graf utuh : {n_graf_utuh}')
print(f'subgraf   : {n_subgraf}')
print(f'total     : {n_graf_utuh + n_subgraf}  (panjang ds sebenarnya: {len(ds)})')

print()
print('=== (2) Aturan pemilihan wakil per grup kembar, dan tabel seluruhnya ===')
print('Aturan: anggota grup diurutkan alfabet berdasarkan nama kontrak,')
print('urutan pertama dipertahankan, sisanya dibuang.')
print()

def tabel_grup(nama_versi, entri):
    data = [e for e in entri if not e['folder'].startswith('3_expert_benchmark')]
    grup = defaultdict(list)
    for e in data:
        if e['hash_kode'] is not None:
            grup[e['hash_kode']].append(e['nama_sol'])
    grup_dobel = {k: v for k, v in grup.items() if len(v) > 1}
    print(f'--- {nama_versi}, {len(grup_dobel)} grup kembar, diurutkan dari grup terbesar ---')
    for i, (h, anggota) in enumerate(sorted(grup_dobel.items(), key=lambda x: -len(x[1])), start=1):
        urut = sorted(anggota)
        print(f'  grup {i:>2} ({len(anggota)} anggota)  dipertahankan: {urut[0]}')
        for d in urut[1:]:
            print(f'           dibuang      : {d}')
    return grup_dobel

grup_v1 = tabel_grup('v1', entri_v1)
print()
grup_v2 = tabel_grup('v2', entri_v2)
print()
grup_v3 = tabel_grup('v3', entri_v3)

print()
print('=== (3) Subgraf rentan vs aman di v3, sebelum dan sesudah pembuangan kaskade ===')
subgraf_semua = [g for g in ds if '::' in g.nama]
sebelum = Counter(int(g.y) for g in subgraf_semua)
sesudah = Counter(int(g.y) for g in subgraf_semua if g.contract_id not in hapus_v3_grup)
print(f'sebelum -> aman: {sebelum[0]}, rentan: {sebelum[1]}, total: {sum(sebelum.values())}')
print(f'sesudah -> aman: {sesudah[0]}, rentan: {sesudah[1]}, total: {sum(sesudah.values())}')
print(f'dibuang -> aman: {sebelum[0]-sesudah[0]}, rentan: {sebelum[1]-sesudah[1]}')

print()
print('=== (4) selisih JSON mentah vs berkas .pt yang sebenarnya dipakai training, v1 dan v2 ===')
nama_json_v1 = {e['nama_sol'] for e in entri_v1 if not e['folder'].startswith('3_expert_benchmark')}
obj_v1p = torch.load(f'{D}/data/processed/pyg/cfg_prune_0.02.pt', map_location='cpu', weights_only=False)
nama_pt_v1 = {g.nama for g in obj_v1p['train']}
print(f'v1, json: {len(nama_json_v1)}, pt pruned: {len(nama_pt_v1)}')
print('  di json tapi tidak di pt:', nama_json_v1 - nama_pt_v1)
print('  di pt tapi tidak di json:', nama_pt_v1 - nama_json_v1)

nama_json_v2 = {e['nama_sol'] for e in entri_v2 if not e['folder'].startswith('3_expert_benchmark')}
obj_v2p = torch.load(f'{D}/data/processed/pyg/v2_cfg_prune.pt', map_location='cpu', weights_only=False)
obj_v2pp = torch.load(f'{D}/data/processed/pyg/v2_cfg_prune_pos.pt', map_location='cpu', weights_only=False)
nama_pt_v2 = {g.nama for g in obj_v2p['train']}
nama_pt_v2pos = {g.nama for g in obj_v2pp['train']}
print(f'v2, json: {len(nama_json_v2)}, v2_cfg_prune.pt: {len(nama_pt_v2)}, v2_cfg_prune_pos.pt: {len(nama_pt_v2pos)}')
print('  v2_cfg_prune.pt == v2_cfg_prune_pos.pt ?', nama_pt_v2 == nama_pt_v2pos)
print('  di json tapi tidak di v2_cfg_prune.pt:', nama_json_v2 - nama_pt_v2)
print('  di v2_cfg_prune.pt tapi tidak di json:', nama_pt_v2 - nama_json_v2)

=== (1) Jumlah graf utuh dan subgraf di v3, dihitung langsung dari atribut berkas ===
graf utuh : 607
subgraf   : 1105
total     : 1712  (panjang ds sebenarnya: 1712)

=== (2) Aturan pemilihan wakil per grup kembar, dan tabel seluruhnya ===
Aturan: anggota grup diurutkan alfabet berdasarkan nama kontrak,
urutan pertama dipertahankan, sisanya dibuang.

--- v1, 24 grup kembar, diurutkan dari grup terbesar ---
  grup  1 (9 anggota)  dipertahankan: 0x001831de9b6Ba15207e36A4f2ecA01322E411293.sol
           dibuang      : 0x00E3c1F30dC416dBF841435cB1b2188c1A268F7E.sol
           dibuang      : 0x00c8187C790fA909C88e20a7c90bC1Ac817B1774.sol
           dibuang      : 0x00dE0F192931353837B4D6Bc2208f4b58C1EC5C6.sol
           dibuang      : 0x0129602852C366AcaFa0aD4465022125353038F9.sol
           dibuang      : 0x0164F85A0D4EA7C2f938cc06d7FA38C69047BF75.sol
           dibuang      : 0x017429367D6E4f9FA7B673bDb2247887Bd9233e8.sol
           dibuang      : 0x01B7B838b0ffac5E2e438E10C9273b705231f1

In [ ]:
print('=== cek label per grup kembar, v1/v2/v3 ===')
def cek_label_grup(nama_versi, entri):
    peta_label = {e['nama_sol']: e['label'] for e in entri}
    data = [e for e in entri if not e['folder'].startswith('3_expert_benchmark')]
    grup = defaultdict(list)
    for e in data:
        if e['hash_kode'] is not None:
            grup[e['hash_kode']].append(e['nama_sol'])
    grup_dobel = {k: v for k, v in grup.items() if len(v) > 1}
    label_per_grup = Counter()
    campur = []
    for anggota in grup_dobel.values():
        labels = {peta_label[a] for a in anggota}
        if len(labels) > 1:
            campur.append(anggota)
        else:
            label_per_grup[labels.pop()] += 1
    print(f'{nama_versi}: grup label aman={label_per_grup[0]}, grup label rentan={label_per_grup[1]}, grup label campur={len(campur)}')
    for c in campur:
        print('  CAMPUR:', c)

cek_label_grup('v1', entri_v1)
cek_label_grup('v2', entri_v2)
cek_label_grup('v3', entri_v3)

print()
print('=== pastikan daftar hapus tidak bersinggungan dengan entri yang hilang di pt ===')
print('v1, beririsan dengan 2 nama yang hilang di pt?', hapus_v1 & {'0x0000000000a84fe7f5d858c8a22121c975ff0b42.sol', '0x000000000000cd17345801aa8147b8D3950260FF.sol'})
print('v2, beririsan dengan nama yang hilang di pt?', hapus_v2 & {'0x04d79f02e8d529CCEDe8393413973a9AceD36213.sol'})

print()
print('=== tulis berkas dedup, v1 ===')
obj_v1p = torch.load(f'{D}/data/processed/pyg/cfg_prune_0.02.pt', map_location='cpu', weights_only=False)
train_v1_sebelum = obj_v1p['train']
hapus_valid_v1 = hapus_v1 & {g.nama for g in train_v1_sebelum}
print(f'nama di daftar hapus yang memang ada di pt: {len(hapus_valid_v1)} dari {len(hapus_v1)}')
train_v1_sesudah = [g for g in train_v1_sebelum if g.nama not in hapus_valid_v1]
print(f'v1 train: {len(train_v1_sebelum)} -> {len(train_v1_sesudah)}')

obj_v1_dedup = {'train': train_v1_sesudah, 'benchmark': obj_v1p['benchmark'], 'dim': obj_v1p['dim'], 'tipe': obj_v1p['tipe']}
torch.save(obj_v1_dedup, '/content/cfg_prune_0.02_dedup.pt')
!cp /content/cfg_prune_0.02_dedup.pt "{D}/data/processed/pyg/cfg_prune_0.02_dedup.pt"
ukuran_asli = os.path.getsize(f'{D}/data/processed/pyg/cfg_prune_0.02.pt')
ukuran_baru = os.path.getsize(f'{D}/data/processed/pyg/cfg_prune_0.02_dedup.pt')
print(f'tersimpan, ukuran asli {ukuran_asli/1e6:.2f} MB, ukuran dedup {ukuran_baru/1e6:.2f} MB')

print()
print('=== tulis berkas dedup, v2 (dua varian) ===')
for nama_berkas in ['v2_cfg_prune.pt', 'v2_cfg_prune_pos.pt']:
    obj = torch.load(f'{D}/data/processed/pyg/{nama_berkas}', map_location='cpu', weights_only=False)
    train_sebelum = obj['train']
    hapus_valid_v2 = hapus_v2 & {g.nama for g in train_sebelum}
    train_sesudah = [g for g in train_sebelum if g.nama not in hapus_valid_v2]
    print(f'{nama_berkas}: {len(train_sebelum)} -> {len(train_sesudah)} (hapus valid {len(hapus_valid_v2)} dari {len(hapus_v2)})')
    obj_dedup = {'train': train_sesudah, 'benchmark': obj['benchmark'], 'dim': obj['dim'], 'tipe': obj['tipe']}
    nama_dedup = nama_berkas.replace('.pt', '_dedup.pt')
    torch.save(obj_dedup, f'/content/{nama_dedup}')
    !cp /content/{nama_dedup} "{D}/data/processed/pyg/{nama_dedup}"
    ua = os.path.getsize(f'{D}/data/processed/pyg/{nama_berkas}')
    ub = os.path.getsize(f'{D}/data/processed/pyg/{nama_dedup}')
    print(f'  tersimpan, ukuran asli {ua/1e6:.2f} MB, ukuran dedup {ub/1e6:.2f} MB')

print()
print('=== tulis berkas dedup, v3 (ikut kaskade ke subgraf) ===')
for g in ds:
    if not hasattr(g, 'contract_id'):
        g.contract_id = ambil_contract_id(g.nama)
train_v3_sesudah = [g for g in ds if g.contract_id not in hapus_v3_grup]
print(f'v3 train: {len(ds)} -> {len(train_v3_sesudah)}')

obj_v3 = torch.load(f'{D}/data/processed/pyg/v3_augmented.pt', map_location='cpu', weights_only=False)
obj_v3_dedup = {'train': train_v3_sesudah, 'benchmark': obj_v3['benchmark'], 'dim': obj_v3['dim'], 'tipe': obj_v3['tipe']}
torch.save(obj_v3_dedup, '/content/v3_augmented_dedup.pt')
!cp /content/v3_augmented_dedup.pt "{D}/data/processed/pyg/v3_augmented_dedup.pt"
ua = os.path.getsize(f'{D}/data/processed/pyg/v3_augmented.pt')
ub = os.path.getsize(f'{D}/data/processed/pyg/v3_augmented_dedup.pt')
print(f'tersimpan, ukuran asli {ua/1e6:.2f} MB, ukuran dedup {ub/1e6:.2f} MB')

print()
print('=== ringkasan akhir sebelum dan sesudah, berdasarkan berkas .pt sebenarnya ===')
print(f'v1 (cfg_prune_0.02.pt)      : {len(train_v1_sebelum)} -> {len(train_v1_sesudah)}')
print(f'v2 (v2_cfg_prune.pt)        : {len(train_sebelum) if nama_berkas=="v2_cfg_prune_pos.pt" else "?"}')
print(f'v3 (v3_augmented.pt)        : {len(ds)} -> {len(train_v3_sesudah)}')

=== cek label per grup kembar, v1/v2/v3 ===
v1: grup label aman=9, grup label rentan=15, grup label campur=0
v2: grup label aman=17, grup label rentan=15, grup label campur=0
v3: grup label aman=17, grup label rentan=15, grup label campur=0

=== pastikan daftar hapus tidak bersinggungan dengan entri yang hilang di pt ===
v1, beririsan dengan 2 nama yang hilang di pt? set()
v2, beririsan dengan nama yang hilang di pt? set()

=== tulis berkas dedup, v1 ===
nama di daftar hapus yang memang ada di pt: 46 dari 46
v1 train: 742 -> 696
tersimpan, ukuran asli 28.16 MB, ukuran dedup 21.67 MB

=== tulis berkas dedup, v2 (dua varian) ===
v2_cfg_prune.pt: 610 -> 542 (hapus valid 68 dari 68)
  tersimpan, ukuran asli 31.17 MB, ukuran dedup 23.92 MB
v2_cfg_prune_pos.pt: 610 -> 542 (hapus valid 68 dari 68)
  tersimpan, ukuran asli 32.29 MB, ukuran dedup 24.78 MB

=== tulis berkas dedup, v3 (ikut kaskade ke subgraf) ===
v3 train: 1712 -> 1452
tersimpan, ukuran asli 49.55 MB, ukuran dedup 38.62 MB

=== 

In [ ]:
from sklearn.model_selection import StratifiedGroupKFold

SEED = 42

def siapkan_split(train_list, nama_grup_attr):
    y = np.array([int(g.y) for g in train_list])
    groups = np.array([getattr(g, nama_grup_attr) for g in train_list])
    return y, groups

def jalankan_dan_laporkan(nama_versi, train_list, nama_grup_attr):
    y, groups = siapkan_split(train_list, nama_grup_attr)
    skf = StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=SEED)
    print(f'=== {nama_versi}, total {len(train_list)} graf, {len(set(groups))} grup unik ===')
    for fold, (tr, va) in enumerate(skf.split(np.zeros(len(train_list)), y, groups)):
        assert set(groups[tr]).isdisjoint(set(groups[va])), f"bocor pada fold {fold}, versi {nama_versi}"
        y_tr, y_va = y[tr], y[va]
        print(f'  fold {fold}: latih {len(tr):>4} graf ({len(set(groups[tr])):>4} grup, aman={np.sum(y_tr==0):>4}, rentan={np.sum(y_tr==1):>4})'
              f'  |  validasi {len(va):>4} graf ({len(set(groups[va])):>3} grup, aman={np.sum(y_va==0):>3}, rentan={np.sum(y_va==1):>3})')
    print('  assertion lolos untuk seluruh fold, tidak ada grup yang terbelah ke dua sisi\n')

print('memuat dataset dedup...')
obj_v1 = torch.load(f'{D}/data/processed/pyg/cfg_prune_0.02_dedup.pt', map_location='cpu', weights_only=False)
obj_v2 = torch.load(f'{D}/data/processed/pyg/v2_cfg_prune_dedup.pt', map_location='cpu', weights_only=False)
obj_v3 = torch.load(f'{D}/data/processed/pyg/v3_augmented_dedup.pt', map_location='cpu', weights_only=False)

for g in obj_v3['train']:
    if not hasattr(g, 'contract_id'):
        g.contract_id = ambil_contract_id(g.nama)

jalankan_dan_laporkan('v1 (cfg_prune_0.02_dedup)', obj_v1['train'], 'nama')
jalankan_dan_laporkan('v2 (v2_cfg_prune_dedup)', obj_v2['train'], 'nama')
jalankan_dan_laporkan('v3 (v3_augmented_dedup)', obj_v3['train'], 'contract_id')

print('catatan: v2_cfg_prune_pos_dedup.pt punya nama graf identik dengan v2_cfg_prune_dedup.pt,')
print('jadi pembagian foldnya otomatis sama, tidak perlu dijalankan terpisah.')

memuat dataset dedup...
=== v1 (cfg_prune_0.02_dedup), total 696 graf, 696 grup unik ===
  fold 0: latih  557 graf ( 557 grup, aman= 380, rentan= 177)  |  validasi  139 graf (139 grup, aman= 99, rentan= 40)
  fold 1: latih  556 graf ( 556 grup, aman= 379, rentan= 177)  |  validasi  140 graf (140 grup, aman=100, rentan= 40)
  fold 2: latih  557 graf ( 557 grup, aman= 388, rentan= 169)  |  validasi  139 graf (139 grup, aman= 91, rentan= 48)
  fold 3: latih  557 graf ( 557 grup, aman= 383, rentan= 174)  |  validasi  139 graf (139 grup, aman= 96, rentan= 43)
  fold 4: latih  557 graf ( 557 grup, aman= 386, rentan= 171)  |  validasi  139 graf (139 grup, aman= 93, rentan= 46)
  assertion lolos untuk seluruh fold, tidak ada grup yang terbelah ke dua sisi

=== v2 (v2_cfg_prune_dedup), total 542 graf, 542 grup unik ===
  fold 0: latih  433 graf ( 433 grup, aman= 258, rentan= 175)  |  validasi  109 graf (109 grup, aman= 67, rentan= 42)
  fold 1: latih  434 graf ( 434 grup, aman= 258, rentan= 176

In [ ]:
import torch.nn as nn
import torch.nn.functional as F
from torch_geometric.nn import GATConv, global_mean_pool
from torch_geometric.loader import DataLoader
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score
import json, time

class MultiScaleGAT_MeanPool(nn.Module):
    def __init__(self, dim_input=22, dim_hidden=512, n_kelas=2, dropout=0.5):
        super().__init__()
        h = dim_hidden
        self.gat1 = GATConv(dim_input, h // 8, heads=8, dropout=dropout)
        self.gat2 = GATConv(h, h // 4, heads=4, dropout=dropout)
        self.gat3 = GATConv(h, h // 4, heads=4, dropout=dropout)
        self.proj = nn.Linear(dim_input, h)
        self.norm1 = nn.LayerNorm(h)
        self.norm2 = nn.LayerNorm(h)
        self.norm3 = nn.LayerNorm(h)
        self.classifier = nn.Sequential(
            nn.Linear(h, 128), nn.ReLU(), nn.Dropout(dropout),
            nn.Linear(128, 64), nn.ReLU(), nn.Dropout(dropout),
            nn.Linear(64, n_kelas),
        )
        self.dropout = dropout
    def encode(self, x, edge_index, batch):
        h0 = self.proj(x)
        h1 = self.norm1(F.elu(self.gat1(x, edge_index)) + h0)
        h1 = F.dropout(h1, p=self.dropout, training=self.training)
        h2 = self.norm2(F.elu(self.gat2(h1, edge_index)) + h1)
        h2 = F.dropout(h2, p=self.dropout, training=self.training)
        h3 = self.norm3(F.elu(self.gat3(h2, edge_index)) + h2)
        return global_mean_pool(h3, batch)
    def forward(self, data):
        z = self.encode(data.x, data.edge_index, data.batch)
        return self.classifier(z)

DEV = 'cuda' if torch.cuda.is_available() else 'cpu'

def latih_satu_fold(train_list, idx_tr, idx_va, dim_input, seed):
    torch.manual_seed(seed)
    model = MultiScaleGAT_MeanPool(dim_input=dim_input).to(DEV)
    y_tr = np.array([int(train_list[i].y) for i in idx_tr])
    bobot = torch.tensor([1.0 / max((y_tr == 0).sum(), 1), 1.0 / max((y_tr == 1).sum(), 1)], dtype=torch.float).to(DEV)
    bobot = bobot / bobot.sum() * 2
    opt = torch.optim.AdamW(model.parameters(), lr=0.001)
    lf = nn.CrossEntropyLoss(weight=bobot)

    dl_tr = DataLoader([train_list[i] for i in idx_tr], batch_size=32, shuffle=True)
    dl_va = DataLoader([train_list[i] for i in idx_va], batch_size=64)

    terbaik_f1, terbaik_state, sabar = -1, None, 0
    for epoch in range(50):
        model.train()
        for b in dl_tr:
            b = b.to(DEV)
            opt.zero_grad()
            out = model(b)
            loss = lf(out, b.y)
            loss.backward()
            opt.step()

        model.eval()
        pred_va, y_va = [], []
        with torch.no_grad():
            for b in dl_va:
                b = b.to(DEV)
                p = torch.argmax(model(b), 1).cpu().numpy()
                pred_va.extend(p.tolist())
                y_va.extend(b.y.cpu().numpy().tolist())
        f1 = f1_score(y_va, pred_va, zero_division=0)
        if f1 > terbaik_f1:
            terbaik_f1, terbaik_state, sabar = f1, {k: v.clone() for k, v in model.state_dict().items()}, 0
        else:
            sabar += 1
            if sabar >= 15:
                break

    model.load_state_dict(terbaik_state)
    return model, terbaik_f1

def evaluasi(model, data_list):
    model.eval()
    dl = DataLoader(data_list, batch_size=64)
    prob, pred, y_asli = [], [], []
    with torch.no_grad():
        for b in dl:
            b = b.to(DEV)
            out = torch.softmax(model(b), 1)
            prob.extend(out[:, 1].cpu().numpy().tolist())
            pred.extend(torch.argmax(out, 1).cpu().numpy().tolist())
            y_asli.extend(b.y.cpu().numpy().tolist())
    return np.array(prob), np.array(pred), np.array(y_asli)

def latih_konfigurasi(kode, train_list, bench_list, dim_input, seed, grup_attr='nama'):
    waktu_mulai = time.time()
    y = np.array([int(g.y) for g in train_list])
    groups = np.array([getattr(g, grup_attr) for g in train_list])
    skf = StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=seed)

    metrik_fold, model_tiap_fold, prob_bench_tiap_fold = [], [], []
    for fold, (tr, va) in enumerate(skf.split(np.zeros(len(train_list)), y, groups)):
        assert set(groups[tr]).isdisjoint(set(groups[va])), f"bocor di fold {fold}"
        model, f1_val_terbaik = latih_satu_fold(train_list, tr, va, dim_input, seed)

        prob_va, pred_va, y_va = evaluasi(model, [train_list[i] for i in va])
        metrik_fold.append({
            'fold': fold, 'accuracy': accuracy_score(y_va, pred_va),
            'precision': precision_score(y_va, pred_va, zero_division=0),
            'recall': recall_score(y_va, pred_va, zero_division=0),
            'f1': f1_score(y_va, pred_va, zero_division=0),
            'auc': roc_auc_score(y_va, prob_va) if len(set(y_va)) > 1 else float('nan'),
        })
        model_tiap_fold.append({k: v.cpu() for k, v in model.state_dict().items()})

        prob_b, _, _ = evaluasi(model, bench_list)
        prob_bench_tiap_fold.append(prob_b)
        print(f'  {kode} fold {fold}: F1 val {metrik_fold[-1]["f1"]:.4f}, AUC val {metrik_fold[-1]["auc"]:.4f}')

    df_metrik = pd.DataFrame(metrik_fold)
    rata = df_metrik.drop(columns='fold').mean()
    rata['fold'] = 'rata_rata'
    df_metrik = pd.concat([df_metrik, pd.DataFrame([rata])], ignore_index=True)

    prob_bench_rata = np.mean(prob_bench_tiap_fold, axis=0)
    df_bench = pd.DataFrame({
        'nama_kontrak': [g.nama for g in bench_list],
        'label': [int(g.y) for g in bench_list],
        'jumlah_node': [g.num_nodes for g in bench_list],
        'probabilitas_rentan': prob_bench_rata,
    })

    konfigurasi = {
        'kode': kode, 'seed': seed, 'dim_input': dim_input, 'dim_hidden': 512,
        'optimizer': 'AdamW', 'lr': 0.001, 'batch_size': 32, 'max_epoch': 50, 'patience': 15,
        'n_splits': 5, 'grup_attr': grup_attr, 'n_train': len(train_list),
        'torch_version': torch.__version__, 'torch_geometric_version': torch_geometric.__version__,
        'waktu_training_detik': round(time.time() - waktu_mulai, 1),
    }

    return df_metrik, df_bench, model_tiap_fold, konfigurasi

def simpan_hasil(kode, seed, df_metrik, df_bench, model_tiap_fold, konfigurasi):
    tag = f'{kode}_seed{seed}'
    df_metrik.to_csv(f'/content/{tag}_fold_metrics.csv', index=False)
    df_bench.to_csv(f'/content/{tag}_benchmark_preds.csv', index=False, float_format='%.10f')
    with open(f'/content/{tag}_config.json', 'w') as f:
        json.dump(konfigurasi, f, indent=2)
    torch.save(model_tiap_fold, f'/content/{tag}_model_tiap_fold.pt')

    for nm in [f'{tag}_fold_metrics.csv', f'{tag}_benchmark_preds.csv', f'{tag}_config.json', f'{tag}_model_tiap_fold.pt']:
        get_ipython().system(f'cp /content/{nm} "{D}/hasil/{nm}"')
        ukuran = os.path.getsize(f'{D}/hasil/{nm}')
        print(f'  tersimpan {nm}  ({ukuran/1e6:.3f} MB)')

print('=== C3, mean pooling, class-weighted, data v2 dedup, seed 42 ===')
df_m, df_b, models, konf = latih_konfigurasi('C3', obj_v2['train'], obj_v2['benchmark'], dim_input=22, seed=42, grup_attr='nama')
print(df_m)
simpan_hasil('C3', 42, df_m, df_b, models, konf)
print('selesai C3')

=== C3, mean pooling, class-weighted, data v2 dedup, seed 42 ===
  C3 fold 0: F1 val 0.9286, AUC val 0.9559
  C3 fold 1: F1 val 0.8718, AUC val 0.9068
  C3 fold 2: F1 val 0.8571, AUC val 0.9354
  C3 fold 3: F1 val 0.8317, AUC val 0.8902
  C3 fold 4: F1 val 0.8608, AUC val 0.9377
        fold  accuracy  precision    recall        f1       auc
0          0  0.944954   0.928571  0.928571  0.928571  0.955935
1          1  0.907407   0.918919  0.829268  0.871795  0.906807
2          2  0.879630   0.829787  0.886364  0.857143  0.935369
3          3  0.844037   0.777778  0.893617  0.831683  0.890185
4          4  0.898148   0.944444  0.790698  0.860759  0.937746
5  rata_rata  0.894835   0.879900  0.865704  0.869990  0.925209
  tersimpan C3_seed42_fold_metrics.csv  (0.001 MB)
  tersimpan C3_seed42_benchmark_preds.csv  (0.001 MB)
  tersimpan C3_seed42_config.json  (0.000 MB)
  tersimpan C3_seed42_model_tiap_fold.pt  (12.622 MB)
selesai C3


In [ ]:
print('=== C5, mean pooling, class-weighted, data v3 dedup, seed 42 ===')
df_m, df_b, models, konf = latih_konfigurasi('C5', obj_v3['train'], obj_v3['benchmark'], dim_input=22, seed=42, grup_attr='contract_id')
print(df_m)
simpan_hasil('C5', 42, df_m, df_b, models, konf)
print('selesai C5')

=== C5, mean pooling, class-weighted, data v3 dedup, seed 42 ===
  C5 fold 0: F1 val 0.8201, AUC val 0.8995
  C5 fold 1: F1 val 0.8374, AUC val 0.9022
  C5 fold 2: F1 val 0.9113, AUC val 0.9582
  C5 fold 3: F1 val 0.8643, AUC val 0.9364
  C5 fold 4: F1 val 0.8775, AUC val 0.9421
        fold  accuracy  precision    recall        f1       auc
0          0  0.834983   0.838235  0.802817  0.820144  0.899528
1          1  0.860627   0.792308  0.887931  0.837398  0.902198
2          2  0.923875   0.926230  0.896825  0.911290  0.958224
3          3  0.906574   0.851485  0.877551  0.864322  0.936425
4          4  0.890845   0.867188  0.888000  0.877470  0.942088
5  rata_rata  0.883381   0.855089  0.870625  0.862125  0.927692
  tersimpan C5_seed42_fold_metrics.csv  (0.001 MB)
  tersimpan C5_seed42_benchmark_preds.csv  (0.001 MB)
  tersimpan C5_seed42_config.json  (0.000 MB)
  tersimpan C5_seed42_model_tiap_fold.pt  (12.622 MB)
selesai C5


In [ ]:
import os, copy, time, json, shutil, subprocess, sys
if not os.path.exists('/content/drive/MyDrive'):
    from google.colab import drive
    drive.mount('/content/drive')
try:
    import torch_geometric
except ImportError:
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'torch_geometric'])

import numpy as np, pandas as pd, sklearn
import torch, torch.nn as nn, torch.nn.functional as F
import torch_geometric
from torch_geometric.loader import DataLoader
from torch_geometric.nn import GATConv, SAGPooling, global_mean_pool
from sklearn.model_selection import StratifiedGroupKFold
from sklearn.metrics import (accuracy_score, precision_score, recall_score,
                             f1_score, roc_auc_score)

D = '/content/drive/MyDrive/PROJECT'
DEV = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
SEED = 42
def ambil_contract_id(nama): return nama.split('::')[0]

# ---------- model (salinan persis dari notebook asli) ----------
class MultiScaleGAT_MeanPool(nn.Module):
    def __init__(self, dim_input=22, dim_hidden=512, n_kelas=2, dropout=0.5):
        super().__init__()
        h = dim_hidden
        self.gat1 = GATConv(dim_input, h // 8, heads=8, dropout=dropout)
        self.gat2 = GATConv(h, h // 4, heads=4, dropout=dropout)
        self.gat3 = GATConv(h, h // 4, heads=4, dropout=dropout)
        self.proj = nn.Linear(dim_input, h)
        self.norm1 = nn.LayerNorm(h); self.norm2 = nn.LayerNorm(h); self.norm3 = nn.LayerNorm(h)
        self.classifier = nn.Sequential(
            nn.Linear(h, 128), nn.ReLU(), nn.Dropout(dropout),
            nn.Linear(128, 64), nn.ReLU(), nn.Dropout(dropout),
            nn.Linear(64, n_kelas))
        self.dropout = dropout
    def encode(self, x, edge_index, batch):
        h0 = self.proj(x)
        h1 = self.norm1(F.elu(self.gat1(x, edge_index)) + h0)
        h1 = F.dropout(h1, p=self.dropout, training=self.training)
        h2 = self.norm2(F.elu(self.gat2(h1, edge_index)) + h1)
        h2 = F.dropout(h2, p=self.dropout, training=self.training)
        h3 = self.norm3(F.elu(self.gat3(h2, edge_index)) + h2)
        return global_mean_pool(h3, batch)
    def forward(self, data):
        return self.classifier(self.encode(data.x, data.edge_index, data.batch))

class GAT_SingleScale(nn.Module):
    def __init__(self, dim_input=22, dim_hidden=512, n_kelas=2, dropout=0.5):
        super().__init__()
        h = dim_hidden
        self.gat1 = GATConv(dim_input, h // 8, heads=8, dropout=dropout)
        self.proj = nn.Linear(dim_input, h)
        self.norm1 = nn.LayerNorm(h)
        self.classifier = nn.Sequential(
            nn.Linear(h, 128), nn.ReLU(), nn.Dropout(dropout),
            nn.Linear(128, 64), nn.ReLU(), nn.Dropout(dropout),
            nn.Linear(64, n_kelas))
        self.dropout = dropout
    def encode(self, x, edge_index, batch):
        h0 = self.proj(x)
        h1 = self.norm1(F.elu(self.gat1(x, edge_index)) + h0)
        h1 = F.dropout(h1, p=self.dropout, training=self.training)
        return global_mean_pool(h1, batch)
    def forward(self, data):
        return self.classifier(self.encode(data.x, data.edge_index, data.batch))

class MultiScaleGAT_SAGPool(nn.Module):
    def __init__(self, dim_input=22, dim_hidden=512, n_kelas=2, dropout=0.5):
        super().__init__()
        h = dim_hidden
        self.gat1 = GATConv(dim_input, h // 8, heads=8, dropout=dropout)
        self.gat2 = GATConv(h, h // 4, heads=4, dropout=dropout)
        self.gat3 = GATConv(h, h // 4, heads=4, dropout=dropout)
        self.proj = nn.Linear(dim_input, h)
        self.norm1 = nn.LayerNorm(h); self.norm2 = nn.LayerNorm(h); self.norm3 = nn.LayerNorm(h)
        self.pool = SAGPooling(h, ratio=0.5)
        self.classifier = nn.Sequential(
            nn.Linear(h, 128), nn.ReLU(), nn.Dropout(dropout),
            nn.Linear(128, 64), nn.ReLU(), nn.Dropout(dropout),
            nn.Linear(64, n_kelas))
        self.dropout = dropout
    def encode(self, x, edge_index, batch):
        h0 = self.proj(x)
        h1 = self.norm1(F.elu(self.gat1(x, edge_index)) + h0)
        h1 = F.dropout(h1, p=self.dropout, training=self.training)
        h2 = self.norm2(F.elu(self.gat2(h1, edge_index)) + h1)
        h2 = F.dropout(h2, p=self.dropout, training=self.training)
        h3 = self.norm3(F.elu(self.gat3(h2, edge_index)) + h2)
        x_p, ei_p, _, batch_p, _, _ = self.pool(h3, edge_index, batch=batch)
        return global_mean_pool(x_p, batch_p)
    def forward(self, data):
        return self.classifier(self.encode(data.x, data.edge_index, data.batch))

# ---------- GraphSMOTE (salinan persis) ----------
def graphsmote_augment(h, y, minoritas=1, k=5, target_ratio=1.0):
    idx_min = (y == minoritas).nonzero(as_tuple=True)[0]
    n_min = len(idx_min); n_maj = len(y) - n_min
    n_sintesis = max(0, int(n_maj * target_ratio) - n_min)
    if n_min < 2 or n_sintesis == 0:
        return h, y
    h_min = h[idx_min]
    jarak = torch.cdist(h_min, h_min)
    jarak.fill_diagonal_(float('inf'))
    k_aktual = min(k, n_min - 1)
    _, tetangga_idx = torch.topk(jarak, k_aktual, largest=False, dim=1)
    h_sintesis = []
    for _ in range(n_sintesis):
        i = torch.randint(0, n_min, (1,)).item()
        j = tetangga_idx[i, torch.randint(0, k_aktual, (1,)).item()].item()
        delta = torch.rand(1, device=h.device).item()
        h_sintesis.append(h_min[i] + delta * (h_min[j] - h_min[i]))
    h_sintesis = torch.stack(h_sintesis)
    y_sintesis = torch.full((n_sintesis,), minoritas, dtype=y.dtype, device=y.device)
    return torch.cat([h, h_sintesis], 0), torch.cat([y, y_sintesis], 0)

# ---------- training ----------
def bobot_kelas(dataset):
    y = np.array([int(d.y) for d in dataset])
    n0, n1 = (y == 0).sum(), (y == 1).sum()
    return torch.tensor([len(y) / (2 * n0), len(y) / (2 * n1)], dtype=torch.float).to(DEV)

@torch.no_grad()
def prediksi(model, data_list, batch_size=32):
    model.eval(); probs, preds = [], []
    for b in DataLoader(data_list, batch_size=batch_size):
        b = b.to(DEV); logit = model(b)
        probs += torch.softmax(logit, 1)[:, 1].cpu().tolist()
        preds += logit.argmax(1).cpu().tolist()
    return np.array(probs), np.array(preds)

def hitung_metrik(y, prob, pred):
    m = {'accuracy': accuracy_score(y, pred), 'precision': precision_score(y, pred, zero_division=0),
         'recall': recall_score(y, pred, zero_division=0), 'f1': f1_score(y, pred, zero_division=0)}
    try: m['auc'] = roc_auc_score(y, prob)
    except Exception: m['auc'] = float('nan')
    return m

def latih_satu_fold(tr, va, ModelClass, mode, dim_input, seed, max_epoch=50, sabar=15, lr=1e-3, batch_size=32):
    torch.manual_seed(seed)
    model = ModelClass(dim_input=dim_input, dim_hidden=512).to(DEV)
    opt = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=1e-4)
    lossf = nn.CrossEntropyLoss(weight=bobot_kelas(tr)) if mode == 'weighted' else nn.CrossEntropyLoss()
    ld_tr = DataLoader(tr, batch_size=batch_size, shuffle=True)
    y_va = np.array([int(d.y) for d in va])
    f1_terbaik, bobot_terbaik, mandek, ep_terbaik = -1, None, 0, 0
    for ep in range(1, max_epoch + 1):
        model.train()
        for b in ld_tr:
            b = b.to(DEV); opt.zero_grad()
            if mode == 'smote':
                z = model.encode(b.x, b.edge_index, b.batch)
                z_aug, y_aug_b = graphsmote_augment(z, b.y, k=5)
                loss = lossf(model.classifier(z_aug), y_aug_b)
            else:
                loss = lossf(model(b), b.y)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            opt.step()
        prob, pred = prediksi(model, va)
        f1 = f1_score(y_va, pred, zero_division=0)
        if f1 > f1_terbaik:
            f1_terbaik, bobot_terbaik, mandek, ep_terbaik = f1, copy.deepcopy(model.state_dict()), 0, ep
        else:
            mandek += 1
        if mandek >= sabar: break
    model.load_state_dict(bobot_terbaik)
    return model, ep_terbaik, ep

def latih_konfigurasi(kode, mode, ModelClass, data_list, bench_list, dim_input, seed, grup_attr):
    y = np.array([int(d.y) for d in data_list])
    grup = np.array([getattr(d, grup_attr) for d in data_list])
    cv = StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=seed)
    metrik_rows, val_rows, bench_probs, states, info_fold = [], [], [], [], []
    t0 = time.time()
    for k, (i_tr, i_va) in enumerate(cv.split(np.zeros(len(y)), y, grup), 1):
        assert set(grup[i_tr]).isdisjoint(set(grup[i_va])), 'GRUP BOCOR ANTAR FOLD'
        tr = [data_list[i] for i in i_tr]; va = [data_list[i] for i in i_va]
        model, ep_best, ep_stop = latih_satu_fold(tr, va, ModelClass, mode, dim_input, seed)
        prob, pred = prediksi(model, va)
        y_va = y[i_va]
        m = hitung_metrik(y_va, prob, pred); m['fold'] = k
        metrik_rows.append(m)
        for d, p, pr in zip(va, prob, pred):
            val_rows.append({'fold': k, 'nama': d.nama, 'contract_id': getattr(d, 'contract_id', ambil_contract_id(d.nama)),
                             'label': int(d.y), 'prob_rentan': float(p), 'pred': int(pr)})
        pb, _ = prediksi(model, bench_list)
        bench_probs.append(pb)
        states.append({kk: v.detach().cpu() for kk, v in model.state_dict().items()})
        info_fold.append({'fold': k, 'n_train': len(tr), 'n_val': len(va), 'n_grup_train': len(set(grup[i_tr])),
                          'n_grup_val': len(set(grup[i_va])), 'epoch_terbaik': ep_best, 'epoch_berhenti': ep_stop})
        print(f'  [{kode} seed{seed}] fold {k}: F1={m["f1"]:.4f} AUC={m["auc"]:.4f} (epoch terbaik {ep_best})')
    menit = (time.time() - t0) / 60
    df_m = pd.DataFrame(metrik_rows)[['fold', 'accuracy', 'precision', 'recall', 'f1', 'auc']]
    rata = df_m.mean(numeric_only=True); rata['fold'] = 'rata_rata'
    df_m = pd.concat([df_m, pd.DataFrame([rata])], ignore_index=True)
    df_v = pd.DataFrame(val_rows)
    bp = np.array(bench_probs)
    df_b = pd.DataFrame({
        'pasangan': [d.nama.replace('_fixed.sol', '.sol') for d in bench_list],
        'nama_kontrak': [d.nama for d in bench_list],
        'tipe': ['aman' if d.nama.endswith('_fixed.sol') else 'rentan' for d in bench_list],
        'label': [int(d.y) for d in bench_list],
        'jumlah_node': [int(d.num_nodes) for d in bench_list],
        'jumlah_edge': [int(d.edge_index.shape[1]) for d in bench_list]})
    for k in range(bp.shape[0]): df_b[f'prob_fold{k+1}'] = bp[k]
    df_b['prob_rata'] = bp.mean(axis=0)
    cfg = {'kode': kode, 'mode': mode, 'model': ModelClass.__name__, 'seed': seed,
           'pembagi_fold': 'StratifiedGroupKFold(n_splits=5, shuffle=True)', 'atribut_grup': grup_attr,
           'n_graf': len(data_list), 'n_grup': int(len(set(grup))), 'fold': info_fold,
           'hiperparameter': {'optimizer': 'AdamW', 'lr': 1e-3, 'weight_decay': 1e-4, 'batch_size': 32,
                              'max_epoch': 50, 'sabar': 15, 'dim_hidden': 512, 'dropout': 0.5,
                              'grad_clip': 1.0, 'graphsmote_k': 5 if mode == 'smote' else None},
           'versi': {'torch': torch.__version__, 'torch_geometric': torch_geometric.__version__,
                     'sklearn': sklearn.__version__, 'numpy': np.__version__},
           'perangkat': torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'cpu',
           'waktu_latih_menit': round(menit, 2)}
    return df_m, df_v, df_b, states, cfg

def ringkas_benchmark(df_b):
    auc = roc_auc_score(df_b['label'], df_b['prob_rata'])
    r = df_b[df_b.tipe == 'rentan'].set_index('pasangan')['prob_rata']
    a = df_b[df_b.tipe == 'aman'].set_index('pasangan')['prob_rata']
    benar = int(sum(r[p] > a[p] for p in r.index if p in a.index))
    return auc, benar, len(r)

def simpan_hasil(kode, seed, df_m, df_v, df_b, states, cfg):
    os.makedirs(f'{D}/hasil', exist_ok=True)
    pre = f'{kode}_seed{seed}'
    df_m.to_csv(f'/content/{pre}_fold_metrics.csv', index=False, float_format='%.10f')
    df_v.to_csv(f'/content/{pre}_val_predictions.csv', index=False, float_format='%.10f')
    df_b.to_csv(f'/content/{pre}_benchmark_preds.csv', index=False, float_format='%.10f')
    with open(f'/content/{pre}_config.json', 'w') as f: json.dump(cfg, f, indent=2)
    torch.save(states, f'/content/{pre}_model_tiap_fold.pt')
    for suf in ['fold_metrics.csv', 'val_predictions.csv', 'benchmark_preds.csv', 'config.json', 'model_tiap_fold.pt']:
        src, dst = f'/content/{pre}_{suf}', f'{D}/hasil/{pre}_{suf}'
        shutil.copy(src, dst)
        s1, s2 = os.path.getsize(src), os.path.getsize(dst)
        print(f'  {pre}_{suf}: {s1:,} B -> Drive {s2:,} B {"OK" if s1 == s2 else "BEDA!"}')
        assert s1 == s2

def jalankan(kode, mode, ModelClass, data_list, bench_list, dim_input, seed, grup_attr):
    df_m, df_v, df_b, states, cfg = latih_konfigurasi(kode, mode, ModelClass, data_list, bench_list, dim_input, seed, grup_attr)
    simpan_hasil(kode, seed, df_m, df_v, df_b, states, cfg)
    auc_b, benar, total = ringkas_benchmark(df_b)
    r = df_m.iloc[-1]
    print(f'\n== {kode} seed {seed} ({cfg["waktu_latih_menit"]} menit) ==')
    print(f'CV rata-rata: acc={r.accuracy:.4f} prec={r.precision:.4f} rec={r.recall:.4f} F1={r.f1:.4f} AUC={r.auc:.4f}')
    print(f'Benchmark: AUC={auc_b:.4f}, pasangan benar={benar}/{total}\n')
    return {'kode': kode, 'seed': seed, 'f1_cv': r.f1, 'auc_cv': r.auc, 'auc_bench': auc_b, 'pasangan_benar': benar}

# ---------- data (dedup) ----------
obj_v3 = torch.load(f'{D}/data/processed/pyg/v3_augmented_dedup.pt', weights_only=False)
train_v3, bench = obj_v3['train'], obj_v3['benchmark']
DIM = obj_v3['dim']
for g in train_v3: g.contract_id = ambil_contract_id(g.nama)
print(f'device: {DEV} | v3 dedup: {len(train_v3)} graf, {len(set(g.contract_id for g in train_v3))} grup, '
      f'benchmark: {len(bench)}, dim: {DIM}')
RINGKASAN = []

device: cuda | v3 dedup: 1452 graf, 539 grup, benchmark: 16, dim: 22


In [ ]:
for kode, kelas in [('C6', GAT_SingleScale), ('C7', MultiScaleGAT_SAGPool)]:
    RINGKASAN.append(jalankan(kode, 'weighted', kelas, train_v3, bench, DIM, 42, 'contract_id'))
print(pd.DataFrame(RINGKASAN).to_string(index=False))

  [C6 seed42] fold 1: F1=0.8767 AUC=0.9308 (epoch terbaik 40)
  [C6 seed42] fold 2: F1=0.8475 AUC=0.9347 (epoch terbaik 50)
  [C6 seed42] fold 3: F1=0.8837 AUC=0.9401 (epoch terbaik 13)
  [C6 seed42] fold 4: F1=0.8544 AUC=0.9432 (epoch terbaik 48)
  [C6 seed42] fold 5: F1=0.8433 AUC=0.9259 (epoch terbaik 50)
  C6_seed42_fold_metrics.csv: 448 B -> Drive 448 B OK
  C6_seed42_val_predictions.csv: 177,994 B -> Drive 177,994 B OK
  C6_seed42_benchmark_preds.csv: 2,208 B -> Drive 2,208 B OK
  C6_seed42_config.json: 1,558 B -> Drive 1,558 B OK
  C6_seed42_model_tiap_fold.pt: 2,015,209 B -> Drive 2,015,209 B OK

== C6 seed 42 (1.76 menit) ==
CV rata-rata: acc=0.8801 prec=0.8311 rec=0.8940 F1=0.8611 AUC=0.9349
Benchmark: AUC=0.6406, pasangan benar=7/8



/tmp/ipykernel_590/901933586.py:197: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value 'rata_rata' has dtype incompatible with float64, please explicitly cast to a compatible dtype first.
  rata = df_m.mean(numeric_only=True); rata['fold'] = 'rata_rata'


  [C7 seed42] fold 1: F1=0.8562 AUC=0.9134 (epoch terbaik 48)
  [C7 seed42] fold 2: F1=0.8117 AUC=0.8923 (epoch terbaik 47)
  [C7 seed42] fold 3: F1=0.9055 AUC=0.9293 (epoch terbaik 32)
  [C7 seed42] fold 4: F1=0.8657 AUC=0.9490 (epoch terbaik 34)
  [C7 seed42] fold 5: F1=0.8525 AUC=0.9406 (epoch terbaik 47)
  C7_seed42_fold_metrics.csv: 448 B -> Drive 448 B OK
  C7_seed42_val_predictions.csv: 177,994 B -> Drive 177,994 B OK
  C7_seed42_benchmark_preds.csv: 2,208 B -> Drive 2,208 B OK
  C7_seed42_config.json: 1,564 B -> Drive 1,564 B OK
  C7_seed42_model_tiap_fold.pt: 12,648,587 B -> Drive 12,648,587 B OK

== C7 seed 42 (4.95 menit) ==
CV rata-rata: acc=0.8796 prec=0.8442 rec=0.8740 F1=0.8583 AUC=0.9249
Benchmark: AUC=0.5625, pasangan benar=4/8

kode  seed    f1_cv   auc_cv  auc_bench  pasangan_benar
  C6    42 0.861109 0.934943   0.640625               7
  C7    42 0.858309 0.924902   0.562500               4


/tmp/ipykernel_590/901933586.py:197: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value 'rata_rata' has dtype incompatible with float64, please explicitly cast to a compatible dtype first.
  rata = df_m.mean(numeric_only=True); rata['fold'] = 'rata_rata'


In [ ]:
for s in [42, 1, 7]:
    RINGKASAN.append(jalankan('C9', 'smote', MultiScaleGAT_MeanPool, train_v3, bench, DIM, s, 'contract_id'))
print(pd.DataFrame(RINGKASAN).to_string(index=False))

  [C9 seed42] fold 1: F1=0.8658 AUC=0.9236 (epoch terbaik 33)
  [C9 seed42] fold 2: F1=0.8319 AUC=0.9144 (epoch terbaik 25)
  [C9 seed42] fold 3: F1=0.9120 AUC=0.9474 (epoch terbaik 16)
  [C9 seed42] fold 4: F1=0.8195 AUC=0.9269 (epoch terbaik 6)
  [C9 seed42] fold 5: F1=0.8640 AUC=0.9380 (epoch terbaik 31)
  C9_seed42_fold_metrics.csv: 448 B -> Drive 448 B OK
  C9_seed42_val_predictions.csv: 177,994 B -> Drive 177,994 B OK
  C9_seed42_benchmark_preds.csv: 2,208 B -> Drive 2,208 B OK
  C9_seed42_config.json: 1,558 B -> Drive 1,558 B OK
  C9_seed42_model_tiap_fold.pt: 12,621,771 B -> Drive 12,621,771 B OK

== C9 seed 42 (3.55 menit) ==
CV rata-rata: acc=0.8809 prec=0.8414 rec=0.8776 F1=0.8586 AUC=0.9300
Benchmark: AUC=0.7031, pasangan benar=7/8



/tmp/ipykernel_590/901933586.py:197: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value 'rata_rata' has dtype incompatible with float64, please explicitly cast to a compatible dtype first.
  rata = df_m.mean(numeric_only=True); rata['fold'] = 'rata_rata'


  [C9 seed1] fold 1: F1=0.8596 AUC=0.9329 (epoch terbaik 16)
  [C9 seed1] fold 2: F1=0.8980 AUC=0.9507 (epoch terbaik 17)
  [C9 seed1] fold 3: F1=0.8686 AUC=0.9401 (epoch terbaik 49)
  [C9 seed1] fold 4: F1=0.8780 AUC=0.9353 (epoch terbaik 27)
  [C9 seed1] fold 5: F1=0.8594 AUC=0.9214 (epoch terbaik 20)
  C9_seed1_fold_metrics.csv: 448 B -> Drive 448 B OK
  C9_seed1_val_predictions.csv: 177,994 B -> Drive 177,994 B OK
  C9_seed1_benchmark_preds.csv: 2,208 B -> Drive 2,208 B OK
  C9_seed1_config.json: 1,559 B -> Drive 1,559 B OK
  C9_seed1_model_tiap_fold.pt: 12,621,635 B -> Drive 12,621,635 B OK

== C9 seed 1 (3.64 menit) ==
CV rata-rata: acc=0.8913 prec=0.8528 rec=0.8960 F1=0.8727 AUC=0.9361
Benchmark: AUC=0.6094, pasangan benar=6/8



/tmp/ipykernel_590/901933586.py:197: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value 'rata_rata' has dtype incompatible with float64, please explicitly cast to a compatible dtype first.
  rata = df_m.mean(numeric_only=True); rata['fold'] = 'rata_rata'


  [C9 seed7] fold 1: F1=0.8527 AUC=0.9279 (epoch terbaik 35)
  [C9 seed7] fold 2: F1=0.8571 AUC=0.9423 (epoch terbaik 29)
  [C9 seed7] fold 3: F1=0.9286 AUC=0.9536 (epoch terbaik 19)
  [C9 seed7] fold 4: F1=0.8707 AUC=0.9443 (epoch terbaik 44)
  [C9 seed7] fold 5: F1=0.8789 AUC=0.9362 (epoch terbaik 25)
  C9_seed7_fold_metrics.csv: 448 B -> Drive 448 B OK
  C9_seed7_val_predictions.csv: 177,994 B -> Drive 177,994 B OK
  C9_seed7_benchmark_preds.csv: 2,208 B -> Drive 2,208 B OK
  C9_seed7_config.json: 1,558 B -> Drive 1,558 B OK
  C9_seed7_model_tiap_fold.pt: 12,621,635 B -> Drive 12,621,635 B OK

== C9 seed 7 (4.14 menit) ==
CV rata-rata: acc=0.8964 prec=0.8586 rec=0.8987 F1=0.8776 AUC=0.9409
Benchmark: AUC=0.6562, pasangan benar=6/8

kode  seed    f1_cv   auc_cv  auc_bench  pasangan_benar
  C6    42 0.861109 0.934943   0.640625               7
  C7    42 0.858309 0.924902   0.562500               4
  C9    42 0.858643 0.930045   0.703125               7
  C9     1 0.872742 0.936100   

/tmp/ipykernel_590/901933586.py:197: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value 'rata_rata' has dtype incompatible with float64, please explicitly cast to a compatible dtype first.
  rata = df_m.mean(numeric_only=True); rata['fold'] = 'rata_rata'


In [ ]:
def benchmark_dari_csv_lama(kode):
    df = pd.read_csv(f'{D}/hasil/{kode}_seed42_benchmark_preds.csv')
    p = df['probabilitas_rentan'].values
    auc = roc_auc_score(df['label'], p)
    nama = df['nama_kontrak'].tolist()
    benar, total = 0, 0
    for n, l in zip(nama, df['label']):
        if l == 1 and n.replace('.sol', '_fixed.sol') in nama:
            total += 1
            benar += p[nama.index(n)] > p[nama.index(n.replace('.sol', '_fixed.sol'))]
    return auc, int(benar), total

for kode in ['C3', 'C5']:
    auc, benar, total = benchmark_dari_csv_lama(kode)
    print(f'{kode}: AUC benchmark={auc:.4f}, pasangan benar={benar}/{total}')

C3: AUC benchmark=0.5938, pasangan benar=5/8
C5: AUC benchmark=0.6562, pasangan benar=6/8


In [ ]:
# 1. arsipkan berkas run pertama (tidak ditimpa, hanya disalin)
arsip = f'{D}/hasil/arsip_run_pertama'
os.makedirs(arsip, exist_ok=True)
for kode in ['C3', 'C5']:
    for suf in ['fold_metrics.csv', 'benchmark_preds.csv', 'config.json', 'model_tiap_fold.pt']:
        src = f'{D}/hasil/{kode}_seed42_{suf}'
        if os.path.exists(src):
            dst = f'{arsip}/{kode}_seed42_{suf}'
            shutil.copy(src, dst)
            assert os.path.getsize(src) == os.path.getsize(dst)
            print(f'diarsipkan: {kode}_seed42_{suf} ({os.path.getsize(dst):,} B)')

# 2. angka run pertama (dari output Sel 18 dan 19) untuk pembanding
RUN_PERTAMA = {'C3': (0.8700, 0.9252), 'C5': (0.8621, 0.9277)}

# 3. data v2 dedup
obj_v2 = torch.load(f'{D}/data/processed/pyg/v2_cfg_prune_dedup.pt', weights_only=False)
train_v2, bench_v2 = obj_v2['train'], obj_v2['benchmark']
print(f'v2 dedup: {len(train_v2)} graf, benchmark {len(bench_v2)}, dim {obj_v2["dim"]}')

# 4. jalankan ulang
r3 = jalankan('C3', 'weighted', MultiScaleGAT_MeanPool, train_v2, bench_v2, obj_v2['dim'], 42, 'nama')
r5 = jalankan('C5', 'weighted', MultiScaleGAT_MeanPool, train_v3, bench, DIM, 42, 'contract_id')
RINGKASAN_ULANG = [r3, r5]

# 5. perbandingan run pertama vs jalankan ulang
for r in RINGKASAN_ULANG:
    f1_0, auc_0 = RUN_PERTAMA[r['kode']]
    print(f"{r['kode']}: F1 CV {f1_0:.4f} -> {r['f1_cv']:.4f} | AUC CV {auc_0:.4f} -> {r['auc_cv']:.4f} | "
          f"AUC benchmark {r['auc_bench']:.4f} | pasangan {r['pasangan_benar']}/8")

diarsipkan: C3_seed42_fold_metrics.csv (626 B)
diarsipkan: C3_seed42_benchmark_preds.csv (672 B)
diarsipkan: C3_seed42_config.json (332 B)
diarsipkan: C3_seed42_model_tiap_fold.pt (12,621,771 B)
diarsipkan: C5_seed42_fold_metrics.csv (601 B)
diarsipkan: C5_seed42_benchmark_preds.csv (672 B)
diarsipkan: C5_seed42_config.json (340 B)
diarsipkan: C5_seed42_model_tiap_fold.pt (12,621,771 B)
v2 dedup: 542 graf, benchmark 16, dim 22
  [C3 seed42] fold 1: F1=0.9286 AUC=0.9545 (epoch terbaik 28)
  [C3 seed42] fold 2: F1=0.8675 AUC=0.9079 (epoch terbaik 4)
  [C3 seed42] fold 3: F1=0.8837 AUC=0.9403 (epoch terbaik 50)
  [C3 seed42] fold 4: F1=0.8257 AUC=0.8662 (epoch terbaik 14)
  [C3 seed42] fold 5: F1=0.8395 AUC=0.9259 (epoch terbaik 31)
  C3_seed42_fold_metrics.csv: 448 B -> Drive 448 B OK
  C3_seed42_val_predictions.csv: 61,291 B -> Drive 61,291 B OK
  C3_seed42_benchmark_preds.csv: 2,208 B -> Drive 2,208 B OK
  C3_seed42_config.json: 1,552 B -> Drive 1,552 B OK
  C3_seed42_model_tiap_fold.p

/tmp/ipykernel_590/901933586.py:197: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value 'rata_rata' has dtype incompatible with float64, please explicitly cast to a compatible dtype first.
  rata = df_m.mean(numeric_only=True); rata['fold'] = 'rata_rata'


  [C5 seed42] fold 1: F1=0.8741 AUC=0.9374 (epoch terbaik 49)
  [C5 seed42] fold 2: F1=0.8374 AUC=0.9043 (epoch terbaik 18)
  [C5 seed42] fold 3: F1=0.9113 AUC=0.9557 (epoch terbaik 32)
  [C5 seed42] fold 4: F1=0.8469 AUC=0.9411 (epoch terbaik 50)
  [C5 seed42] fold 5: F1=0.8672 AUC=0.9315 (epoch terbaik 40)
  C5_seed42_fold_metrics.csv: 448 B -> Drive 448 B OK
  C5_seed42_val_predictions.csv: 177,994 B -> Drive 177,994 B OK
  C5_seed42_benchmark_preds.csv: 2,208 B -> Drive 2,208 B OK
  C5_seed42_config.json: 1,565 B -> Drive 1,565 B OK
  C5_seed42_model_tiap_fold.pt: 12,621,771 B -> Drive 12,621,771 B OK

== C5 seed 42 (4.08 menit) ==
CV rata-rata: acc=0.8884 prec=0.8562 rec=0.8800 F1=0.8674 AUC=0.9340
Benchmark: AUC=0.7812, pasangan benar=8/8

C3: F1 CV 0.8700 -> 0.8690 | AUC CV 0.9252 -> 0.9190 | AUC benchmark 0.5938 | pasangan 5/8
C5: F1 CV 0.8621 -> 0.8674 | AUC CV 0.9277 -> 0.9340 | AUC benchmark 0.7812 | pasangan 8/8


/tmp/ipykernel_590/901933586.py:197: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value 'rata_rata' has dtype incompatible with float64, please explicitly cast to a compatible dtype first.
  rata = df_m.mean(numeric_only=True); rata['fold'] = 'rata_rata'


In [ ]:
from math import factorial
from itertools import permutations
from torch_geometric.data import Data
from torch_geometric.data import Batch

def muat_ensemble(kode, seed, dim):
    sds = torch.load(f'{D}/hasil/{kode}_seed{seed}_model_tiap_fold.pt', weights_only=False)
    ms = []
    for sd in sds:
        m = MultiScaleGAT_MeanPool(dim_input=dim).to(DEV)
        m.load_state_dict(sd); m.eval(); ms.append(m)
    return ms

# Skema masking (sama dengan notebook GraphSMOTE asli):
# node tertutup = vektor fitur dikali 0 (x * mask), node TETAP ada di graf,
# edge_index TIDAK diubah, jadi semua sisi ke/dari node tertutup tetap ada.
def graf_termask(data, bits):
    n = data.num_nodes
    mask = torch.tensor([(bits >> i) & 1 for i in range(n)], dtype=torch.float).unsqueeze(1)
    return Data(x=data.x * mask, edge_index=data.edge_index)

@torch.no_grad()
def v_batch(models, data, batch_size=512):
    n = data.num_nodes
    daftar = [graf_termask(data, bits) for bits in range(2 ** n)]
    hasil = []
    for b in DataLoader(daftar, batch_size=batch_size):
        b = b.to(DEV)
        p = np.mean([torch.softmax(m(b), 1)[:, 1].cpu().numpy() for m in models], axis=0)
        hasil.extend(p.tolist())
    return np.array(hasil)

@torch.no_grad()
def v_satu_satu(models, data):
    n = data.num_nodes
    v = np.zeros(2 ** n)
    for bits in range(2 ** n):
        b = Batch.from_data_list([graf_termask(data, bits)]).to(DEV)
        v[bits] = np.mean([torch.softmax(m(b), 1)[:, 1].item() for m in models])
    return v

def shapley_dari_cache(v, n):
    nilai = np.zeros(n)
    for i in range(n):
        for bits in range(2 ** n):
            if (bits >> i) & 1: continue
            r = bin(bits).count('1')
            nilai[i] += factorial(r) * factorial(n - r - 1) / factorial(n) * (v[bits | (1 << i)] - v[bits])
    return nilai

def shapley_permutasi(v, n):
    nilai = np.zeros(n); jml = 0
    for perm in permutations(range(n)):
        bits = 0
        for i in perm:
            nilai[i] += v[bits | (1 << i)] - v[bits]
            bits |= (1 << i)
        jml += 1
    return nilai / jml

def node_gt(x): return (x[:, 17] > 0) | (x[:, 18] > 0) | (x[:, 19] > 0)

def jalankan_shap(kode, seed, data_bench, dim):
    models = muat_ensemble(kode, seed, dim)
    baris, node_rows, hasil = [], [], {}
    for d in data_bench:
        n = d.num_nodes
        if n > 15:
            print(f'  {kode} {d.nama} dilewati (n={n})'); continue
        v = v_batch(models, d)
        phi = shapley_dari_cache(v, n)
        hasil[d.nama] = (phi, v)
        gt = set(np.where(node_gt(d.x).numpy())[0].tolist())
        top2 = set(np.argsort(-phi)[:2].tolist())
        iou = len(top2 & gt) / len(top2 | gt) if gt else np.nan
        m = len(gt); inter = 2 * m / n; iou_acak = inter / (2 + m - inter) if gt else np.nan
        baris.append({'model': kode, 'kontrak': d.nama, 'n_node': n, 'n_gt': m,
                      'std_shap': float(np.std(phi)),
                      'pola': 'flat' if np.std(phi) < 0.001 else 'discriminative',
                      'top1_gt': int(np.argmax(phi)) in gt, 'iou_k2': iou, 'iou_acak': iou_acak,
                      'cek_efisiensi': float(phi.sum() - (v[-1] - v[0]))})
        for i in range(n):
            node_rows.append({'model': kode, 'kontrak': d.nama, 'node': i, 'shapley': float(phi[i]),
                              'is_gt': i in gt, 'peringkat': int((phi > phi[i]).sum() + 1)})
    return pd.DataFrame(baris), pd.DataFrame(node_rows), hasil, models

KONF_SHAP = [('C3', bench_v2, obj_v2['dim']), ('C5', bench, DIM), ('C9', bench, DIM)]
SHAP_RINGKAS, SHAP_NODE, SHAP_CACHE = {}, {}, {}
for kode, db, dim in KONF_SHAP:
    t0 = time.time()
    r, nd, hasil, models = jalankan_shap(kode, 42, db, dim)
    SHAP_RINGKAS[kode], SHAP_NODE[kode], SHAP_CACHE[kode] = r, nd, hasil
    print(f'{kode}: {len(r)} kontrak, {time.time()-t0:.0f} dtk, '
          f'maks |galat efisiensi|={r.cek_efisiensi.abs().max():.2e}')

# verifikasi brute force (permutasi, evaluasi satu per satu) pada kontrak n terkecil, model C5
db_c5 = {d.nama: d for d in bench}
nama_kecil = min((d for d in bench if d.num_nodes <= 8), key=lambda d: d.num_nodes).nama
d = db_c5[nama_kecil]
ms = muat_ensemble('C5', 42, DIM)
v1 = v_satu_satu(ms, d)
phi_bf = shapley_permutasi(v1, d.num_nodes)
phi_b, v_b = SHAP_CACHE['C5'][nama_kecil]
print(f'\nVerifikasi brute force ({nama_kecil}, n={d.num_nodes}): '
      f'maks |v batch - v satu-satu|={np.abs(v_b - v1).max():.2e}, '
      f'maks |phi batch - phi permutasi|={np.abs(phi_b - phi_bf).max():.2e}')

# ringkasan akhir
print('\n=== Ringkasan SHAP (seed 42, k=2) ===')
for kode in ['C3', 'C5', 'C9']:
    r = SHAP_RINGKAS[kode]
    print(f"{kode}: IoU rata-rata={r.iou_k2.mean():.3f} (acak {r.iou_acak.mean():.3f}), "
          f"flat={int((r.pola=='flat').sum())}/{len(r)}, discriminative={int((r.pola=='discriminative').sum())}/{len(r)}, "
          f"top1 di GT={int(r.top1_gt.sum())}/{len(r)}")

# simpan
pd.concat(SHAP_RINGKAS.values()).to_csv('/content/shap_ringkasan_seed42.csv', index=False, float_format='%.10f')
pd.concat(SHAP_NODE.values()).to_csv('/content/shap_nilai_per_node_seed42.csv', index=False, float_format='%.10f')
for f in ['shap_ringkasan_seed42.csv', 'shap_nilai_per_node_seed42.csv']:
    shutil.copy(f'/content/{f}', f'{D}/hasil/{f}')
    s1, s2 = os.path.getsize(f'/content/{f}'), os.path.getsize(f'{D}/hasil/{f}')
    print(f'{f}: {s1:,} B -> Drive {s2:,} B {"OK" if s1 == s2 else "BEDA!"}'); assert s1 == s2
print(pd.concat(SHAP_RINGKAS.values())[['model','kontrak','n_node','std_shap','pola','top1_gt','iou_k2']].to_string(index=False))

C3: 16 kontrak, 6 dtk, maks |galat efisiensi|=1.11e-16
C5: 16 kontrak, 4 dtk, maks |galat efisiensi|=3.47e-17
C9: 16 kontrak, 4 dtk, maks |galat efisiensi|=1.39e-17

Verifikasi brute force (reentrancy_insecure_fixed.sol, n=6): maks |v batch - v satu-satu|=6.05e-08, maks |phi batch - phi permutasi|=7.88e-09

=== Ringkasan SHAP (seed 42, k=2) ===
C3: IoU rata-rata=0.333 (acak 0.184), flat=2/16, discriminative=14/16, top1 di GT=5/16
C5: IoU rata-rata=0.306 (acak 0.184), flat=5/16, discriminative=11/16, top1 di GT=8/16
C9: IoU rata-rata=0.223 (acak 0.184), flat=6/16, discriminative=10/16, top1 di GT=7/16
shap_ringkasan_seed42.csv: 4,834 B -> Drive 4,834 B OK
shap_nilai_per_node_seed42.csv: 18,854 B -> Drive 18,854 B OK
model                       kontrak  n_node  std_shap           pola  top1_gt   iou_k2
   C3           etherbank_fixed.sol       7  0.000941           flat     True 0.666667
   C3          etherstore_fixed.sol      10  0.033905 discriminative    False 0.166667
   C3 modifier

In [ ]:
from scipy.stats import spearmanr

# A. SHAP dari model arsip C5 run pertama (konfigurasi, data, seed sama dengan jalankan ulang)
_muat_asli = muat_ensemble
def _muat_dari_arsip(kode, seed, dim):
    sds = torch.load(f'{D}/hasil/arsip_run_pertama/{kode}_seed{seed}_model_tiap_fold.pt', weights_only=False)
    ms = []
    for sd in sds:
        m = MultiScaleGAT_MeanPool(dim_input=dim).to(DEV)
        m.load_state_dict(sd); m.eval(); ms.append(m)
    return ms
muat_ensemble = _muat_dari_arsip
r_run1, nd_run1, cache_run1, _ = jalankan_shap('C5', 42, bench, DIM)
muat_ensemble = _muat_asli
r_run1['model'] = 'C5_run1'; nd_run1['model'] = 'C5_run1'

r_ulang = SHAP_RINGKAS['C5']
print('=== C5 seed 42: run 1 (arsip) vs jalankan ulang ===')
for nama, r in [('run 1', r_run1), ('ulang', r_ulang)]:
    print(f"{nama:6}: IoU={r.iou_k2.mean():.3f}, flat={int((r.pola=='flat').sum())}/16, "
          f"disc={int((r.pola=='discriminative').sum())}/16, top1 di GT={int(r.top1_gt.sum())}/16")
rho = []
for k in SHAP_CACHE['C5']:
    a, b = SHAP_CACHE['C5'][k][0], cache_run1[k][0]
    rho.append(spearmanr(a, b).correlation)
print(f'Korelasi peringkat Spearman Shapley per kontrak antar dua run: rata-rata={np.nanmean(rho):.3f}, '
      f'min={np.nanmin(rho):.3f}, maks={np.nanmax(rho):.3f}')
pd.concat([r_run1, nd_run1.assign(x=0).iloc[0:0]]) if False else None
r_run1.to_csv('/content/shap_ringkasan_C5_run1_arsip.csv', index=False, float_format='%.10f')
nd_run1.to_csv('/content/shap_nilai_per_node_C5_run1_arsip.csv', index=False, float_format='%.10f')
for f in ['shap_ringkasan_C5_run1_arsip.csv', 'shap_nilai_per_node_C5_run1_arsip.csv']:
    shutil.copy(f'/content/{f}', f'{D}/hasil/{f}')
    s1, s2 = os.path.getsize(f'/content/{f}'), os.path.getsize(f'{D}/hasil/{f}')
    print(f'{f}: {s1:,} B -> Drive {s2:,} B {"OK" if s1 == s2 else "BEDA!"}'); assert s1 == s2

# B. C5 pada seed 1 dan 7
HASIL_C5_SEED = []
for s in [1, 7]:
    HASIL_C5_SEED.append(jalankan('C5', 'weighted', MultiScaleGAT_MeanPool, train_v3, bench, DIM, s, 'contract_id'))
print(pd.DataFrame([RINGKASAN_ULANG[1]] + HASIL_C5_SEED).to_string(index=False))
print('\nPembanding C9 :')
print(pd.DataFrame([x for x in RINGKASAN if x['kode'] == 'C9']).to_string(index=False))

=== C5 seed 42: run 1 (arsip) vs jalankan ulang ===
run 1 : IoU=0.237, flat=4/16, disc=12/16, top1 di GT=7/16
ulang : IoU=0.306, flat=5/16, disc=11/16, top1 di GT=8/16
Korelasi peringkat Spearman Shapley per kontrak antar dua run: rata-rata=0.613, min=0.179, maks=1.000
shap_ringkasan_C5_run1_arsip.csv: 1,744 B -> Drive 1,744 B OK
shap_nilai_per_node_C5_run1_arsip.csv: 6,987 B -> Drive 6,987 B OK
  [C5 seed1] fold 1: F1=0.8739 AUC=0.9368 (epoch terbaik 25)
  [C5 seed1] fold 2: F1=0.8735 AUC=0.9411 (epoch terbaik 18)
  [C5 seed1] fold 3: F1=0.8737 AUC=0.9330 (epoch terbaik 48)
  [C5 seed1] fold 4: F1=0.8560 AUC=0.9262 (epoch terbaik 17)
  [C5 seed1] fold 5: F1=0.8066 AUC=0.9119 (epoch terbaik 11)
  C5_seed1_fold_metrics.csv: 448 B -> Drive 448 B OK
  C5_seed1_val_predictions.csv: 177,994 B -> Drive 177,994 B OK
  C5_seed1_benchmark_preds.csv: 2,208 B -> Drive 2,208 B OK
  C5_seed1_config.json: 1,565 B -> Drive 1,565 B OK
  C5_seed1_model_tiap_fold.pt: 12,621,635 B -> Drive 12,621,635 B O

/tmp/ipykernel_590/901933586.py:197: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value 'rata_rata' has dtype incompatible with float64, please explicitly cast to a compatible dtype first.
  rata = df_m.mean(numeric_only=True); rata['fold'] = 'rata_rata'


  [C5 seed7] fold 1: F1=0.8571 AUC=0.9327 (epoch terbaik 48)
  [C5 seed7] fold 2: F1=0.8945 AUC=0.9532 (epoch terbaik 47)
  [C5 seed7] fold 3: F1=0.9249 AUC=0.9568 (epoch terbaik 36)
  [C5 seed7] fold 4: F1=0.8667 AUC=0.9365 (epoch terbaik 46)
  [C5 seed7] fold 5: F1=0.8505 AUC=0.9097 (epoch terbaik 8)
  C5_seed7_fold_metrics.csv: 448 B -> Drive 448 B OK
  C5_seed7_val_predictions.csv: 177,994 B -> Drive 177,994 B OK
  C5_seed7_benchmark_preds.csv: 2,208 B -> Drive 2,208 B OK
  C5_seed7_config.json: 1,563 B -> Drive 1,563 B OK
  C5_seed7_model_tiap_fold.pt: 12,621,635 B -> Drive 12,621,635 B OK

== C5 seed 7 (3.97 menit) ==
CV rata-rata: acc=0.8981 prec=0.8693 rec=0.8903 F1=0.8787 AUC=0.9378
Benchmark: AUC=0.7344, pasangan benar=7/8

kode  seed    f1_cv   auc_cv  auc_bench  pasangan_benar
  C5    42 0.867388 0.934002   0.781250               8
  C5     1 0.856736 0.929782   0.609375               6
  C5     7 0.878736 0.937769   0.734375               7

Pembanding C9 :
kode  seed    f

/tmp/ipykernel_590/901933586.py:197: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value 'rata_rata' has dtype incompatible with float64, please explicitly cast to a compatible dtype first.
  rata = df_m.mean(numeric_only=True); rata['fold'] = 'rata_rata'


In [ ]:
obj_v1 = torch.load(f'{D}/data/processed/pyg/cfg_prune_0.02_dedup.pt', weights_only=False)
obj_v2p = torch.load(f'{D}/data/processed/pyg/v2_cfg_prune_pos_dedup.pt', weights_only=False)
for nm, o, exp in [('v1', obj_v1, 696), ('v2', obj_v2, 542), ('v2_pos', obj_v2p, 542)]:
    print(f"{nm}: {len(o['train'])} graf, dim {o['dim']}, benchmark {len(o['benchmark'])}")
    assert len(o['train']) == exp and len(o['benchmark']) == 16
    assert len(set(g.nama for g in o['train'])) == len(o['train'])   # tiap graf = 1 kontrak = 1 grup

KONFIG_BARU = [('C1', 'weighted', MultiScaleGAT_SAGPool, obj_v1),
               ('C2', 'weighted', MultiScaleGAT_SAGPool, obj_v2),
               ('C4', 'weighted', MultiScaleGAT_MeanPool, obj_v2p),
               ('C8', 'smote',    MultiScaleGAT_MeanPool, obj_v2)]
for kode, mode, kelas, o in KONFIG_BARU:
    RINGKASAN.append(jalankan(kode, mode, kelas, o['train'], o['benchmark'], o['dim'], 42, 'nama'))

# pembanding dengan Tabel II lama: (F1 CV, AUC CV, AUC bench, pasangan)
LAMA = {'C1': (0.8397, 0.9298, 0.5391, 0), 'C2': (0.8612, 0.9318, 0.5938, 4),
        'C3': (0.8930, 0.9472, 0.5625, 4), 'C4': (0.8850, 0.9472, 0.5938, 4),
        'C8': (0.8857, 0.9454, 0.6250, 6)}
print('\nkode | F1 CV lama -> baru | AUC CV lama -> baru | AUC bench lama -> baru | pasangan lama -> baru')
baru = {r['kode']: r for r in RINGKASAN if r['seed'] == 42}
baru['C3'] = r3   # jalankan ulang C3 dari Sel 24
for k in ['C1', 'C2', 'C3', 'C4', 'C8']:
    l, b = LAMA[k], baru[k]
    print(f"{k} | {l[0]:.4f} -> {b['f1_cv']:.4f} | {l[1]:.4f} -> {b['auc_cv']:.4f} | "
          f"{l[2]:.4f} -> {b['auc_bench']:.4f} | {l[3]}/8 -> {b['pasangan_benar']}/8")

v1: 696 graf, dim 22, benchmark 16
v2: 542 graf, dim 22, benchmark 16
v2_pos: 542 graf, dim 23, benchmark 16
  [C1 seed42] fold 1: F1=0.8409 AUC=0.9192 (epoch terbaik 11)
  [C1 seed42] fold 2: F1=0.8193 AUC=0.9320 (epoch terbaik 31)
  [C1 seed42] fold 3: F1=0.8485 AUC=0.9306 (epoch terbaik 16)
  [C1 seed42] fold 4: F1=0.8889 AUC=0.9593 (epoch terbaik 43)
  [C1 seed42] fold 5: F1=0.8039 AUC=0.9271 (epoch terbaik 10)
  C1_seed42_fold_metrics.csv: 448 B -> Drive 448 B OK
  C1_seed42_val_predictions.csv: 78,693 B -> Drive 78,693 B OK
  C1_seed42_benchmark_preds.csv: 2,208 B -> Drive 2,208 B OK
  C1_seed42_config.json: 1,552 B -> Drive 1,552 B OK
  C1_seed42_model_tiap_fold.pt: 12,648,587 B -> Drive 12,648,587 B OK

== C1 seed 42 (2.03 menit) ==
CV rata-rata: acc=0.8937 prec=0.7937 rec=0.8943 F1=0.8403 AUC=0.9336
Benchmark: AUC=0.5625, pasangan benar=6/8



/tmp/ipykernel_590/901933586.py:197: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value 'rata_rata' has dtype incompatible with float64, please explicitly cast to a compatible dtype first.
  rata = df_m.mean(numeric_only=True); rata['fold'] = 'rata_rata'


  [C2 seed42] fold 1: F1=0.8989 AUC=0.9240 (epoch terbaik 16)
  [C2 seed42] fold 2: F1=0.8810 AUC=0.9680 (epoch terbaik 10)
  [C2 seed42] fold 3: F1=0.7901 AUC=0.8782 (epoch terbaik 33)
  [C2 seed42] fold 4: F1=0.7611 AUC=0.8195 (epoch terbaik 10)
  [C2 seed42] fold 5: F1=0.7912 AUC=0.9005 (epoch terbaik 9)
  C2_seed42_fold_metrics.csv: 448 B -> Drive 448 B OK
  C2_seed42_val_predictions.csv: 61,291 B -> Drive 61,291 B OK
  C2_seed42_benchmark_preds.csv: 2,208 B -> Drive 2,208 B OK
  C2_seed42_config.json: 1,551 B -> Drive 1,551 B OK
  C2_seed42_model_tiap_fold.pt: 12,648,587 B -> Drive 12,648,587 B OK

== C2 seed 42 (1.73 menit) ==
CV rata-rata: acc=0.8488 prec=0.7956 rec=0.8668 F1=0.8244 AUC=0.8980
Benchmark: AUC=0.5938, pasangan benar=6/8



/tmp/ipykernel_590/901933586.py:197: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value 'rata_rata' has dtype incompatible with float64, please explicitly cast to a compatible dtype first.
  rata = df_m.mean(numeric_only=True); rata['fold'] = 'rata_rata'


  [C4 seed42] fold 1: F1=0.9070 AUC=0.9545 (epoch terbaik 20)
  [C4 seed42] fold 2: F1=0.9048 AUC=0.9530 (epoch terbaik 32)
  [C4 seed42] fold 3: F1=0.9231 AUC=0.9545 (epoch terbaik 39)
  [C4 seed42] fold 4: F1=0.8602 AUC=0.9283 (epoch terbaik 22)
  [C4 seed42] fold 5: F1=0.8000 AUC=0.9038 (epoch terbaik 6)
  C4_seed42_fold_metrics.csv: 448 B -> Drive 448 B OK
  C4_seed42_val_predictions.csv: 61,291 B -> Drive 61,291 B OK
  C4_seed42_benchmark_preds.csv: 2,208 B -> Drive 2,208 B OK
  C4_seed42_config.json: 1,552 B -> Drive 1,552 B OK
  C4_seed42_model_tiap_fold.pt: 12,642,251 B -> Drive 12,642,251 B OK

== C4 seed 42 (1.95 menit) ==
CV rata-rata: acc=0.9022 prec=0.8686 rec=0.8903 F1=0.8790 AUC=0.9388
Benchmark: AUC=0.6562, pasangan benar=6/8



/tmp/ipykernel_590/901933586.py:197: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value 'rata_rata' has dtype incompatible with float64, please explicitly cast to a compatible dtype first.
  rata = df_m.mean(numeric_only=True); rata['fold'] = 'rata_rata'


  [C8 seed42] fold 1: F1=0.9302 AUC=0.9488 (epoch terbaik 21)
  [C8 seed42] fold 2: F1=0.8810 AUC=0.9479 (epoch terbaik 11)
  [C8 seed42] fold 3: F1=0.9333 AUC=0.9531 (epoch terbaik 48)
  [C8 seed42] fold 4: F1=0.8542 AUC=0.9255 (epoch terbaik 24)
  [C8 seed42] fold 5: F1=0.8354 AUC=0.9227 (epoch terbaik 27)
  C8_seed42_fold_metrics.csv: 448 B -> Drive 448 B OK
  C8_seed42_val_predictions.csv: 61,291 B -> Drive 61,291 B OK
  C8_seed42_benchmark_preds.csv: 2,208 B -> Drive 2,208 B OK
  C8_seed42_config.json: 1,547 B -> Drive 1,547 B OK
  C8_seed42_model_tiap_fold.pt: 12,621,771 B -> Drive 12,621,771 B OK

== C8 seed 42 (2.09 menit) ==
CV rata-rata: acc=0.9096 prec=0.8872 rec=0.8898 F1=0.8868 AUC=0.9396
Benchmark: AUC=0.5938, pasangan benar=5/8


kode | F1 CV lama -> baru | AUC CV lama -> baru | AUC bench lama -> baru | pasangan lama -> baru
C1 | 0.8397 -> 0.8403 | 0.9298 -> 0.9336 | 0.5391 -> 0.5625 | 0/8 -> 6/8
C2 | 0.8612 -> 0.8244 | 0.9318 -> 0.8980 | 0.5938 -> 0.5938 | 4/8 -> 6/8
C3

/tmp/ipykernel_590/901933586.py:197: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value 'rata_rata' has dtype incompatible with float64, please explicitly cast to a compatible dtype first.
  rata = df_m.mean(numeric_only=True); rata['fold'] = 'rata_rata'


In [ ]:
from sklearn.linear_model import LogisticRegression

def fitur_ukuran(d):
    return [float(np.log1p(d.num_nodes)), float(np.log1p(d.edge_index.size(1)))]

def bangun_df_bench(bench_list, bp):
    df_b = pd.DataFrame({
        'pasangan': [d.nama.replace('_fixed.sol', '.sol') for d in bench_list],
        'nama_kontrak': [d.nama for d in bench_list],
        'tipe': ['aman' if d.nama.endswith('_fixed.sol') else 'rentan' for d in bench_list],
        'label': [int(d.y) for d in bench_list],
        'jumlah_node': [int(d.num_nodes) for d in bench_list],
        'jumlah_edge': [int(d.edge_index.shape[1]) for d in bench_list]})
    bp = np.array(bp)
    for k in range(bp.shape[0]): df_b[f'prob_fold{k+1}'] = bp[k]
    df_b['prob_rata'] = bp.mean(axis=0)
    return df_b

def jalankan_B0(data_list, bench_list, seed, grup_attr='nama'):
    t0 = time.time()
    X = np.array([fitur_ukuran(d) for d in data_list]); y = np.array([int(d.y) for d in data_list])
    grup = np.array([getattr(d, grup_attr) for d in data_list])
    Xb = np.array([fitur_ukuran(d) for d in bench_list])
    cv = StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=seed)
    metrik_rows, val_rows, bench_probs, states, info = [], [], [], [], []
    oof_prob = np.zeros(len(y)); oof_pred = np.zeros(len(y), dtype=int)
    for k, (i_tr, i_va) in enumerate(cv.split(X, y, grup), 1):
        assert set(grup[i_tr]).isdisjoint(set(grup[i_va])), 'GRUP BOCOR ANTAR FOLD'
        lr = LogisticRegression(class_weight='balanced').fit(X[i_tr], y[i_tr])
        prob = lr.predict_proba(X[i_va])[:, 1]; pred = lr.predict(X[i_va])
        oof_prob[i_va] = prob; oof_pred[i_va] = pred
        m = hitung_metrik(y[i_va], prob, pred); m['fold'] = k; metrik_rows.append(m)
        for i, p, pr in zip(i_va, prob, pred):
            d = data_list[i]
            val_rows.append({'fold': k, 'nama': d.nama, 'contract_id': getattr(d, 'contract_id', ambil_contract_id(d.nama)),
                             'label': int(d.y), 'prob_rentan': float(p), 'pred': int(pr)})
        bench_probs.append(lr.predict_proba(Xb)[:, 1])
        states.append({'coef': lr.coef_.copy(), 'intercept': lr.intercept_.copy()})
        info.append({'fold': k, 'n_train': len(i_tr), 'n_val': len(i_va)})
    df_m = pd.DataFrame(metrik_rows)[['fold', 'accuracy', 'precision', 'recall', 'f1', 'auc']]
    rata = df_m.mean(numeric_only=True).astype(object); rata['fold'] = 'rata_rata'
    df_m = pd.concat([df_m, pd.DataFrame([rata])], ignore_index=True)
    df_v = pd.DataFrame(val_rows); df_b = bangun_df_bench(bench_list, bench_probs)
    f1_gab, auc_gab = f1_score(y, oof_pred, zero_division=0), roc_auc_score(y, oof_prob)
    cfg = {'kode': 'B0', 'mode': 'logistic_regression', 'model': 'LogisticRegression(class_weight=balanced)',
           'fitur': ['log1p(jumlah_node)', 'log1p(jumlah_edge)'], 'seed': seed,
           'pembagi_fold': 'StratifiedGroupKFold(n_splits=5, shuffle=True)', 'atribut_grup': grup_attr,
           'n_graf': len(data_list), 'n_grup': int(len(set(grup))), 'fold': info,
           'metrik_gabungan_out_of_fold': {'f1': f1_gab, 'auc': auc_gab},
           'versi': {'sklearn': sklearn.__version__, 'numpy': np.__version__},
           'waktu_latih_menit': round((time.time() - t0) / 60, 3)}
    simpan_hasil('B0', seed, df_m, df_v, df_b, states, cfg)
    auc_b, benar, total = ringkas_benchmark(df_b)
    r = df_m.iloc[-1]
    seri = df_b[df_b.tipe == 'rentan'].set_index('pasangan')['prob_rata']
    seri_a = df_b[df_b.tipe == 'aman'].set_index('pasangan')['prob_rata']
    seri_n = int(sum(np.isclose(seri[p], seri_a[p]) for p in seri.index))
    print(f'== B0 seed {seed} == rata-rata per fold: F1={r.f1:.4f} AUC={r.auc:.4f} | '
          f'gabungan out-of-fold: F1={f1_gab:.4f} AUC={auc_gab:.4f} | '
          f'benchmark AUC={auc_b:.4f}, pasangan benar={benar}/{total} (pasangan probabilitas sama: {seri_n})')
    return {'kode': 'B0', 'seed': seed, 'f1_cv': float(r.f1), 'auc_cv': float(r.auc),
            'auc_bench': auc_b, 'pasangan_benar': benar}

for s in [42, 1, 7, 123, 2024]:
    RINGKASAN.append(jalankan_B0(obj_v1['train'], obj_v1['benchmark'], s))

  B0_seed42_fold_metrics.csv: 448 B -> Drive 448 B OK
  B0_seed42_val_predictions.csv: 78,693 B -> Drive 78,693 B OK
  B0_seed42_benchmark_preds.csv: 2,208 B -> Drive 2,208 B OK
  B0_seed42_config.json: 882 B -> Drive 882 B OK
  B0_seed42_model_tiap_fold.pt: 2,329 B -> Drive 2,329 B OK
== B0 seed 42 == rata-rata per fold: F1=0.7170 AUC=0.8730 | gabungan out-of-fold: F1=0.7178 AUC=0.8714 | benchmark AUC=0.6172, pasangan benar=1/8 (pasangan probabilitas sama: 7)
  B0_seed1_fold_metrics.csv: 448 B -> Drive 448 B OK
  B0_seed1_val_predictions.csv: 78,693 B -> Drive 78,693 B OK
  B0_seed1_benchmark_preds.csv: 2,208 B -> Drive 2,208 B OK
  B0_seed1_config.json: 881 B -> Drive 881 B OK
  B0_seed1_model_tiap_fold.pt: 2,323 B -> Drive 2,323 B OK
== B0 seed 1 == rata-rata per fold: F1=0.7142 AUC=0.8723 | gabungan out-of-fold: F1=0.7149 AUC=0.8711 | benchmark AUC=0.6172, pasangan benar=1/8 (pasangan probabilitas sama: 7)
  B0_seed7_fold_metrics.csv: 448 B -> Drive 448 B OK
  B0_seed7_val_predicti

In [ ]:
pd.set_option('display.float_format', lambda v: f'{v:.6f}')
for kode in ['C1', 'C5', 'C9']:
    df = pd.read_csv(f'{D}/hasil/{kode}_seed42_benchmark_preds.csv')
    r = df[df.tipe == 'rentan'].set_index('pasangan'); a = df[df.tipe == 'aman'].set_index('pasangan')
    out = pd.DataFrame({'node_R': r.jumlah_node, 'node_A': a.jumlah_node, 'edge_R': r.jumlah_edge, 'edge_A': a.jumlah_edge,
                        'prob_R': r.prob_rata, 'prob_A': a.prob_rata})
    out['selisih'] = out.prob_R - out.prob_A
    out['benar'] = out.selisih > 0
    print(f'\n== {kode} seed 42, benar={int(out.benar.sum())}/8 ==')
    print(out.to_string())


== C1 seed 42, benar=6/8 ==
                         node_R  node_A  edge_R  edge_A   prob_R   prob_A   selisih  benar
pasangan                                                                                  
etherbank.sol                 8       8       5       5 0.000680 0.010425 -0.009744  False
etherstore.sol               10      10       7       7 0.028412 0.021889  0.006523   True
modifier_reentrancy.sol      14       7      10       4 0.179776 0.023276  0.156501   True
reentrance.sol               10      10       7       7 0.008340 0.004663  0.003677   True
reentrancy_dao.sol            9       9       7       7 0.318629 0.294073  0.024556   True
reentrancy_insecure.sol       6       6       5       5 0.406217 0.264159  0.142058   True
reentrancy_simple.sol         8       8       5       5 0.001218 0.010425 -0.009206  False
simple_dao.sol                8       8       6       6 0.287243 0.008693  0.278550   True

== C5 seed 42, benar=8/8 ==
                         node_R 

In [ ]:
objs = {'v1': obj_v1, 'v2': obj_v2, 'v2_pos': obj_v2p, 'v3': obj_v3}
dasar = {g.nama: g for g in obj_v3['benchmark']}
print(f'{"kontrak":32}' + ''.join(f'{k:>14}' for k in objs))
for nama in sorted(dasar):
    baris = f'{nama:32}'
    for k, o in objs.items():
        g = {x.nama: x for x in o['benchmark']}[nama]
        ref = dasar[nama]
        sama = (g.num_nodes == ref.num_nodes and g.edge_index.shape[1] == ref.edge_index.shape[1]
                and g.x.shape == ref.x.shape[:1] + g.x.shape[1:]
                and (g.x[:, :ref.x.shape[1]] == ref.x).all().item() if g.x.shape[0] == ref.x.shape[0] else False)
        baris += f'{g.num_nodes}n/{g.edge_index.shape[1]}e{"=" if sama else "!"}'.rjust(14)
    print(baris)
print('= sama dengan v3 (node, edge, fitur), ! berbeda')

kontrak                                     v1            v2        v2_pos            v3
etherbank.sol                           8n/5e!        7n/4e=        7n/4e=        7n/4e=
etherbank_fixed.sol                     8n/5e!        7n/4e=        7n/4e=        7n/4e=
etherstore.sol                         10n/7e=       10n/7e=       10n/7e=       10n/7e=
etherstore_fixed.sol                   10n/7e=       10n/7e=       10n/7e=       10n/7e=
modifier_reentrancy.sol               14n/10e=      14n/10e=      14n/10e=      14n/10e=
modifier_reentrancy_fixed.sol           7n/4e=        7n/4e=        7n/4e=        7n/4e=
reentrance.sol                         10n/7e=       10n/7e=       10n/7e=       10n/7e=
reentrance_fixed.sol                   10n/7e=       10n/7e=       10n/7e=       10n/7e=
reentrancy_dao.sol                      9n/7e=        9n/7e=        9n/7e=        9n/7e=
reentrancy_dao_fixed.sol                9n/7e=        9n/7e=        9n/7e=        9n/7e=
reentrancy_insecure.s

In [ ]:
SUF = ['fold_metrics.csv', 'val_predictions.csv', 'benchmark_preds.csv', 'config.json', 'model_tiap_fold.pt']
def sudah_ada(kode, seed):
    return all(os.path.exists(f'{D}/hasil/{kode}_seed{seed}_{s}') for s in SUF)

def ringkasan_dari_disk(kode, seed):
    pre = f'{D}/hasil/{kode}_seed{seed}'
    m = pd.read_csv(f'{pre}_fold_metrics.csv'); r = m[m['fold'] == 'rata_rata'].iloc[0]
    auc_b, benar, _ = ringkas_benchmark(pd.read_csv(f'{pre}_benchmark_preds.csv'))
    return {'kode': kode, 'seed': seed, 'f1_cv': float(r.f1), 'auc_cv': float(r.auc),
            'auc_bench': auc_b, 'pasangan_benar': benar}

DATA = {'v2': (train_v2, bench_v2, obj_v2['dim'], 'nama'), 'v3': (train_v3, bench, DIM, 'contract_id')}
SEEDS5, SEEDS3 = [42, 1, 7, 123, 2024], [42, 1, 7]

def jalankan_jadwal(jadwal):
    for kode, mode, kelas, dkey, seeds in jadwal:
        tr, bn, dim, ga = DATA[dkey]
        for s in seeds:
            if sudah_ada(kode, s):
                print(f'lewati {kode} seed {s} (berkas sudah lengkap di Drive)'); continue
            jalankan(kode, mode, kelas, tr, bn, dim, s, ga)

jalankan_jadwal([('C3', 'weighted', MultiScaleGAT_MeanPool, 'v2', SEEDS5),
                 ('C6', 'weighted', GAT_SingleScale, 'v3', SEEDS3),
                 ('C5', 'weighted', MultiScaleGAT_MeanPool, 'v3', SEEDS5)])
print('selesai: C3, C6, C5')

lewati C3 seed 42 (berkas sudah lengkap di Drive)
  [C3 seed1] fold 1: F1=0.8571 AUC=0.9626 (epoch terbaik 13)
  [C3 seed1] fold 2: F1=0.8791 AUC=0.9199 (epoch terbaik 30)
  [C3 seed1] fold 3: F1=0.9038 AUC=0.9540 (epoch terbaik 31)
  [C3 seed1] fold 4: F1=0.9111 AUC=0.9574 (epoch terbaik 42)
  [C3 seed1] fold 5: F1=0.7711 AUC=0.8942 (epoch terbaik 14)
  C3_seed1_fold_metrics.csv: 448 B -> Drive 448 B OK
  C3_seed1_val_predictions.csv: 61,291 B -> Drive 61,291 B OK
  C3_seed1_benchmark_preds.csv: 2,208 B -> Drive 2,208 B OK
  C3_seed1_config.json: 1,552 B -> Drive 1,552 B OK
  C3_seed1_model_tiap_fold.pt: 12,621,635 B -> Drive 12,621,635 B OK

== C3 seed 1 (2.15 menit) ==
CV rata-rata: acc=0.8931 prec=0.8652 rec=0.8703 F1=0.8645 AUC=0.9376
Benchmark: AUC=0.5625, pasangan benar=4/8



/tmp/ipykernel_590/901933586.py:197: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value 'rata_rata' has dtype incompatible with float64, please explicitly cast to a compatible dtype first.
  rata = df_m.mean(numeric_only=True); rata['fold'] = 'rata_rata'


  [C3 seed7] fold 1: F1=0.8571 AUC=0.9439 (epoch terbaik 38)
  [C3 seed7] fold 2: F1=0.8831 AUC=0.9662 (epoch terbaik 49)
  [C3 seed7] fold 3: F1=0.8824 AUC=0.9243 (epoch terbaik 7)
  [C3 seed7] fold 4: F1=0.8941 AUC=0.9468 (epoch terbaik 49)
  [C3 seed7] fold 5: F1=0.8542 AUC=0.9226 (epoch terbaik 20)
  C3_seed7_fold_metrics.csv: 448 B -> Drive 448 B OK
  C3_seed7_val_predictions.csv: 61,291 B -> Drive 61,291 B OK
  C3_seed7_benchmark_preds.csv: 2,208 B -> Drive 2,208 B OK
  C3_seed7_config.json: 1,551 B -> Drive 1,551 B OK
  C3_seed7_model_tiap_fold.pt: 12,621,635 B -> Drive 12,621,635 B OK

== C3 seed 7 (2.14 menit) ==
CV rata-rata: acc=0.8985 prec=0.8755 rec=0.8774 F1=0.8742 AUC=0.9408
Benchmark: AUC=0.6250, pasangan benar=6/8



/tmp/ipykernel_590/901933586.py:197: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value 'rata_rata' has dtype incompatible with float64, please explicitly cast to a compatible dtype first.
  rata = df_m.mean(numeric_only=True); rata['fold'] = 'rata_rata'


  [C3 seed123] fold 1: F1=0.8608 AUC=0.9303 (epoch terbaik 29)
  [C3 seed123] fold 2: F1=0.9474 AUC=0.9731 (epoch terbaik 11)
  [C3 seed123] fold 3: F1=0.8736 AUC=0.9534 (epoch terbaik 35)
  [C3 seed123] fold 4: F1=0.8511 AUC=0.9055 (epoch terbaik 12)
  [C3 seed123] fold 5: F1=0.8387 AUC=0.9266 (epoch terbaik 35)
  C3_seed123_fold_metrics.csv: 448 B -> Drive 448 B OK
  C3_seed123_val_predictions.csv: 61,291 B -> Drive 61,291 B OK
  C3_seed123_benchmark_preds.csv: 2,208 B -> Drive 2,208 B OK
  C3_seed123_config.json: 1,554 B -> Drive 1,554 B OK
  C3_seed123_model_tiap_fold.pt: 12,621,907 B -> Drive 12,621,907 B OK

== C3 seed 123 (2.03 menit) ==
CV rata-rata: acc=0.8967 prec=0.8525 rec=0.9021 F1=0.8743 AUC=0.9378
Benchmark: AUC=0.6406, pasangan benar=6/8



/tmp/ipykernel_590/901933586.py:197: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value 'rata_rata' has dtype incompatible with float64, please explicitly cast to a compatible dtype first.
  rata = df_m.mean(numeric_only=True); rata['fold'] = 'rata_rata'


... (6 outputs left out)

  [C5 seed123] fold 1: F1=0.8523 AUC=0.9231 (epoch terbaik 35)
  [C5 seed123] fold 2: F1=0.8394 AUC=0.9522 (epoch terbaik 46)
  [C5 seed123] fold 3: F1=0.8797 AUC=0.9183 (epoch terbaik 13)
  [C5 seed123] fold 4: F1=0.8842 AUC=0.9508 (epoch terbaik 12)
  [C5 seed123] fold 5: F1=0.8199 AUC=0.9195 (epoch terbaik 26)
  C5_seed123_fold_metrics.csv: 448 B -> Drive 448 B OK
  C5_seed123_val_predictions.csv: 177,994 B -> Drive 177,994 B OK
  C5_seed123_benchmark_preds.csv: 2,208 B -> Drive 2,208 B OK
  C5_seed123_config.json: 1,567 B -> Drive 1,567 B OK
  C5_seed123_model_tiap_fold.pt: 12,621,907 B -> Drive 12,621,907 B OK

== C5 seed 123 (3.48 menit) ==
CV rata-rata: acc=0.8767 prec=0.8454 rec=0.8691 F1=0.8551 AUC=0.9328
Benchmark: AUC=0.6250, pasangan benar=6/8



/tmp/ipykernel_590/901933586.py:197: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value 'rata_rata' has dtype incompatible with float64, please explicitly cast to a compatible dtype first.
  rata = df_m.mean(numeric_only=True); rata['fold'] = 'rata_rata'


  [C5 seed2024] fold 1: F1=0.8808 AUC=0.9366 (epoch terbaik 8)
  [C5 seed2024] fold 2: F1=0.8603 AUC=0.9591 (epoch terbaik 45)
  [C5 seed2024] fold 3: F1=0.8516 AUC=0.9411 (epoch terbaik 47)
  [C5 seed2024] fold 4: F1=0.8357 AUC=0.8838 (epoch terbaik 45)
  [C5 seed2024] fold 5: F1=0.8986 AUC=0.9391 (epoch terbaik 21)
  C5_seed2024_fold_metrics.csv: 448 B -> Drive 448 B OK
  C5_seed2024_val_predictions.csv: 177,994 B -> Drive 177,994 B OK
  C5_seed2024_benchmark_preds.csv: 2,208 B -> Drive 2,208 B OK
  C5_seed2024_config.json: 1,567 B -> Drive 1,567 B OK
  C5_seed2024_model_tiap_fold.pt: 12,622,043 B -> Drive 12,622,043 B OK

== C5 seed 2024 (3.73 menit) ==
CV rata-rata: acc=0.8882 prec=0.8663 rec=0.8661 F1=0.8654 AUC=0.9320
Benchmark: AUC=0.6562, pasangan benar=7/8

selesai: C3, C6, C5


/tmp/ipykernel_590/901933586.py:197: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value 'rata_rata' has dtype incompatible with float64, please explicitly cast to a compatible dtype first.
  rata = df_m.mean(numeric_only=True); rata['fold'] = 'rata_rata'


In [ ]:
bench_v3 = obj_v3['benchmark']

def evaluasi_gnn_pada_bench_v3(kode, kelas, seeds):
    for s in seeds:
        if not sudah_ada(kode, s): continue
        sds = torch.load(f'{D}/hasil/{kode}_seed{s}_model_tiap_fold.pt', weights_only=False)
        bp = []
        for sd in sds:
            m = kelas(dim_input=22).to(DEV); m.load_state_dict(sd); m.eval()
            p, _ = prediksi(m, bench_v3); bp.append(p)
        df_b = bangun_df_bench(bench_v3, bp)
        auc_b, benar, _ = ringkas_benchmark(df_b)
        asli = pd.read_csv(f'{D}/hasil/{kode}_seed{s}_benchmark_preds.csv')
        auc_a, benar_a, _ = ringkas_benchmark(asli)
        print(f'{kode} seed {s}: benchmark v3-graf AUC={auc_b:.4f}, pasangan={benar}/8 | benchmark bawaan: AUC={auc_a:.4f}, pasangan={benar_a}/8')

evaluasi_gnn_pada_bench_v3('C1', MultiScaleGAT_SAGPool, [42])

def evaluasi_B0_pada_bench_v3(seeds):
    Xb = np.array([fitur_ukuran(d) for d in bench_v3]); yb = np.array([int(d.y) for d in bench_v3])
    for s in seeds:
        st = torch.load(f'{D}/hasil/B0_seed{s}_model_tiap_fold.pt', weights_only=False)
        p = np.mean([1 / (1 + np.exp(-(Xb @ x['coef'].ravel() + x['intercept'][0]))) for x in st], axis=0)
        df_b = bangun_df_bench(bench_v3, [p])
        auc_b, benar, _ = ringkas_benchmark(df_b)
        print(f'B0 seed {s}: benchmark v3-graf AUC={auc_b:.4f}, pasangan={benar}/8')
evaluasi_B0_pada_bench_v3([42])

C1 seed 42: benchmark v3-graf AUC=0.5938, pasangan=6/8 | benchmark bawaan: AUC=0.5625, pasangan=6/8
B0 seed 42: benchmark v3-graf AUC=0.5938, pasangan=1/8


In [ ]:
jalankan_jadwal([('C9', 'smote', MultiScaleGAT_MeanPool, 'v3', SEEDS5),
                 ('C7', 'weighted', MultiScaleGAT_SAGPool, 'v3', SEEDS3)])

rows = []
for kode, seeds in [('C3', SEEDS5), ('C5', SEEDS5), ('C9', SEEDS5), ('C6', SEEDS3), ('C7', SEEDS3)]:
    rows += [ringkasan_dari_disk(kode, s) for s in seeds if sudah_ada(kode, s)]
df_all = pd.DataFrame(rows)
print(df_all.to_string(index=False))
ag = df_all.groupby('kode').agg(n_seed=('seed', 'count'),
    f1_mean=('f1_cv', 'mean'), f1_sd=('f1_cv', 'std'), auc_cv_mean=('auc_cv', 'mean'), auc_cv_sd=('auc_cv', 'std'),
    bench_mean=('auc_bench', 'mean'), bench_sd=('auc_bench', 'std'),
    pasangan_mean=('pasangan_benar', 'mean'), pasangan_sd=('pasangan_benar', 'std'))
print('\n', ag.round(4).to_string())
df_all.to_csv('/content/ringkasan_semua_run.csv', index=False, float_format='%.10f')
shutil.copy('/content/ringkasan_semua_run.csv', f'{D}/hasil/ringkasan_semua_run.csv')
assert os.path.getsize('/content/ringkasan_semua_run.csv') == os.path.getsize(f'{D}/hasil/ringkasan_semua_run.csv')

lewati C9 seed 42 (berkas sudah lengkap di Drive)
lewati C9 seed 1 (berkas sudah lengkap di Drive)
lewati C9 seed 7 (berkas sudah lengkap di Drive)
  [C9 seed123] fold 1: F1=0.8526 AUC=0.9254 (epoch terbaik 27)
  [C9 seed123] fold 2: F1=0.8387 AUC=0.9341 (epoch terbaik 31)
  [C9 seed123] fold 3: F1=0.8731 AUC=0.9411 (epoch terbaik 36)
  [C9 seed123] fold 4: F1=0.8973 AUC=0.9602 (epoch terbaik 32)
  [C9 seed123] fold 5: F1=0.8207 AUC=0.9032 (epoch terbaik 15)
  C9_seed123_fold_metrics.csv: 448 B -> Drive 448 B OK
  C9_seed123_val_predictions.csv: 177,994 B -> Drive 177,994 B OK
  C9_seed123_benchmark_preds.csv: 2,208 B -> Drive 2,208 B OK
  C9_seed123_config.json: 1,561 B -> Drive 1,561 B OK
  C9_seed123_model_tiap_fold.pt: 12,621,907 B -> Drive 12,621,907 B OK

== C9 seed 123 (4.21 menit) ==
CV rata-rata: acc=0.8776 prec=0.8428 rec=0.8732 F1=0.8565 AUC=0.9328
Benchmark: AUC=0.7500, pasangan benar=8/8



/tmp/ipykernel_590/901933586.py:197: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value 'rata_rata' has dtype incompatible with float64, please explicitly cast to a compatible dtype first.
  rata = df_m.mean(numeric_only=True); rata['fold'] = 'rata_rata'


  [C9 seed2024] fold 1: F1=0.8962 AUC=0.9496 (epoch terbaik 33)
  [C9 seed2024] fold 2: F1=0.8288 AUC=0.9263 (epoch terbaik 10)
  [C9 seed2024] fold 3: F1=0.8504 AUC=0.9398 (epoch terbaik 39)
  [C9 seed2024] fold 4: F1=0.8241 AUC=0.8867 (epoch terbaik 15)
  [C9 seed2024] fold 5: F1=0.8889 AUC=0.9422 (epoch terbaik 29)
  C9_seed2024_fold_metrics.csv: 448 B -> Drive 448 B OK
  C9_seed2024_val_predictions.csv: 177,994 B -> Drive 177,994 B OK
  C9_seed2024_benchmark_preds.csv: 2,208 B -> Drive 2,208 B OK
  C9_seed2024_config.json: 1,562 B -> Drive 1,562 B OK
  C9_seed2024_model_tiap_fold.pt: 12,622,043 B -> Drive 12,622,043 B OK

== C9 seed 2024 (3.79 menit) ==
CV rata-rata: acc=0.8795 prec=0.8467 rec=0.8694 F1=0.8577 AUC=0.9289
Benchmark: AUC=0.6875, pasangan benar=7/8

lewati C7 seed 42 (berkas sudah lengkap di Drive)


/tmp/ipykernel_590/901933586.py:197: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value 'rata_rata' has dtype incompatible with float64, please explicitly cast to a compatible dtype first.
  rata = df_m.mean(numeric_only=True); rata['fold'] = 'rata_rata'


  [C7 seed1] fold 1: F1=0.8297 AUC=0.9026 (epoch terbaik 39)
  [C7 seed1] fold 2: F1=0.8608 AUC=0.9134 (epoch terbaik 35)
  [C7 seed1] fold 3: F1=0.7774 AUC=0.8541 (epoch terbaik 12)
  [C7 seed1] fold 4: F1=0.7965 AUC=0.8978 (epoch terbaik 43)
  [C7 seed1] fold 5: F1=0.8087 AUC=0.8596 (epoch terbaik 17)
  C7_seed1_fold_metrics.csv: 448 B -> Drive 448 B OK
  C7_seed1_val_predictions.csv: 177,994 B -> Drive 177,994 B OK
  C7_seed1_benchmark_preds.csv: 2,208 B -> Drive 2,208 B OK
  C7_seed1_config.json: 1,563 B -> Drive 1,563 B OK
  C7_seed1_model_tiap_fold.pt: 12,648,431 B -> Drive 12,648,431 B OK

== C7 seed 1 (4.2 menit) ==
CV rata-rata: acc=0.8485 prec=0.8340 rec=0.8012 F1=0.8146 AUC=0.8855
Benchmark: AUC=0.4219, pasangan benar=2/8



/tmp/ipykernel_590/901933586.py:197: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value 'rata_rata' has dtype incompatible with float64, please explicitly cast to a compatible dtype first.
  rata = df_m.mean(numeric_only=True); rata['fold'] = 'rata_rata'


  [C7 seed7] fold 1: F1=0.7917 AUC=0.8810 (epoch terbaik 20)
  [C7 seed7] fold 2: F1=0.8571 AUC=0.9264 (epoch terbaik 49)
  [C7 seed7] fold 3: F1=0.8889 AUC=0.9408 (epoch terbaik 27)
  [C7 seed7] fold 4: F1=0.8392 AUC=0.9088 (epoch terbaik 21)
  [C7 seed7] fold 5: F1=0.8514 AUC=0.9243 (epoch terbaik 22)
  C7_seed7_fold_metrics.csv: 448 B -> Drive 448 B OK
  C7_seed7_val_predictions.csv: 177,994 B -> Drive 177,994 B OK
  C7_seed7_benchmark_preds.csv: 2,208 B -> Drive 2,208 B OK
  C7_seed7_config.json: 1,563 B -> Drive 1,563 B OK
  C7_seed7_model_tiap_fold.pt: 12,648,431 B -> Drive 12,648,431 B OK

== C7 seed 7 (4.03 menit) ==
CV rata-rata: acc=0.8695 prec=0.8314 rec=0.8632 F1=0.8457 AUC=0.9163
Benchmark: AUC=0.3750, pasangan benar=4/8



/tmp/ipykernel_590/901933586.py:197: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value 'rata_rata' has dtype incompatible with float64, please explicitly cast to a compatible dtype first.
  rata = df_m.mean(numeric_only=True); rata['fold'] = 'rata_rata'


kode  seed    f1_cv   auc_cv  auc_bench  pasangan_benar
  C3    42 0.868991 0.918971   0.593750               5
  C3     1 0.864461 0.937635   0.562500               4
  C3     7 0.874179 0.940757   0.625000               6
  C3   123 0.874293 0.937803   0.640625               6
  C3  2024 0.852899 0.930538   0.593750               5
  C5    42 0.867388 0.934002   0.781250               8
  C5     1 0.856736 0.929782   0.609375               6
  C5     7 0.878736 0.937769   0.734375               7
  C5   123 0.855114 0.932772   0.625000               6
  C5  2024 0.865383 0.931957   0.656250               7
  C9    42 0.858643 0.930045   0.703125               7
  C9     1 0.872742 0.936100   0.609375               6
  C9     7 0.877602 0.940877   0.656250               6
  C9   123 0.856490 0.932774   0.750000               8
  C9  2024 0.857660 0.928918   0.687500               7
  C6    42 0.861109 0.934943   0.640625               7
  C6     1 0.866565 0.935010   0.593750         

In [ ]:
DATA['v1'] = (obj_v1['train'], obj_v1['benchmark'], obj_v1['dim'], 'nama')
DATA['v2p'] = (obj_v2p['train'], obj_v2p['benchmark'], obj_v2p['dim'], 'nama')
jalankan_jadwal([('C1', 'weighted', MultiScaleGAT_SAGPool, 'v1', SEEDS3),
                 ('C2', 'weighted', MultiScaleGAT_SAGPool, 'v2', SEEDS3),
                 ('C4', 'weighted', MultiScaleGAT_MeanPool, 'v2p', SEEDS3),
                 ('C8', 'smote', MultiScaleGAT_MeanPool, 'v2', SEEDS3)])

DAFTAR = [('B0', SEEDS5), ('C1', SEEDS3), ('C2', SEEDS3), ('C3', SEEDS5), ('C4', SEEDS3),
          ('C5', SEEDS5), ('C6', SEEDS3), ('C7', SEEDS3), ('C8', SEEDS3), ('C9', SEEDS5)]
rows = [ringkasan_dari_disk(k, s) for k, seeds in DAFTAR for s in seeds if sudah_ada(k, s)]
df_all = pd.DataFrame(rows)
ag = df_all.groupby('kode').agg(n_seed=('seed', 'count'),
    f1_mean=('f1_cv', 'mean'), f1_sd=('f1_cv', 'std'),
    auc_cv_mean=('auc_cv', 'mean'), auc_cv_sd=('auc_cv', 'std'),
    bench_mean=('auc_bench', 'mean'), bench_sd=('auc_bench', 'std'),
    pasangan_mean=('pasangan_benar', 'mean'), pasangan_sd=('pasangan_benar', 'std'))
print(df_all.to_string(index=False)); print('\n', ag.round(4).to_string())
df_all.to_csv('/content/ringkasan_semua_run.csv', index=False, float_format='%.10f')
shutil.copy('/content/ringkasan_semua_run.csv', f'{D}/hasil/ringkasan_semua_run.csv')
assert os.path.getsize('/content/ringkasan_semua_run.csv') == os.path.getsize(f'{D}/hasil/ringkasan_semua_run.csv')

lewati C1 seed 42 (berkas sudah lengkap di Drive)
  [C1 seed1] fold 1: F1=0.8750 AUC=0.9315 (epoch terbaik 46)
  [C1 seed1] fold 2: F1=0.9038 AUC=0.9327 (epoch terbaik 16)
  [C1 seed1] fold 3: F1=0.8500 AUC=0.9536 (epoch terbaik 39)
  [C1 seed1] fold 4: F1=0.8506 AUC=0.9373 (epoch terbaik 14)
  [C1 seed1] fold 5: F1=0.9091 AUC=0.9675 (epoch terbaik 31)
  C1_seed1_fold_metrics.csv: 448 B -> Drive 448 B OK
  C1_seed1_val_predictions.csv: 78,693 B -> Drive 78,693 B OK
  C1_seed1_benchmark_preds.csv: 2,208 B -> Drive 2,208 B OK
  C1_seed1_config.json: 1,551 B -> Drive 1,551 B OK
  C1_seed1_model_tiap_fold.pt: 12,648,431 B -> Drive 12,648,431 B OK

== C1 seed 1 (2.29 menit) ==
CV rata-rata: acc=0.9238 prec=0.8700 rec=0.8878 F1=0.8777 AUC=0.9445
Benchmark: AUC=0.5156, pasangan benar=6/8



/tmp/ipykernel_590/901933586.py:197: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value 'rata_rata' has dtype incompatible with float64, please explicitly cast to a compatible dtype first.
  rata = df_m.mean(numeric_only=True); rata['fold'] = 'rata_rata'


  [C1 seed7] fold 1: F1=0.8049 AUC=0.9259 (epoch terbaik 50)
  [C1 seed7] fold 2: F1=0.8980 AUC=0.9451 (epoch terbaik 21)
  [C1 seed7] fold 3: F1=0.8916 AUC=0.9644 (epoch terbaik 48)
  [C1 seed7] fold 4: F1=0.7826 AUC=0.9595 (epoch terbaik 18)
  [C1 seed7] fold 5: F1=0.8571 AUC=0.9467 (epoch terbaik 11)
  C1_seed7_fold_metrics.csv: 448 B -> Drive 448 B OK
  C1_seed7_val_predictions.csv: 78,693 B -> Drive 78,693 B OK
  C1_seed7_benchmark_preds.csv: 2,208 B -> Drive 2,208 B OK
  C1_seed7_config.json: 1,551 B -> Drive 1,551 B OK
  C1_seed7_model_tiap_fold.pt: 12,648,431 B -> Drive 12,648,431 B OK

== C1 seed 7 (2.13 menit) ==
CV rata-rata: acc=0.9095 prec=0.8713 rec=0.8245 F1=0.8468 AUC=0.9483
Benchmark: AUC=0.3750, pasangan benar=3/8

lewati C2 seed 42 (berkas sudah lengkap di Drive)


/tmp/ipykernel_590/901933586.py:197: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value 'rata_rata' has dtype incompatible with float64, please explicitly cast to a compatible dtype first.
  rata = df_m.mean(numeric_only=True); rata['fold'] = 'rata_rata'


  [C2 seed1] fold 1: F1=0.8169 AUC=0.8835 (epoch terbaik 4)
  [C2 seed1] fold 2: F1=0.7872 AUC=0.8353 (epoch terbaik 2)
  [C2 seed1] fold 3: F1=0.9293 AUC=0.9533 (epoch terbaik 33)
  [C2 seed1] fold 4: F1=0.8602 AUC=0.8959 (epoch terbaik 13)
  [C2 seed1] fold 5: F1=0.8148 AUC=0.9139 (epoch terbaik 14)
  C2_seed1_fold_metrics.csv: 448 B -> Drive 448 B OK
  C2_seed1_val_predictions.csv: 61,291 B -> Drive 61,291 B OK
  C2_seed1_benchmark_preds.csv: 2,208 B -> Drive 2,208 B OK
  C2_seed1_config.json: 1,548 B -> Drive 1,548 B OK
  C2_seed1_model_tiap_fold.pt: 12,648,431 B -> Drive 12,648,431 B OK

== C2 seed 1 (1.6 menit) ==
CV rata-rata: acc=0.8745 prec=0.8413 rec=0.8479 F1=0.8417 AUC=0.8964
Benchmark: AUC=0.5000, pasangan benar=4/8



/tmp/ipykernel_590/901933586.py:197: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value 'rata_rata' has dtype incompatible with float64, please explicitly cast to a compatible dtype first.
  rata = df_m.mean(numeric_only=True); rata['fold'] = 'rata_rata'


... (7 outputs left out)

/tmp/ipykernel_590/901933586.py:197: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value 'rata_rata' has dtype incompatible with float64, please explicitly cast to a compatible dtype first.
  rata = df_m.mean(numeric_only=True); rata['fold'] = 'rata_rata'


  [C8 seed7] fold 1: F1=0.8706 AUC=0.9461 (epoch terbaik 17)
  [C8 seed7] fold 2: F1=0.9114 AUC=0.9787 (epoch terbaik 45)
  [C8 seed7] fold 3: F1=0.8269 AUC=0.9181 (epoch terbaik 5)
  [C8 seed7] fold 4: F1=0.8780 AUC=0.9373 (epoch terbaik 49)
  [C8 seed7] fold 5: F1=0.8776 AUC=0.9250 (epoch terbaik 30)
  C8_seed7_fold_metrics.csv: 448 B -> Drive 448 B OK
  C8_seed7_val_predictions.csv: 61,291 B -> Drive 61,291 B OK
  C8_seed7_benchmark_preds.csv: 2,208 B -> Drive 2,208 B OK
  C8_seed7_config.json: 1,545 B -> Drive 1,545 B OK
  C8_seed7_model_tiap_fold.pt: 12,621,635 B -> Drive 12,621,635 B OK

== C8 seed 7 (2.14 menit) ==
CV rata-rata: acc=0.8930 prec=0.8532 rec=0.8982 F1=0.8729 AUC=0.9410
Benchmark: AUC=0.6094, pasangan benar=5/8



/tmp/ipykernel_590/901933586.py:197: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value 'rata_rata' has dtype incompatible with float64, please explicitly cast to a compatible dtype first.
  rata = df_m.mean(numeric_only=True); rata['fold'] = 'rata_rata'


kode  seed    f1_cv   auc_cv  auc_bench  pasangan_benar
  B0    42 0.717002 0.873036   0.617188               1
  B0     1 0.714222 0.872335   0.617188               1
  B0     7 0.714179 0.875536   0.617188               1
  B0   123 0.714476 0.871308   0.617188               1
  B0  2024 0.715484 0.873432   0.617188               1
  C1    42 0.840296 0.933639   0.562500               6
  C1     1 0.877702 0.944528   0.515625               6
  C1     7 0.846831 0.948326   0.375000               3
  C2    42 0.824445 0.898028   0.593750               6
  C2     1 0.841692 0.896379   0.500000               4
  C2     7 0.851706 0.914281   0.546875               4
  C3    42 0.868991 0.918971   0.593750               5
  C3     1 0.864461 0.937635   0.562500               4
  C3     7 0.874179 0.940757   0.625000               6
  C3   123 0.874293 0.937803   0.640625               6
  C3  2024 0.852899 0.930538   0.593750               5
  C4    42 0.879006 0.938826   0.656250         

In [ ]:
from scipy.special import logit as _logit
def lg(p): return _logit(np.clip(p, 1e-9, 1 - 1e-9))
baris = []
for kode, seeds in DAFTAR:
    for s in seeds:
        if not sudah_ada(kode, s): continue
        df = pd.read_csv(f'{D}/hasil/{kode}_seed{s}_benchmark_preds.csv')
        r = df[df.tipe == 'rentan'].set_index('pasangan'); a = df[df.tipe == 'aman'].set_index('pasangan')
        for p in r.index:
            baris.append({'kode': kode, 'seed': s, 'pasangan': p, 'prob_R': r.prob_rata[p], 'prob_A': a.prob_rata[p]})
dfp = pd.DataFrame(baris)
dfp['selisih'] = dfp.prob_R - dfp.prob_A
dfp['margin_logit'] = lg(dfp.prob_R) - lg(dfp.prob_A)
dfp['benar'] = dfp.selisih > 0
dfp['tak_terpisah'] = dfp.selisih.abs() < 0.01

urut = [k for k, _ in DAFTAR]
print('=== win rate per pasangan (fraksi seed dengan versi rentan > versi aman) ===')
print(dfp.pivot_table(index='pasangan', columns='kode', values='benar', aggfunc='mean')[urut].round(2).to_string())
per_run = dfp.groupby(['kode', 'seed']).agg(margin=('margin_logit', 'mean'), tak_terpisah=('tak_terpisah', 'sum')).reset_index()
print('\n=== margin log-odds rata-rata per run, lalu mean dan simpangan baku antar seed ===')
print(per_run.groupby('kode').agg(margin_mean=('margin', 'mean'), margin_sd=('margin', 'std'),
      tak_terpisah_mean=('tak_terpisah', 'mean')).loc[urut].round(3).to_string())
dfp.to_csv('/content/benchmark_per_pasangan_semua_run.csv', index=False, float_format='%.10f')
shutil.copy('/content/benchmark_per_pasangan_semua_run.csv', f'{D}/hasil/benchmark_per_pasangan_semua_run.csv')
assert os.path.getsize('/content/benchmark_per_pasangan_semua_run.csv') == os.path.getsize(f'{D}/hasil/benchmark_per_pasangan_semua_run.csv')

=== win rate per pasangan (fraksi seed dengan versi rentan > versi aman) ===
kode                          B0       C1       C2       C3       C4       C5       C6       C7       C8       C9
pasangan                                                                                                         
etherbank.sol           0.000000 0.330000 0.670000 0.800000 0.670000 0.600000 0.330000 0.330000 0.670000 0.600000
etherstore.sol          0.000000 0.670000 0.330000 0.000000 0.000000 0.200000 0.000000 0.000000 0.000000 0.200000
modifier_reentrancy.sol 1.000000 1.000000 1.000000 1.000000 1.000000 1.000000 1.000000 0.000000 1.000000 1.000000
reentrance.sol          0.000000 0.670000 0.330000 0.800000 1.000000 1.000000 1.000000 0.670000 0.330000 1.000000
reentrancy_dao.sol      0.000000 1.000000 0.670000 0.800000 1.000000 1.000000 1.000000 0.670000 1.000000 1.000000
reentrancy_insecure.sol 0.000000 0.330000 0.000000 0.800000 1.000000 1.000000 1.000000 0.670000 1.000000 1.000000
reentrancy_

In [ ]:
from math import comb
def iou_acak_eksak(n, m):
    if m == 0: return np.nan
    tot = comb(n, 2); e = 0.0
    for j in range(0, min(2, m) + 1):
        if 2 - j > n - m: continue
        e += comb(m, j) * comb(n - m, 2 - j) / tot * (j / (2 + m - j))
    return e

SHAP_SEMUA, NODE_SEMUA = [], []
for kode, db, dim in KONF_SHAP:
    for s in SEEDS5:
        r, nd, _, _ = jalankan_shap(kode, s, db, dim)
        r['seed'] = s; nd['seed'] = s
        r['iou_acak_eksak'] = [iou_acak_eksak(n, m) for n, m in zip(r.n_node, r.n_gt)]
        SHAP_SEMUA.append(r); NODE_SEMUA.append(nd)
        print(f'{kode} seed {s}: IoU={r.iou_k2.mean():.3f}, flat={int((r.pola=="flat").sum())}/16, top1 di GT={int(r.top1_gt.sum())}/16')
dfs = pd.concat(SHAP_SEMUA, ignore_index=True); dfn = pd.concat(NODE_SEMUA, ignore_index=True)

per_seed = dfs.groupby(['model', 'seed']).agg(iou=('iou_k2', 'mean'), flat=('pola', lambda x: int((x == 'flat').sum())),
                                              top1=('top1_gt', 'sum')).reset_index()
print('\n=== IoU per model: mean dan simpangan baku antar seed ===')
print(per_seed.groupby('model').agg(iou_mean=('iou', 'mean'), iou_sd=('iou', 'std'),
      flat_mean=('flat', 'mean'), top1_mean=('top1', 'mean')).round(3).to_string())

# bootstrap antar kontrak (IoU tiap kontrak dirata-rata antar seed dulu), seed RNG tetap = 0
rng = np.random.default_rng(0)
pk = dfs.groupby(['model', 'kontrak']).agg(iou=('iou_k2', 'mean'), acak_eksak=('iou_acak_eksak', 'first'),
                                           acak_rasio=('iou_acak', 'first')).reset_index()
lebar = pk.pivot(index='kontrak', columns='model', values='iou').dropna()
acak = pk.drop_duplicates('kontrak').set_index('kontrak').loc[lebar.index]
B = 10000; idx = rng.integers(0, len(lebar), (B, len(lebar)))
print(f'\nbootstrap {B}x atas {len(lebar)} kontrak (seed RNG 0):')
for m in lebar.columns:
    bs = lebar[m].values[idx].mean(1)
    print(f'  {m}: IoU={lebar[m].mean():.3f}, CI95=({np.percentile(bs, 2.5):.3f}, {np.percentile(bs, 97.5):.3f})')
for a_, b_ in [('C3', 'C5'), ('C5', 'C9'), ('C3', 'C9')]:
    d = (lebar[a_] - lebar[b_]).values[idx].mean(1)
    print(f'  selisih {a_}-{b_}: {(lebar[a_]-lebar[b_]).mean():+.3f}, CI95=({np.percentile(d, 2.5):+.3f}, {np.percentile(d, 97.5):+.3f})')
print(f'baseline acak: rasio nilai harapan={acak.acak_rasio.mean():.3f}, nilai harapan eksak={acak.acak_eksak.mean():.3f}')

dfs.to_csv('/content/shap_ringkasan_semua_seed.csv', index=False, float_format='%.10f')
dfn.to_csv('/content/shap_nilai_per_node_semua_seed.csv', index=False, float_format='%.10f')
for f in ['shap_ringkasan_semua_seed.csv', 'shap_nilai_per_node_semua_seed.csv']:
    shutil.copy(f'/content/{f}', f'{D}/hasil/{f}')
    s1, s2 = os.path.getsize(f'/content/{f}'), os.path.getsize(f'{D}/hasil/{f}')
    print(f'{f}: {s1:,} B -> Drive {s2:,} B {"OK" if s1 == s2 else "BEDA!"}'); assert s1 == s2

C3 seed 42: IoU=0.333, flat=2/16, top1 di GT=5/16
C3 seed 1: IoU=0.228, flat=4/16, top1 di GT=7/16
C3 seed 7: IoU=0.333, flat=0/16, top1 di GT=6/16
C3 seed 123: IoU=0.302, flat=3/16, top1 di GT=7/16
C3 seed 2024: IoU=0.369, flat=1/16, top1 di GT=7/16
C5 seed 42: IoU=0.306, flat=5/16, top1 di GT=8/16
C5 seed 1: IoU=0.182, flat=4/16, top1 di GT=5/16
C5 seed 7: IoU=0.222, flat=7/16, top1 di GT=4/16
C5 seed 123: IoU=0.211, flat=5/16, top1 di GT=9/16
C5 seed 2024: IoU=0.222, flat=5/16, top1 di GT=3/16
C9 seed 42: IoU=0.223, flat=6/16, top1 di GT=7/16
C9 seed 1: IoU=0.198, flat=5/16, top1 di GT=4/16
C9 seed 7: IoU=0.192, flat=6/16, top1 di GT=5/16
C9 seed 123: IoU=0.227, flat=9/16, top1 di GT=6/16
C9 seed 2024: IoU=0.193, flat=6/16, top1 di GT=5/16

=== IoU per model: mean dan simpangan baku antar seed ===
       iou_mean   iou_sd  flat_mean  top1_mean
model                                         
C3     0.313000 0.053000   2.000000   6.400000
C5     0.229000 0.046000   5.200000   5.800000


In [ ]:
kon = dfs.drop_duplicates('kontrak')[['kontrak', 'n_node', 'n_gt']].copy()
kon['p_acak'] = kon.n_gt / kon.n_node
E = kon.p_acak.sum(); SD = np.sqrt((kon.p_acak * (1 - kon.p_acak)).sum())
print(f'baseline acak: harapan top-1 di GT = {E:.2f} dari {len(kon)} kontrak (SD {SD:.2f})')

dist = np.array([1.0])
for p in kon.p_acak: dist = np.convolve(dist, [1 - p, p])          # distribusi eksak jumlah kena
obs = dfs.groupby(['model', 'seed']).top1_gt.sum().reset_index()
obs['p_ge_obs'] = [dist[int(k):].sum() for k in obs.top1_gt]       # P(acak >= teramati)
print('\n=== per run: top-1 teramati dan peluang acak mencapai nilai itu atau lebih ===')
print(obs.round(3).to_string(index=False))
print('\n=== ringkasan per model ===')
ring = obs.groupby('model').agg(top1_mean=('top1_gt', 'mean'), top1_sd=('top1_gt', 'std'),
                                p_median=('p_ge_obs', 'median')).round(3)
ring['harapan_acak'] = round(E, 2)
print(ring.to_string())

hit = dfs.pivot_table(index='kontrak', columns='model', values='top1_gt', aggfunc='mean')
hit['acak'] = kon.set_index('kontrak').p_acak
print('\n=== fraksi seed dengan top-1 di GT, per kontrak (acak = m/n) ===')
print(hit.round(2).to_string())

kon.to_csv('/content/baseline_top1_acak.csv', index=False, float_format='%.10f')
obs.to_csv('/content/top1_vs_acak_per_run.csv', index=False, float_format='%.10f')
for f in ['baseline_top1_acak.csv', 'top1_vs_acak_per_run.csv']:
    shutil.copy(f'/content/{f}', f'{D}/hasil/{f}')
    assert os.path.getsize(f'/content/{f}') == os.path.getsize(f'{D}/hasil/{f}')
    print(f'{f}: {os.path.getsize(f"/content/{f}"):,} B -> Drive OK')

baseline acak: harapan top-1 di GT = 6.73 dari 16 kontrak (SD 1.95)

=== per run: top-1 teramati dan peluang acak mencapai nilai itu atau lebih ===
model  seed  top1_gt  p_ge_obs
   C3     1        7  0.543000
   C3     7        6  0.734000
   C3    42        5  0.876000
   C3   123        7  0.543000
   C3  2024        7  0.543000
   C5     1        5  0.876000
   C5     7        4  0.956000
   C5    42        8  0.344000
   C5   123        9  0.182000
   C5  2024        3  0.989000
   C9     1        4  0.956000
   C9     7        5  0.876000
   C9    42        7  0.543000
   C9   123        6  0.734000
   C9  2024        5  0.876000

=== ringkasan per model ===
       top1_mean  top1_sd  p_median  harapan_acak
model                                            
C3      6.400000 0.894000  0.543000      6.730000
C5      5.800000 2.588000  0.876000      6.730000
C9      5.400000 1.140000  0.876000      6.730000

=== fraksi seed dengan top-1 di GT, per kontrak (acak = m/n) ===
model      

In [ ]:
script = r'''
import sys, os, json, copy, numpy as np, torch, torch.nn as nn, torch.nn.functional as F
from torch_geometric.loader import DataLoader
from torch_geometric.nn import GATConv, global_mean_pool
from sklearn.model_selection import StratifiedGroupKFold
from sklearn.metrics import f1_score
import warnings
mode = sys.argv[1]; D = '/content/drive/MyDrive/PROJECT'; DEV = torch.device('cuda')
nd_peringatan = []
if mode == 'deterministik':
    torch.use_deterministic_algorithms(True, warn_only=True)
    torch.backends.cudnn.deterministic = True; torch.backends.cudnn.benchmark = False
warnings.showwarning = lambda m, c, f, l, file=None, line=None: nd_peringatan.append(str(m)[:90])

class M(nn.Module):
    def __init__(s, dim_input=22, h=512, k=2, p=0.5):
        super().__init__()
        s.gat1 = GATConv(dim_input, h // 8, heads=8, dropout=p); s.gat2 = GATConv(h, h // 4, heads=4, dropout=p)
        s.gat3 = GATConv(h, h // 4, heads=4, dropout=p); s.proj = nn.Linear(dim_input, h)
        s.n1 = nn.LayerNorm(h); s.n2 = nn.LayerNorm(h); s.n3 = nn.LayerNorm(h)
        s.cls = nn.Sequential(nn.Linear(h, 128), nn.ReLU(), nn.Dropout(p), nn.Linear(128, 64), nn.ReLU(), nn.Dropout(p), nn.Linear(64, k))
        s.p = p
    def forward(s, d):
        h0 = s.proj(d.x); h1 = s.n1(F.elu(s.gat1(d.x, d.edge_index)) + h0); h1 = F.dropout(h1, p=s.p, training=s.training)
        h2 = s.n2(F.elu(s.gat2(h1, d.edge_index)) + h1); h2 = F.dropout(h2, p=s.p, training=s.training)
        h3 = s.n3(F.elu(s.gat3(h2, d.edge_index)) + h2)
        return s.cls(global_mean_pool(h3, d.batch))

@torch.no_grad()
def prob(m, data):
    m.eval(); out = []
    for b in DataLoader(data, batch_size=32):
        out += torch.softmax(m(b.to(DEV)), 1)[:, 1].cpu().tolist()
    return np.array(out)

o = torch.load(f'{D}/data/processed/pyg/v3_augmented_dedup.pt', weights_only=False)
tr_all, bench = o['train'], o['benchmark']
y = np.array([int(d.y) for d in tr_all]); g = np.array([d.nama.split('::')[0] for d in tr_all])
i_tr, i_va = next(iter(StratifiedGroupKFold(5, shuffle=True, random_state=42).split(np.zeros(len(y)), y, g)))
tr = [tr_all[i] for i in i_tr]; va = [tr_all[i] for i in i_va]; yv = y[i_va]

def latih():
    torch.manual_seed(42)
    m = M(dim_input=o['dim']).to(DEV); opt = torch.optim.AdamW(m.parameters(), lr=1e-3, weight_decay=1e-4)
    yt = np.array([int(d.y) for d in tr]); n0, n1 = (yt == 0).sum(), (yt == 1).sum()
    lf = nn.CrossEntropyLoss(weight=torch.tensor([len(yt) / (2 * n0), len(yt) / (2 * n1)], dtype=torch.float).to(DEV))
    ld = DataLoader(tr, batch_size=32, shuffle=True); best, bw, mandek, ep_b = -1, None, 0, 0
    for ep in range(1, 51):
        m.train()
        for b in ld:
            b = b.to(DEV); opt.zero_grad(); lf(m(b), b.y).backward()
            torch.nn.utils.clip_grad_norm_(m.parameters(), 1.0); opt.step()
        f1 = f1_score(yv, (prob(m, va) >= 0.5).astype(int), zero_division=0)
        if f1 > best: best, bw, mandek, ep_b = f1, copy.deepcopy(m.state_dict()), 0, ep
        else: mandek += 1
        if mandek >= 15: break
    m.load_state_dict(bw); return m, ep_b

mA, eA = latih(); mB, eB = latih()
pA, pB = prob(mA, va), prob(mB, va); bA, bB = prob(mA, bench), prob(mB, bench)
print(json.dumps({'mode': mode, 'epoch_terbaik': [eA, eB], 'maks_selisih_prob_val': float(np.abs(pA - pB).max()),
                  'prediksi_val_beda': int(((pA >= .5) != (pB >= .5)).sum()), 'n_val': len(va),
                  'maks_selisih_prob_bench': float(np.abs(bA - bB).max()),
                  'peringatan_nondeterministik': sorted(set(nd_peringatan))[:3]}))
'''
open('/content/diag_nondet.py', 'w').write(script)
import subprocess, sys
for mode in ['biasa', 'deterministik']:
    env = dict(os.environ, CUBLAS_WORKSPACE_CONFIG=':4096:8')
    r = subprocess.run([sys.executable, '/content/diag_nondet.py', mode], capture_output=True, text=True, env=env)
    print(r.stdout.strip().split('\n')[-1] if r.stdout.strip() else r.stderr[-1500:])

{"mode": "biasa", "epoch_terbaik": [49, 36], "maks_selisih_prob_val": 0.7916684374213219, "prediksi_val_beda": 27, "n_val": 303, "maks_selisih_prob_bench": 0.13465504301711917, "peringatan_nondeterministik": []}
{"mode": "deterministik", "epoch_terbaik": [42, 42], "maks_selisih_prob_val": 0.0, "prediksi_val_beda": 0, "n_val": 303, "maks_selisih_prob_bench": 0.0, "peringatan_nondeterministik": []}


In [ ]:
import os, glob, pandas as pd
D = '/content/drive/MyDrive/PROJECT'
H = f'{D}/hasil'

# 1. Cek Drive ter-mount dan folder ada
assert os.path.isdir(H), 'Folder hasil tidak ditemukan, cek mount Drive'

# 2. Cek berkas agregat penting (ukuran dalam KB, 0 atau tidak ada berarti gagal simpan)
penting = [
    'ringkasan_semua_run.csv',
    'benchmark_per_pasangan_semua_run.csv',
    'shap_ringkasan_semua_seed.csv',
    'shap_nilai_per_node_semua_seed.csv',
    'baseline_top1_acak.csv',
    'top1_vs_acak_per_run.csv',
    'shap_ringkasan_seed42.csv',
    'shap_nilai_per_node_seed42.csv',
    'shap_ringkasan_C5_run1_arsip.csv',
    'shap_nilai_per_node_C5_run1_arsip.csv',
]
print('=== Berkas agregat ===')
for f in penting:
    p = f'{H}/{f}'
    print(f'{"OK     " if os.path.exists(p) and os.path.getsize(p)>0 else "HILANG "}{f:45s}',
          f'{os.path.getsize(p)/1024:8.1f} KB' if os.path.exists(p) else '')

# 3. Cek 38 run: tiap run harus punya 5 berkas
jadwal = {'B0':[42,1,7,123,2024],'C1':[42,1,7],'C2':[42,1,7],'C4':[42,1,7],'C8':[42,1,7],
          'C3':[42,1,7,123,2024],'C5':[42,1,7,123,2024],'C9':[42,1,7,123,2024],
          'C6':[42,1,7],'C7':[42,1,7]}
akhiran = ['fold_metrics.csv','val_predictions.csv','benchmark_preds.csv','config.json','model_tiap_fold.pt']
kurang, total = [], 0
for k, ss in jadwal.items():
    for s in ss:
        total += 1
        for a in akhiran:
            p = f'{H}/{k}_seed{s}_{a}'
            if not (os.path.exists(p) and os.path.getsize(p) > 0):
                kurang.append(os.path.basename(p))
print(f'\n=== Run: {total} (harus 38) ===')
print('Semua lengkap' if not kurang else f'Kurang {len(kurang)} berkas:\n' + '\n'.join(kurang))

# 4. Cek arsip run pertama
print('\n=== arsip_run_pertama ===')
for f in sorted(glob.glob(f'{H}/arsip_run_pertama/*')):
    print(f'{os.path.basename(f):50s}{os.path.getsize(f)/1024:8.1f} KB')

# 5. Cek ringkasan_semua_run.csv bisa dibaca dan jumlah barisnya
p = f'{H}/ringkasan_semua_run.csv'
if os.path.exists(p):
    df = pd.read_csv(p); print(f'\nringkasan_semua_run.csv: {len(df)} baris, kolom: {list(df.columns)[:8]}')

=== Berkas agregat ===
OK     ringkasan_semua_run.csv                            1.8 KB
OK     benchmark_per_pasangan_semua_run.csv              26.1 KB
OK     shap_ringkasan_semua_seed.csv                     27.1 KB
OK     shap_nilai_per_node_semua_seed.csv                98.1 KB
OK     baseline_top1_acak.csv                             0.6 KB
OK     top1_vs_acak_per_run.csv                           0.3 KB
OK     shap_ringkasan_seed42.csv                          4.7 KB
OK     shap_nilai_per_node_seed42.csv                    18.4 KB
OK     shap_ringkasan_C5_run1_arsip.csv                   1.7 KB
OK     shap_nilai_per_node_C5_run1_arsip.csv              6.8 KB

=== Run: 38 (harus 38) ===
Semua lengkap

=== arsip_run_pertama ===
C3_seed42_benchmark_preds.csv                          0.7 KB
C3_seed42_config.json                                  0.3 KB
C3_seed42_fold_metrics.csv                             0.6 KB
C3_seed42_model_tiap_fold.pt                       12325.9 KB
C5_seed42_

In [ ]:
import shutil
shutil.copy('/content/diag_nondet.py', f'{D}/hasil/diag_nondet.py')
print(os.path.getsize(f'{D}/hasil/diag_nondet.py'), 'byte tersalin')

3595 byte tersalin


INI BISA DI HAPUS NANTI :

In [2]:
from google.colab import drive
drive.mount('/content/drive')

import os
D = '/content/drive/MyDrive/PROJECT'
H = f'{D}/hasil'
print('Folder hasil ada:', os.path.isdir(H))
print('Jumlah berkas:', len(os.listdir(H)) if os.path.isdir(H) else 0)

Mounted at /content/drive
Folder hasil ada: True
Jumlah berkas: 234


In [3]:
import os, json
D = '/content/drive/MyDrive/PROJECT'
H = f'{D}/hasil'

for f in ['diag_nondet.py', 'tahap1_dedup_final.xlsx']:
    p = f'{H}/{f}'
    print(f, 'OK' if os.path.exists(p) else 'BELUM ADA')

for k in ['C5', 'C9']:
    c = json.load(open(f'{H}/{k}_seed42_config.json'))
    print('\n', k, json.dumps(c, indent=1, ensure_ascii=False))

diag_nondet.py OK
tahap1_dedup_final.xlsx BELUM ADA

 C5 {
 "kode": "C5",
 "mode": "weighted",
 "model": "MultiScaleGAT_MeanPool",
 "seed": 42,
 "pembagi_fold": "StratifiedGroupKFold(n_splits=5, shuffle=True)",
 "atribut_grup": "contract_id",
 "n_graf": 1452,
 "n_grup": 539,
 "fold": [
  {
   "fold": 1,
   "n_train": 1149,
   "n_val": 303,
   "n_grup_train": 425,
   "n_grup_val": 114,
   "epoch_terbaik": 49,
   "epoch_berhenti": 50
  },
  {
   "fold": 2,
   "n_train": 1165,
   "n_val": 287,
   "n_grup_train": 430,
   "n_grup_val": 109,
   "epoch_terbaik": 18,
   "epoch_berhenti": 33
  },
  {
   "fold": 3,
   "n_train": 1163,
   "n_val": 289,
   "n_grup_train": 428,
   "n_grup_val": 111,
   "epoch_terbaik": 32,
   "epoch_berhenti": 47
  },
  {
   "fold": 4,
   "n_train": 1163,
   "n_val": 289,
   "n_grup_train": 444,
   "n_grup_val": 95,
   "epoch_terbaik": 50,
   "epoch_berhenti": 50
  },
  {
   "fold": 5,
   "n_train": 1168,
   "n_val": 284,
   "n_grup_train": 429,
   "n_grup_val": 11

In [6]:
!pip install -q torch_geometric
import torch
D = '/content/drive/MyDrive/PROJECT'
d = torch.load(f'{D}/data/processed/pyg/v3_augmented_dedup.pt', weights_only=False)
y = torch.tensor([int(g.y.view(-1)[0]) for g in d['train']])
print('graf:', len(y), 'rentan:', int(y.sum()), 'aman:', int((y == 0).sum()))

torch.manual_seed(0)
sint, nol, total_batch = [], 0, 0
for ep in range(50):
    perm = torch.randperm(len(y))
    for i in range(0, len(y), 32):
        yb = y[perm[i:i+32]]
        n_min = int((yb == 1).sum()); n_maj = len(yb) - n_min
        n_s = max(0, n_maj - n_min) if n_min >= 2 else 0
        sint.append(n_s); total_batch += 1
        nol += (n_s == 0)
print('rata-rata sintetis per batch:', sum(sint) / len(sint))
print('batch tanpa sintetis:', nol, 'dari', total_batch, f'({nol/total_batch:.0%})')

graf: 1452 rentan: 607 aman: 845
rata-rata sintetis per batch: 5.6921739130434785
batch tanpa sintetis: 519 dari 2300 (23%)


In [8]:
import numpy as np, pandas as pd, torch
from sklearn.model_selection import StratifiedGroupKFold
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import f1_score, roc_auc_score

D = '/content/drive/MyDrive/PROJECT'
P = f'{D}/data/processed/pyg'

def fitur_ukuran(d):
    return [float(np.log1p(d.num_nodes)), float(np.log1p(d.edge_index.size(1)))]

def bangun_df_bench(bench_list, bp):
    df_b = pd.DataFrame({
        'pasangan': [d.nama.replace('_fixed.sol', '.sol') for d in bench_list],
        'tipe': ['aman' if d.nama.endswith('_fixed.sol') else 'rentan' for d in bench_list],
        'label': [int(d.y) for d in bench_list]})
    df_b['prob_rata'] = np.array(bp).mean(axis=0)
    return df_b

def ringkas_benchmark(df_b):
    auc = roc_auc_score(df_b['label'], df_b['prob_rata'])
    r = df_b[df_b.tipe == 'rentan'].set_index('pasangan')['prob_rata']
    a = df_b[df_b.tipe == 'aman'].set_index('pasangan')['prob_rata']
    benar = int(sum(r[p] > a[p] for p in r.index if p in a.index))
    return auc, benar, len(r)

def b0_tanpa_simpan(data_list, bench_list, seed, grup_attr):
    X  = np.array([fitur_ukuran(d) for d in data_list]); y = np.array([int(d.y) for d in data_list])
    g  = np.array([getattr(d, grup_attr) for d in data_list])
    Xb = np.array([fitur_ukuran(d) for d in bench_list])
    cv = StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=seed)
    bp, f1s, aucs = [], [], []
    for tr, va in cv.split(X, y, g):
        assert set(g[tr]).isdisjoint(set(g[va]))
        lr = LogisticRegression(class_weight='balanced').fit(X[tr], y[tr])
        p = lr.predict_proba(X[va])[:, 1]; c = lr.predict(X[va])
        f1s.append(f1_score(y[va], c, zero_division=0)); aucs.append(roc_auc_score(y[va], p))
        bp.append(lr.predict_proba(Xb)[:, 1])
    auc_b, benar, total = ringkas_benchmark(bangun_df_bench(bench_list, bp))
    return np.mean(f1s), np.mean(aucs), auc_b, benar

for nama_berkas, grup_attr in [('v2_cfg_prune_dedup.pt', 'nama'), ('v3_augmented_dedup.pt', 'contract_id')]:
    d = torch.load(f'{P}/{nama_berkas}', weights_only=False)
    print('\n', nama_berkas, 'n_train:', len(d['train']), 'n_bench:', len(d['benchmark']))
    for seed in [42, 1, 7, 123, 2024]:
        f1, auc, auc_b, benar = b0_tanpa_simpan(d['train'], d['benchmark'], seed, grup_attr)
        print(f'  seed {seed}: F1 CV={f1:.4f} AUC CV={auc:.4f} | benchmark AUC={auc_b:.4f}, pasangan benar={benar}')


 v2_cfg_prune_dedup.pt n_train: 542 n_bench: 16
  seed 42: F1 CV=0.5982 AUC CV=0.6933 | benchmark AUC=0.5938, pasangan benar=1
  seed 1: F1 CV=0.5927 AUC CV=0.6897 | benchmark AUC=0.5938, pasangan benar=1
  seed 7: F1 CV=0.5955 AUC CV=0.6912 | benchmark AUC=0.5938, pasangan benar=1
  seed 123: F1 CV=0.6122 AUC CV=0.6914 | benchmark AUC=0.5938, pasangan benar=1
  seed 2024: F1 CV=0.6044 AUC CV=0.6892 | benchmark AUC=0.5938, pasangan benar=1

 v3_augmented_dedup.pt n_train: 1452 n_bench: 16
  seed 42: F1 CV=0.6300 AUC CV=0.7423 | benchmark AUC=0.5625, pasangan benar=1
  seed 1: F1 CV=0.6302 AUC CV=0.7422 | benchmark AUC=0.5625, pasangan benar=1
  seed 7: F1 CV=0.6393 AUC CV=0.7455 | benchmark AUC=0.5625, pasangan benar=1
  seed 123: F1 CV=0.6382 AUC CV=0.7485 | benchmark AUC=0.5625, pasangan benar=1
  seed 2024: F1 CV=0.6357 AUC CV=0.7494 | benchmark AUC=0.5625, pasangan benar=1


In [9]:
import pandas as pd, numpy as np
from sklearn.metrics import f1_score
H = '/content/drive/MyDrive/PROJECT/hasil'

def f1_pada(y, p, t): return f1_score(y, (p >= t).astype(int), zero_division=0)

baris = []
for kode, seeds in [('C5',[42,1,7,123,2024]), ('C9',[42,1,7,123,2024]), ('C6',[42,1,7]), ('C7',[42,1,7])]:
    for s in seeds:
        b = pd.read_csv(f'{H}/{kode}_seed{s}_benchmark_preds.csv')
        v = pd.read_csv(f'{H}/{kode}_seed{s}_val_predictions.csv')
        y, p = b['label'].values, b['prob_rata'].values
        t_bench = max(np.unique(p), key=lambda t: f1_pada(y, p, t))      # ambang terbaik di benchmark (bocor)
        grid = np.linspace(0.01, 0.99, 99)
        t_val = max(grid, key=lambda t: f1_pada(v['label'].values, v['prob_rentan'].values, t))
        baris.append({'kode': kode, 'seed': s,
                      'p_min': p.min(), 'p_maks': p.max(), 'n_di_atas_0.5': int((p > 0.5).sum()),
                      'F1_bench@0.5': f1_pada(y, p, 0.5),
                      't_bench': t_bench, 'F1_bench@t_bench': f1_pada(y, p, t_bench),
                      't_val_opt': t_val, 'F1_bench@t_val': f1_pada(y, p, t_val)})
print(pd.DataFrame(baris).round(4).to_string(index=False))

# Data untuk Gambar 4: C9 seed 42, probabilitas tiap pasangan
b = pd.read_csv(f'{H}/C9_seed42_benchmark_preds.csv')
r = b[b.tipe == 'rentan'].set_index('pasangan')['prob_rata']
a = b[b.tipe == 'aman'].set_index('pasangan')['prob_rata']
g = pd.DataFrame({'rentan': r, 'aman_fixed': a})
g['rasio'] = g['rentan'] / g['aman_fixed']; g['rentan_lebih_tinggi'] = g['rentan'] > g['aman_fixed']
print('\nC9 seed 42, probabilitas per pasangan (4 desimal dan 8 desimal)')
print(g.round(8).to_string())

kode  seed  p_min  p_maks  n_di_atas_0.5  F1_bench@0.5  t_bench  F1_bench@t_bench  t_val_opt  F1_bench@t_val
  C5    42 0.0007  0.2706              0        0.0000   0.0182            0.7500       0.52             0.0
  C5     1 0.0009  0.4044              0        0.0000   0.0014            0.7000       0.51             0.0
  C5     7 0.0007  0.1615              0        0.0000   0.0014            0.7619       0.50             0.0
  C5   123 0.0003  0.2082              0        0.0000   0.0005            0.7000       0.52             0.0
  C5  2024 0.0007  0.2429              0        0.0000   0.0008            0.7619       0.49             0.0
  C9    42 0.0006  0.3497              0        0.0000   0.0007            0.7619       0.50             0.0
  C9     1 0.0002  0.3192              0        0.0000   0.0002            0.6667       0.50             0.0
  C9     7 0.0006  0.3102              0        0.0000   0.0010            0.7000       0.50             0.0
  C9   123 0.0008  

In [10]:
import pandas as pd, numpy as np
from sklearn.metrics import f1_score
H = '/content/drive/MyDrive/PROJECT/hasil'
jadwal = {'B0':[42,1,7,123,2024],'C1':[42,1,7],'C2':[42,1,7],'C3':[42,1,7,123,2024],
          'C4':[42,1,7],'C5':[42,1,7,123,2024],'C6':[42,1,7],'C7':[42,1,7],
          'C8':[42,1,7],'C9':[42,1,7,123,2024]}
baris = []
for k, ss in jadwal.items():
    for s in ss:
        b = pd.read_csv(f'{H}/{k}_seed{s}_benchmark_preds.csv')
        y, p = b['label'].values, b['prob_rata'].values
        baris.append({'kode': k, 'seed': s,
                      'f1_bench_0.5': f1_score(y, (p > 0.5).astype(int), zero_division=0),
                      'n_prediksi_rentan': int((p > 0.5).sum())})
df = pd.DataFrame(baris)
df.to_csv(f'{H}/f1_bench_per_run.csv', index=False, float_format='%.10f')

ringkas = df.groupby('kode', sort=False).agg(n_seed=('seed','count'),
            f1_rata=('f1_bench_0.5','mean'), f1_sd=('f1_bench_0.5','std'),
            prediksi_rentan_rata=('n_prediksi_rentan','mean')).round(4)
print(df.round(4).to_string(index=False))
print('\n', ringkas.to_string())

kode  seed  f1_bench_0.5  n_prediksi_rentan
  B0    42        0.0000                  0
  B0     1        0.0000                  0
  B0     7        0.0000                  0
  B0   123        0.0000                  0
  B0  2024        0.0000                  0
  C1    42        0.0000                  0
  C1     1        0.0000                  0
  C1     7        0.2000                  2
  C2    42        0.3636                  3
  C2     1        0.0000                  1
  C2     7        0.0000                  0
  C3    42        0.3636                  3
  C3     1        0.3333                  4
  C3     7        0.3333                  4
  C3   123        0.4615                  5
  C3  2024        0.3333                  4
  C4    42        0.2222                  1
  C4     1        0.2000                  2
  C4     7        0.0000                  0
  C5    42        0.0000                  0
  C5     1        0.0000                  0
  C5     7        0.0000        

In [11]:
import torch, numpy as np, pandas as pd
D = '/content/drive/MyDrive/PROJECT'
P = f'{D}/data/processed/pyg'

baris = []
for ver, berkas in [('v1','cfg_prune_0.02_dedup.pt'), ('v2','v2_cfg_prune_dedup.pt')]:
    d = torch.load(f'{P}/{berkas}', weights_only=False)
    for g in d['train']:
        baris.append({'sumber': ver, 'kelas': 'rentan' if int(g.y) == 1 else 'aman',
                      'nama': g.nama, 'node': int(g.num_nodes), 'edge': int(g.edge_index.size(1))})
    # benchmark dari berkas yang sama (v1 dan v2 berbeda untuk 3 pasangan)
    for g in d['benchmark']:
        baris.append({'sumber': f'benchmark ({ver})', 'kelas': 'rentan' if int(g.y) == 1 else 'aman',
                      'nama': g.nama, 'node': int(g.num_nodes), 'edge': int(g.edge_index.size(1))})

df = pd.DataFrame(baris)
df.to_csv(f'{D}/hasil/ukuran_graf_per_kelas.csv', index=False)

def ringkas(x):
    q = np.percentile(x, [0, 25, 50, 75, 100])
    return pd.Series({'n': len(x), 'min': q[0], 'Q1': q[1], 'median': q[2], 'Q3': q[3], 'maks': q[4], 'rata': x.mean()})

tabel = df.groupby(['sumber', 'kelas'])['node'].apply(ringkas).unstack().round(1)
print(tabel.to_string())

                           n   min     Q1  median     Q3    maks   rata
sumber         kelas                                                   
benchmark (v1) aman      8.0   6.0    7.8     8.0    9.2    10.0    8.2
               rentan    8.0   6.0    8.0     8.5   10.0    14.0    9.1
benchmark (v2) aman      8.0   6.0    7.0     7.0    9.2    10.0    7.9
               rentan    8.0   6.0    7.0     8.0   10.0    14.0    8.8
v1             aman    479.0   3.0   34.0    88.0  136.0  1587.0  117.4
               rentan  217.0  10.0  165.0   295.0  466.0  4753.0  603.4
v2             aman    325.0   2.0  111.0   177.0  334.0  1349.0  257.3
               rentan  217.0  10.0  165.0   292.0  466.0  4042.0  579.8


In [12]:
import os
P = '/content/drive/MyDrive/PROJECT/data/processed/pyg'
for f in sorted(os.listdir(P)): print(f, f'{os.path.getsize(f"{P}/{f}")/1e6:.1f} MB')

cfg.pt 40.8 MB
cfg_prune_0.02.pt 28.2 MB
cfg_prune_0.02_dedup.pt 21.7 MB
v2_cfg_prune.pt 31.2 MB
v2_cfg_prune_dedup.pt 23.9 MB
v2_cfg_prune_pos.pt 32.3 MB
v2_cfg_prune_pos_dedup.pt 24.8 MB
v3_augmented.pt 49.6 MB
v3_augmented_dedup.pt 38.6 MB


In [13]:
import torch, numpy as np, pandas as pd
P = '/content/drive/MyDrive/PROJECT/data/processed/pyg'

hasil = []
for berkas in ['cfg.pt', 'cfg_prune_0.02.pt', 'cfg_prune_0.02_dedup.pt',
               'v2_cfg_prune.pt', 'v2_cfg_prune_dedup.pt']:
    d = torch.load(f'{P}/{berkas}', weights_only=False)
    data = d['train'] if isinstance(d, dict) and 'train' in d else d
    for kelas, nama in [(0, 'aman'), (1, 'rentan')]:
        n = np.array([int(g.num_nodes) for g in data if int(g.y) == kelas])
        hasil.append({'berkas': berkas, 'kelas': nama, 'n': len(n),
                      'median': np.median(n), 'rata': n.mean().round(1), 'maks': n.max()})
    print(berkas, 'tipe:', type(d).__name__, 'kunci:', list(d.keys()) if isinstance(d, dict) else '-')
print(pd.DataFrame(hasil).to_string(index=False))

cfg.pt tipe: dict kunci: ['train', 'benchmark', 'dim', 'tipe']
cfg_prune_0.02.pt tipe: dict kunci: ['train', 'benchmark', 'dim', 'tipe']
cfg_prune_0.02_dedup.pt tipe: dict kunci: ['train', 'benchmark', 'dim', 'tipe']
v2_cfg_prune.pt tipe: dict kunci: ['train', 'benchmark', 'dim', 'tipe']
v2_cfg_prune_dedup.pt tipe: dict kunci: ['train', 'benchmark', 'dim', 'tipe']
                 berkas  kelas   n  median   rata  maks
                 cfg.pt   aman 500   131.5  166.2  2308
                 cfg.pt rentan 244   475.0 1120.5  6635
      cfg_prune_0.02.pt   aman 498    89.0  117.6  1587
      cfg_prune_0.02.pt rentan 244   317.5  763.9  4753
cfg_prune_0.02_dedup.pt   aman 479    88.0  117.4  1587
cfg_prune_0.02_dedup.pt rentan 217   295.0  603.4  4753
        v2_cfg_prune.pt   aman 366   167.0  257.3  1349
        v2_cfg_prune.pt rentan 244   317.5  737.1  4042
  v2_cfg_prune_dedup.pt   aman 325   177.0  257.3  1349
  v2_cfg_prune_dedup.pt rentan 217   292.0  579.8  4042


In [14]:
import torch, numpy as np
P = '/content/drive/MyDrive/PROJECT/data/processed/pyg'
mentah = {g.nama: g for g in torch.load(f'{P}/cfg.pt', weights_only=False)['train']}

for berkas in ['v2_cfg_prune.pt', 'v2_cfg_prune_dedup.pt']:
    v2 = torch.load(f'{P}/{berkas}', weights_only=False)['train']
    for kelas, nm in [(0, 'aman'), (1, 'rentan')]:
        nama = [g.nama for g in v2 if int(g.y) == kelas]
        ada = [n for n in nama if n in mentah]
        ukuran = np.array([int(mentah[n].num_nodes) for n in ada])
        print(f'{berkas:24s} {nm:7s} total={len(nama)} ditemukan_di_cfg.pt={len(ada)}',
              f'median_mentah={np.median(ukuran):.1f}' if len(ada) else 'tidak ada')

v2_cfg_prune.pt          aman    total=366 ditemukan_di_cfg.pt=0 tidak ada
v2_cfg_prune.pt          rentan  total=244 ditemukan_di_cfg.pt=244 median_mentah=475.0
v2_cfg_prune_dedup.pt    aman    total=325 ditemukan_di_cfg.pt=0 tidak ada
v2_cfg_prune_dedup.pt    rentan  total=217 ditemukan_di_cfg.pt=217 median_mentah=413.0


In [16]:
import zipfile, glob, os
from collections import Counter
D = '/content/drive/MyDrive/PROJECT'
kunci = ['etherbank', 'etherstore', 'modifier_reentrancy', 'reentrance',
         'reentrancy_dao', 'reentrancy_insecure', 'reentrancy_simple', 'simple_dao']

for z in sorted(glob.glob(f'{D}/data/raw/*.zip')):
    with zipfile.ZipFile(z) as f: names = f.namelist()
    sol = [n for n in names if n.endswith('.sol')]
    folder = Counter(os.path.dirname(n) for n in sol)
    print(f'\n=== {os.path.basename(z)}: {len(names)} entri, {len(sol)} berkas .sol')
    for k, v in sorted(folder.items())[:12]: print(f'   {k or "(akar)"}: {v}')
    print('   contoh:', sol[:3])
    ada = [n for n in sol if any(k in os.path.basename(n) for k in kunci)]
    print('   berkas bernama benchmark:', len(ada), ada[:20])


=== dataset_final.zip: 766 entri, 760 berkas .sol
   dataset_penelitian/0_normal: 500
   dataset_penelitian/1_reentrancy: 244
   dataset_penelitian/3_expert_benchmark/aman: 8
   dataset_penelitian/3_expert_benchmark/rentan: 8
   contoh: ['dataset_penelitian/0_normal/0x9cebe2c89d373228411003fa47781fb7eefaf44e.sol', 'dataset_penelitian/0_normal/0x0077d27cb82ff12322987b225bfce0bb6e8931b4.sol', 'dataset_penelitian/0_normal/0x84c722e6f1363e8d5c6db3ea600bef9a006da824.sol']
   berkas bernama benchmark: 16 ['dataset_penelitian/3_expert_benchmark/rentan/reentrancy_insecure.sol', 'dataset_penelitian/3_expert_benchmark/rentan/simple_dao.sol', 'dataset_penelitian/3_expert_benchmark/rentan/reentrancy_simple.sol', 'dataset_penelitian/3_expert_benchmark/rentan/reentrancy_dao.sol', 'dataset_penelitian/3_expert_benchmark/rentan/etherbank.sol', 'dataset_penelitian/3_expert_benchmark/rentan/modifier_reentrancy.sol', 'dataset_penelitian/3_expert_benchmark/rentan/etherstore.sol', 'dataset_penelitian/3_exp

In [17]:
import zipfile, re, heapq, pandas as pd
D = '/content/drive/MyDrive/PROJECT'
KW = set('''function public private internal external view pure payable returns return if else for while do
require assert revert msg sender value call send transfer address uint uint8 uint256 int mapping bool
contract pragma solidity constructor modifier emit event balance this new memory storage delete true false
import is using library interface struct enum now block timestamp tx origin selfdestruct suicide throw
string bytes bytes32 constant indexed anonymous fallback abi'''.split())

def tokens(src, norm):
    src = re.sub(r'/\*.*?\*/', ' ', src, flags=re.S); src = re.sub(r'//[^\n]*', ' ', src)
    t = re.findall(r'[A-Za-z_$][A-Za-z0-9_$]*|\d+|[^\s\w]', src)
    if norm:
        t = ['NUM' if x.isdigit() else ('ID' if (x[0].isalpha() or x[0] in '_$') and x not in KW else x) for x in t]
    return t

def grams(t, n=5): return {hash(tuple(t[i:i+n])) for i in range(len(t) - n + 1)}

def fungsi(src):
    src = re.sub(r'/\*.*?\*/', ' ', src, flags=re.S); src = re.sub(r'//[^\n]*', ' ', src)
    hasil = []
    for m in re.finditer(r'\bfunction\s+([A-Za-z_]\w*)?\s*\(', src):
        i = src.find('{', m.end())
        if i < 0: continue
        depth, j = 0, i
        while j < len(src):
            depth += (src[j] == '{') - (src[j] == '}')
            j += 1
            if depth == 0: break
        hasil.append((m.group(1) or '(fallback)', src[m.start():j]))
    return hasil

zf = zipfile.ZipFile(f'{D}/data/raw/dataset_final.zip')
baca = lambda z, n: z.read(n).decode('utf-8', errors='ignore')

# item benchmark: kontrak utuh dan tiap fungsinya
items = []
for n in sorted(zf.namelist()):
    if '3_expert_benchmark' in n and n.endswith('.sol'):
        src = baca(zf, n); kon = n.split('/')[-1]
        items.append({'kontrak': kon, 'fungsi': None, 'src': src})
        for nama_f, body in fungsi(src):
            if len(grams(tokens(body, False))) >= 20:
                items.append({'kontrak': kon, 'fungsi': nama_f, 'src': body})
for it in items:
    it['g'] = {nm: grams(tokens(it['src'], nm)) for nm in (False, True)}
U = {nm: set().union(*[it['g'][nm] for it in items]) for nm in (False, True)}
print('item benchmark:', len(items), '(kontrak utuh + fungsi berukuran cukup)')

# pool latih: v1 dan v2
pool = []
for berkas in ['dataset_final.zip', 'dataset_v2.zip']:
    z = zipfile.ZipFile(f'{D}/data/raw/{berkas}')
    for n in z.namelist():
        if n.endswith('.sol') and ('/0_normal/' in n or '/1_reentrancy/' in n): pool.append((z, berkas + ':' + n))
print('kontrak pembanding:', len(pool))

best = {(i, nm): (0.0, '-') for i in range(len(items)) for nm in (False, True)}
top = {(i, nm): [] for i, it in enumerate(items) if it['fungsi'] is None for nm in (False, True)}
for z, nama in pool:
    src = baca(z, nama.split(':', 1)[1])
    for nm in (False, True):
        gt = grams(tokens(src, nm)) & U[nm]
        if not gt: continue
        for i, it in enumerate(items):
            c = len(it['g'][nm] & gt) / max(1, len(it['g'][nm]))
            if c > best[(i, nm)][0]: best[(i, nm)] = (c, nama)
            if (i, nm) in top:
                heapq.heappush(top[(i, nm)], (c, nama))
                if len(top[(i, nm)]) > 3: heapq.heappop(top[(i, nm)])

baris = []
for i, it in enumerate(items):
    if it['fungsi'] is not None: continue
    fn = [(best[(j, True)][0], items[j]['fungsi'], best[(j, True)][1]) for j, x in enumerate(items)
          if x['kontrak'] == it['kontrak'] and x['fungsi'] is not None]
    fmax = max(fn) if fn else (0, '-', '-')
    baris.append({'kontrak': it['kontrak'], 'n_gram': len(it['g'][False]),
                  'kont_mentah': round(best[(i, False)][0], 3), 'terdekat_mentah': best[(i, False)][1],
                  'kont_norm': round(best[(i, True)][0], 3), 'terdekat_norm': best[(i, True)][1],
                  'fungsi_maks_norm': round(fmax[0], 3), 'fungsi': fmax[1], 'terdekat_fungsi': fmax[2]})
df = pd.DataFrame(baris).sort_values('kont_norm', ascending=False)
df.to_csv(f'{D}/hasil/containment_benchmark.csv', index=False)
pd.set_option('display.width', 250); pd.set_option('display.max_colwidth', 60)
print(df.to_string(index=False))
print('\nTiga terdekat (norm) untuk lima kontrak teratas:')
for kon in df.kontrak.head(5):
    i = [j for j, x in enumerate(items) if x['kontrak'] == kon and x['fungsi'] is None][0]
    print(kon, [(round(c, 3), n) for c, n in sorted(top[(i, True)], reverse=True)])

item benchmark: 36 (kontrak utuh + fungsi berukuran cukup)
kontrak pembanding: 1488
                      kontrak  n_gram  kont_mentah                                                                                  terdekat_mentah  kont_norm                                                                                    terdekat_norm  fungsi_maks_norm          fungsi                                                                                  terdekat_fungsi
      reentrancy_insecure.sol      69        0.290 dataset_final.zip:dataset_penelitian/1_reentrancy/0x00636D0fD88135b8865c6C2caa4C498D602A4052.sol      0.580 dataset_final.zip:dataset_penelitian/1_reentrancy/0x0111d8e8367e48156ccdeff6fc47ceca547a1a5b.sol             0.609 withdrawBalance dataset_final.zip:dataset_penelitian/1_reentrancy/0x00636D0fD88135b8865c6C2caa4C498D602A4052.sol
      modifier_reentrancy.sol     176        0.170 dataset_final.zip:dataset_penelitian/1_reentrancy/0x89f11af4b6c3cbcb4c55f9033fb403483b6430b

In [18]:
pasangan = [
 ('reentrancy_insecure.sol', 'withdrawBalance', 'dataset_final.zip:dataset_penelitian/1_reentrancy/0x00636D0fD88135b8865c6C2caa4C498D602A4052.sol'),
 ('modifier_reentrancy.sol', 'supportsToken', 'dataset_final.zip:dataset_penelitian/1_reentrancy/0x681192ae9d24643ef2f61dae48325e20f451b862.sol'),
 ('reentrancy_dao.sol', 'deposit', 'dataset_final.zip:dataset_penelitian/0_normal/0xc02aaa39b223fe8d0a0e5c4f27ead9083c756cc2.sol')]
for kon, nama_f, latih in pasangan:
    bench = next(it['src'] for it in items if it['kontrak'] == kon and it['fungsi'] == nama_f)
    berkas, path = latih.split(':', 1)
    isi = zipfile.ZipFile(f'{D}/data/raw/{berkas}').read(path).decode('utf-8', errors='ignore')
    cocok = [b for n, b in fungsi(isi) if n == nama_f]
    print('=' * 70, f'\nBENCHMARK {kon}.{nama_f}\n', bench.strip(),
          f'\n\nLATIH {path.split("/")[-1]}.{nama_f}\n', (cocok[0].strip() if cocok else '(fungsi bernama sama tidak ditemukan)'))

BENCHMARK reentrancy_insecure.sol.withdrawBalance
 function withdrawBalance() public {
        uint amountToWithdraw = userBalances[msg.sender];
         
        (bool success, ) = msg.sender.call.value(amountToWithdraw)("");  
        require(success);
        userBalances[msg.sender] = 0;
    } 

LATIH 0x00636D0fD88135b8865c6C2caa4C498D602A4052.sol.withdrawBalance
 (fungsi bernama sama tidak ditemukan)
BENCHMARK modifier_reentrancy.sol.supportsToken
 function supportsToken() external pure returns(bytes32){
        return(keccak256(abi.encodePacked("Nu Token")));
    } 

LATIH 0x681192ae9d24643ef2f61dae48325e20f451b862.sol.supportsToken
 (fungsi bernama sama tidak ditemukan)
BENCHMARK reentrancy_dao.sol.deposit
 function deposit() public payable {
        credit[msg.sender] += msg.value;
        balance += msg.value;
    } 

LATIH 0xc02aaa39b223fe8d0a0e5c4f27ead9083c756cc2.sol.deposit
 function deposit() public payable {


        balanceOf[msg.sender] += msg.value;


        Deposit

In [22]:
import difflib, collections, torch, pandas as pd
D = '/content/drive/MyDrive/PROJECT'
P = f'{D}/data/processed/pyg/v3_augmented_dedup.pt'
bench = {g.nama: g for g in torch.load(P, weights_only=False)['benchmark']}

baris = []
for n in sorted(zf.namelist()):
    if '3_expert_benchmark/rentan/' in n and n.endswith('.sol'):
        nama = os.path.basename(n); fixed = nama.replace('.sol', '_fixed.sol')
        a = baca(zf, n); b = baca(zf, n.replace('/rentan/', '/aman/').replace('.sol', '_fixed.sol'))
        ta, tb = tokens(a, False), tokens(b, False)
        sama_multiset = collections.Counter(ta) == collections.Counter(tb)
        norm = lambda s: [l.strip() for l in re.sub(r'//[^\n]*', '', re.sub(r'/\*.*?\*/', '', s, flags=re.S)).splitlines() if l.strip()]
        diff = [l for l in difflib.unified_diff(norm(a), norm(b), lineterm='', n=0) if l[:1] in '+-' and l[:3] not in ('+++', '---')]
        baris.append({'pasangan': nama, 'token_sama_urutan_beda': sama_multiset,
                      'baris_berbeda': len(diff),
                      'node_rentan': int(bench[nama].num_nodes), 'node_fixed': int(bench[fixed].num_nodes),
                      'edge_rentan': int(bench[nama].edge_index.size(1)), 'edge_fixed': int(bench[fixed].edge_index.size(1))})
        print('\n', nama, '\n  ', '\n   '.join(diff[:8]))
print(pd.DataFrame(baris).to_string(index=False))


 etherbank.sol 
   +userBalances[msg.sender] = 0;
   -userBalances[msg.sender] = 0;

 etherstore.sol 
   -require(msg.sender.call.value(_weiToWithdraw)());
   +require(msg.sender.call.value(_weiToWithdraw)());

 modifier_reentrancy.sol 
   -function airDrop() hasNoBalance supportsToken  public{
   +function airDrop() hasNoBalance public{
   -}
   -modifier supportsToken() {
   -_;
   -contract attack{
   -bool hasBeenCalled;
   -function supportsToken() external returns(bytes32){

 reentrance.sol 
   +balances[msg.sender] -= _amount;
   -balances[msg.sender] -= _amount;

 reentrancy_dao.sol 
   +credit[msg.sender] = 0;
   -credit[msg.sender] = 0;

 reentrancy_insecure.sol 
   +userBalances[msg.sender] = 0;
   -userBalances[msg.sender] = 0;

 reentrancy_simple.sol 
   -if( ! (msg.sender.call.value(userBalance[msg.sender])() ) ){
   +uint amount = userBalance[msg.sender];
   +userBalance[msg.sender] = 0;
   +if( ! (msg.sender.call.value(amount)() ) ){
   -userBalance[msg.sender] = 0;

 

In [23]:
import glob, json, os
D = '/content/drive/MyDrive'
for f in sorted(glob.glob(f'{D}/**/*.ipynb', recursive=True)):
    if '.ipynb_checkpoints' in f: continue
    try:
        kode = '\n'.join(''.join(c['source']) for c in json.load(open(f))['cells'] if c['cell_type'] == 'code')
    except Exception as e:
        print('gagal baca', f); continue
    print(f'{os.path.relpath(f, D):70s} dedup={"dedup" in kode.lower()}  dataset_v2={"dataset_v2" in kode}  v2_cfg_prune={"v2_cfg_prune" in kode}')

Colab Notebooks/Analisis Data.ipynb                                    dedup=False  dataset_v2=False  v2_cfg_prune=False
Colab Notebooks/Analisis Sentimen Gopay.ipynb                          dedup=False  dataset_v2=False  v2_cfg_prune=False
Colab Notebooks/Analisis_Sentimen_Gopay (1).ipynb                      dedup=False  dataset_v2=False  v2_cfg_prune=False
Colab Notebooks/Copy of PROJECT_v1_eksplorasi.ipynb                  dedup=False  dataset_v2=False  v2_cfg_prune=False
Colab Notebooks/Copy of PROJECT_v2.ipynb                             dedup=False  dataset_v2=True  v2_cfg_prune=True
Colab Notebooks/Copy of PROJECT_v2_fitur_posisi.ipynb                dedup=False  dataset_v2=False  v2_cfg_prune=True
Colab Notebooks/Copy of PROJECT_v2_model_dan_shap.ipynb              dedup=False  dataset_v2=False  v2_cfg_prune=True
Colab Notebooks/Copy of PROJECT_v4_fitur_posisi.ipynb                dedup=False  dataset_v2=False  v2_cfg_prune=True
Colab Notebooks/Copy of PROJECT_v5_augmentasi.i

In [24]:
import glob, json, os, re
D = '/content/drive/MyDrive/Colab Notebooks'
target = ['cfg.pt', 'cfg_prune_0.02.pt', 'dataset_v2.zip', 'v2_cfg_prune.pt', 'v2_cfg_prune_pos.pt',
          'v3_augmented.pt', 'dedup.pt']
for f in sorted(glob.glob(f'{D}/PROJECT*.ipynb')) + [f'{D}/Percobaan Pelengkap.ipynb']:
    nb = json.load(open(f))['cells']
    kode = '\n'.join(''.join(c['source']) for c in nb if c['cell_type'] == 'code')
    baris = [l.strip() for l in kode.splitlines()
             if ('torch.save' in l or 'make_archive' in l or 'ZipFile' in l and "'w'" in l)
             and any(t.split('.')[0] in l for t in target)]
    print(f'\n== {os.path.basename(f)}')
    for l in baris[:8]: print('   ', l[:140])


== PROJECT_GraphSMOTE.ipynb

== PROJECT_v1_eksplorasi.ipynb

== PROJECT_v2.ipynb

== PROJECT_v2_fitur_posisi.ipynb

== PROJECT_v2_model_dan_shap.ipynb

== PROJECT_v3_augmentasi.ipynb

== PROJECT_v6_revisi_reviewer.ipynb
    torch.save(obj_v1_dedup, '/content/cfg_prune_0.02_dedup.pt')
    torch.save(obj_dedup, f'/content/{nama_dedup}')
    torch.save(obj_v3_dedup, '/content/v3_augmented_dedup.pt')

== Percobaan Pelengkap.ipynb


In [25]:
import glob, json, os
D = '/content/drive/MyDrive/Colab Notebooks'
for f in sorted(glob.glob(f'{D}/PROJECT*.ipynb')) + [f'{D}/Percobaan Pelengkap.ipynb']:
    kode = '\n'.join(''.join(c['source']) for c in json.load(open(f))['cells'] if c['cell_type'] == 'code')
    baris = [l.strip() for l in kode.splitlines()
             if 'torch.save' in l or 'make_archive' in l or '.zip' in l or 'to_csv' in l and 'pyg' in l]
    print(f'\n== {os.path.basename(f)}  ({len(baris)} baris)')
    for l in baris[:10]: print('   ', l[:150])


== PROJECT_GraphSMOTE.ipynb  (2 baris)
    torch.save({'state_dicts': [m.state_dict() for m in model_smote], 'hasil': hasil_smote},
    torch.save({'state_dicts': [m.state_dict() for m in model_v2_smote]},

== PROJECT_v1_eksplorasi.ipynb  (3 baris)
    !unzip -q -o /content/drive/MyDrive/PROJECT/data/raw/dataset_final.zip -d /content/data/raw/
    torch.save({'train': train, 'benchmark': bench, 'dim': DIM, 'tipe': TIPE},
    torch.save({'state_dicts': [m.state_dict() for m in model_fold],

== PROJECT_v2.ipynb  (7 baris)
    !unzip -q -o {D}/data/raw/dataset_final.zip -d /content/data/raw/
    !unzip -q -o {D}/data/raw/dataset_final.zip -d /content/data/raw/
    !cd /content/data/raw && zip -qr {D}/data/raw/dataset_v2.zip dataset_v2
    !unzip -q -o {D}/data/raw/dataset_v2.zip -d /content/data/raw/
    print('zip di Drive ada:', os.path.exists(f'{D}/data/raw/dataset_v2.zip'))
    !unzip -q -o {D}/data/raw/dataset_v2.zip -d /content/data/raw/
    torch.save({'train': train, 'benchmark':

In [26]:
import glob, json, os
D = '/content/drive/MyDrive/Colab Notebooks'
kunci = ['cfg_prune', 'v2_cfg_prune_pos', 'cfg.pt', "'train'"]
for f in sorted(glob.glob(f'{D}/*PROJECT*.ipynb')):
    kode = '\n'.join(''.join(c['source']) for c in json.load(open(f))['cells'] if c['cell_type'] == 'code')
    baris = [l.strip() for l in kode.splitlines()
             if ('torch.save' in l or 'torch.load' in l) and any(k in l for k in kunci)]
    if baris:
        print(f'\n== {os.path.basename(f)}')
        for l in baris[:6]: print('   ', l[:150])


== Copy of PROJECT_v1_eksplorasi.ipynb
    torch.save({'train': train, 'benchmark': bench, 'dim': DIM, 'tipe': TIPE},
    paket = torch.load(f'{OUT_PYG}/cfg_prune_0.02.pt', weights_only=False)
    torch.save({'train': train, 'benchmark': bench, 'dim': DIM, 'tipe': TIPE},
    paket = torch.load(f'{OUT_PYG}/cfg_prune_0.02.pt', weights_only=False)
    paket = torch.load(f'{OUT_PYG}/cfg_prune_0.02.pt', weights_only=False)

== Copy of PROJECT_v2.ipynb
    torch.save({'train': train, 'benchmark': bench, 'dim': DIM, 'tipe': TIPE},

== Copy of PROJECT_v2_fitur_posisi.ipynb
    paket_pos = torch.load(f'{D}/data/processed/pyg/v2_cfg_prune_pos.pt', weights_only=False)
    simpan_pos = torch.load(f'{D}/hasil/model_v2_cfg_prune_pos.pt', weights_only=False)

== Copy of PROJECT_v4_fitur_posisi.ipynb
    paket_pos = torch.load(f'{D}/data/processed/pyg/v2_cfg_prune_pos.pt', weights_only=False)
    simpan_pos = torch.load(f'{D}/hasil/model_v2_cfg_prune_pos.pt', weights_only=False)
    torch.save({'trai